## Validation hyperparameter tuning — \(T\), \(\gamma_{\min}\), Top-\(B\), threshold

In [27]:
# ======================================================================
# RQ2 CELL 13 — REVISED v2
# VALIDATION-ONLY HYPERPARAMETER TUNING
# ======================================================================
#
# FIX IN THIS VERSION
# -------------------
# RQ1 metric distinction is now explicit:
#
#   active_hop_rows:
#       number of non-empty plan-hop profiler rows
#
#   active_prefixes:
#       sum_h |P_h| across those rows
#
# Verified RQ1 validation references:
#
#   WebQSP:
#       active_hop_rows = 971
#       active_prefixes = 2440
#
#   CWQ:
#       active_hop_rows = 16564
#       active_prefixes = 57841
#
# TUNING
# ------
# AFP:
#   T
#   gamma_min
#
# Fixed Top-B:
#   B
#
# Fixed Threshold:
#   tau
#
# PREDECLARED SELECTION RULE
# --------------------------
# Primary:
#     maximize SSR subject to validation AR >= 0.99
#
# Fallback:
#     maximize AR first, then SSR
#
# IMPORTANT
# ---------
# - validation only
# - selected scorer already frozen
# - Feature-v2 already frozen
# - relation plans frozen
# - exact RoG graph semantics frozen
# - final-hop protection enabled
# - singleton bypass enabled
# - gold NEVER enters scorer or selector
# - gold used only after traversal for validation Answer Retention
# - no TEST examples accessed
#
# CPU is sufficient.
#
# This version additionally uses:
#   - batched semantic prefill
#   - question-local graph-expansion cache
#   - question-local scorer cache
#
# These are COMPUTATIONAL optimizations only.
# They do not alter any method metric or selection behavior.
# ======================================================================

import hashlib
import json
import math
import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


# ======================================================================
# 1. HARD PREREQUISITES
# ======================================================================

required = [
    "AFP_RUNTIME_SCORE_GROUP",
    "AFP_RUNTIME_SEMANTIC_ENCODER",
    "AFP_RUNTIME_SCORERS",
    "AFP_RUNTIME_STANDARDIZERS",

    "AFP_RUNTIME_FEATURE_VERSION",
    "AFP_RUNTIME_FEATURE_DIM",

    "webqsp_val_plan_rows",
    "cwq_val_plan_rows",

    "webqsp_val_runtime",
    "cwq_val_runtime",

    "webqsp_ckpt_sha",
    "cwq_ckpt_sha",

    "has_readable_entity_surface_runtime",
    "entity_surface_text_runtime",
    "relation_surface_text_runtime",
    "relation_sequence_text_runtime",
]

missing = [
    name
    for name in required
    if name not in globals()
]

assert not missing, (
    "Missing required runtime objects:\n  "
    + "\n  ".join(missing)
)

assert int(
    AFP_RUNTIME_FEATURE_DIM
) == 27


print(
    "Cell 13 prerequisites: PASSED"
)

print(
    "Runtime scorer: READY"
)

print(
    "Runtime device: CPU"
)

print(
    "Feature version:",
    AFP_RUNTIME_FEATURE_VERSION
)


# ======================================================================
# 2. OUTPUT DIRECTORY
# ======================================================================

ROOT = Path(
    "/kaggle/working/"
    "step3_rq2_dev_v1"
)

TUNING_DIR = (
    ROOT
    / "11_validation_tuning"
)

TUNING_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ======================================================================
# 3. PREDECLARED TUNING GRID
# ======================================================================

AR_FLOOR = 0.99


T_GRID = [
    0.5,
    1.0,
    2.0,
]


GAMMA_MIN_GRID = [
    0.50,
    0.70,
    0.80,
    0.90,
    0.95,
]


TOP_B_GRID = [
    1,
    2,
    4,
    8,
    16,
    32,
]


THRESHOLD_GRID = [
    0.1,
    0.2,
    0.3,
    0.4,
    0.5,
    0.6,
    0.7,
    0.8,
    0.9,
]


TIE_ATOL = 1e-8


print(
    "\n"
    + "=" * 108
)

print(
    "PREDECLARED VALIDATION MODEL-SELECTION RULE"
)

print(
    "=" * 108
)


print(
    "Primary objective: maximize SSR "
    f"subject to AR >= {AR_FLOOR}"
)

print(
    "Fallback: maximize AR, then SSR, "
    "if no configuration satisfies AR floor."
)


print(
    "\nAFP grid"
)

print(
    "  T:         ",
    T_GRID
)

print(
    "  gamma_min: ",
    GAMMA_MIN_GRID
)


print(
    "\nFixed Top-B grid:"
)

print(
    " ",
    TOP_B_GRID
)


print(
    "\nFixed Threshold grid:"
)

print(
    " ",
    THRESHOLD_GRID
)


print(
    "\nRandom-B: inherits selected Fixed Top-B B; "
    "NOT independently tuned."
)

print(
    "Adaptive-Budget Random: inherits selected AFP adaptive "
    "budgets; NOT independently tuned."
)


# ======================================================================
# 4. CONFIGURATION LIST
# ======================================================================

CONFIGS = []


for T in T_GRID:

    for gamma_min in GAMMA_MIN_GRID:

        CONFIGS.append(
            {
                "config_id":
                    f"afp_T{T:g}_g{gamma_min:g}",

                "family":
                    "afp",

                "T":
                    float(
                        T
                    ),

                "gamma_min":
                    float(
                        gamma_min
                    ),
            }
        )


for B in TOP_B_GRID:

    CONFIGS.append(
        {
            "config_id":
                f"fixed_top_b_B{B}",

            "family":
                "fixed_top_b",

            "B":
                int(
                    B
                ),
        }
    )


for tau in THRESHOLD_GRID:

    CONFIGS.append(
        {
            "config_id":
                f"fixed_threshold_tau{tau:g}",

            "family":
                "fixed_threshold",

            "tau":
                float(
                    tau
                ),
        }
    )


assert len(
    CONFIGS
) == (
    len(
        T_GRID
    )
    * len(
        GAMMA_MIN_GRID
    )
    + len(
        TOP_B_GRID
    )
    + len(
        THRESHOLD_GRID
    )
)


CONFIG_BY_ID = {
    config[
        "config_id"
    ]:
        config

    for config in CONFIGS
}


print(
    "\nTotal validation configurations:",
    len(
        CONFIGS
    )
)


# ======================================================================
# 5. TUNING SPECIFICATION FINGERPRINT
# ======================================================================

TUNING_SPEC = {
    "version":
        "rq2_cell13_validation_tuning_v2_metric_corrected",

    "selection_rule": {
        "primary":
            "maximize_ssr_subject_to_ar_floor",

        "ar_floor":
            AR_FLOOR,

        "fallback":
            "maximize_ar_then_ssr",
    },

    "metric_definitions": {
        "active_hop_rows":
            "number_of_nonempty_plan_hop_states",

        "active_prefixes":
            "sum_of_active_path_prefix_counts_across_active_hop_rows",

        "primary_search_cost":
            "edges_examined",

        "ssr":
            "1_minus_method_edges_over_rog_edges",

        "answer_retention":
            "method_reachable_questions_over_rog_reachable_questions",
    },

    "afp": {
        "T_grid":
            T_GRID,

        "gamma_min_grid":
            GAMMA_MIN_GRID,

        "tie_atol":
            TIE_ATOL,

        "fully_tied_behavior":
            "retain_all",

        "cutoff_tie_behavior":
            "expand_cutoff_ties",
    },

    "fixed_top_b": {
        "B_grid":
            TOP_B_GRID,

        "cutoff_tie_behavior":
            "expand_cutoff_ties",
    },

    "fixed_threshold": {
        "tau_grid":
            THRESHOLD_GRID,

        "forced_top1":
            False,
    },

    "random_b": {
        "tuned":
            False,

        "budget_source":
            "selected_fixed_top_b",
    },

    "adaptive_budget_random": {
        "tuned":
            False,

        "budget_source":
            "selected_afp_dynamic_budget",
    },

    "final_hop_protection":
        True,

    "singleton_bypass":
        True,

    "test_examples_accessed":
        False,
}


TUNING_SPEC_JSON = json.dumps(
    TUNING_SPEC,
    sort_keys=True,
    separators=(
        ",",
        ":"
    )
)


TUNING_SPEC_SHA = hashlib.sha256(
    TUNING_SPEC_JSON.encode(
        "utf-8"
    )
).hexdigest()


print(
    "Tuning specification SHA256:",
    TUNING_SPEC_SHA
)


# ======================================================================
# 6. BASIC FIELD NORMALIZATION
# ======================================================================

def as_entity_list(
    value
):

    if value is None:

        return []


    if isinstance(
        value,
        str
    ):

        return [
            value
        ]


    if isinstance(
        value,
        (
            list,
            tuple,
            set,
            np.ndarray,
        )
    ):

        return [
            str(
                x
            )
            for x in value
        ]


    return [
        str(
            value
        )
    ]


def normalize_relation_plans(
    value
):

    if value is None:

        return []


    assert isinstance(
        value,
        (
            list,
            tuple,
        )
    )


    plans = []


    for plan in value:

        if plan is None:

            plans.append(
                []
            )

            continue


        assert isinstance(
            plan,
            (
                list,
                tuple,
            )
        )


        plans.append(
            [
                str(
                    relation
                )
                for relation in plan
            ]
        )


    return plans


# ======================================================================
# 7. EXACT RoG GRAPH CONSTRUCTION
# ======================================================================
#
# Official RoG semantics already verified in Cell 12B-R:
#
#   undirected simple graph
#   one edge per unordered entity pair
#   later duplicate pair overwrites relation
#   original neighbor insertion order preserved
# ======================================================================

def build_exact_rog_adjacency(
    triples
):

    adjacency = {}


    for triple in triples:

        assert len(
            triple
        ) == 3


        h, r, t = triple


        h = str(
            h
        )

        t = str(
            t
        )

        r = str(
            r
        ).strip()


        if h not in adjacency:

            adjacency[
                h
            ] = {}


        if t not in adjacency:

            adjacency[
                t
            ] = {}


        # Assignment to an existing dictionary key overwrites the
        # relation but preserves the original insertion position.
        adjacency[
            h
        ][
            t
        ] = r


        adjacency[
            t
        ][
            h
        ] = r


    return adjacency


# ======================================================================
# 8. EXACT UNPRUNED RoG TRAVERSAL
# ======================================================================

def traverse_rog_unpruned(
    adjacency,
    topic_entities,
    plan,
    gold_answers=None
):

    if len(
        plan
    ) == 0:

        return {
            "active_hop_rows":
                0,

            "active_prefixes":
                0,

            "edges_examined":
                0,

            "candidate_branches":
                0,

            "reachable":
                False,

            "final_prefixes":
                [],
        }


    active = [
        (
            str(
                entity
            ),
        )
        for entity in topic_entities
    ]


    active_hop_rows = 0
    active_prefixes_total = 0

    edges_examined = 0
    candidate_branches = 0


    for hop, target_relation in enumerate(
        plan
    ):

        if not active:

            break


        # Number of non-empty plan-hop states.
        active_hop_rows += 1


        # Sum of actual path prefixes present at each active hop.
        active_prefixes_total += len(
            active
        )


        candidates = []


        for prefix in active:

            endpoint = prefix[
                -1
            ]


            neighbors = adjacency.get(
                endpoint,
                {}
            )


            # Every unique neighbor is examined.
            edges_examined += len(
                neighbors
            )


            for neighbor, relation in (
                neighbors.items()
            ):

                if relation == target_relation:

                    candidates.append(
                        prefix
                        + (
                            neighbor,
                        )
                    )


        candidate_branches += len(
            candidates
        )


        active = candidates


    reachable = False


    if (
        gold_answers is not None
        and
        active
    ):

        answer_set = {
            str(
                x
            )
            for x in gold_answers
        }


        reachable = any(
            prefix[
                -1
            ]
            in answer_set

            for prefix in active
        )


    return {
        "active_hop_rows":
            int(
                active_hop_rows
            ),

        "active_prefixes":
            int(
                active_prefixes_total
            ),

        "edges_examined":
            int(
                edges_examined
            ),

        "candidate_branches":
            int(
                candidate_branches
            ),

        "reachable":
            bool(
                reachable
            ),

        "final_prefixes":
            active,
    }


# ======================================================================
# 9. CORRECTED RQ1 RoG FIDELITY REFERENCES
# ======================================================================

EXPECTED_ROG = {
    "webqsp": {
        "questions":
            246,

        "total_predicted_plans":
            721,

        "nonempty_plans":
            721,

        "empty_plans":
            0,

        "active_hop_rows":
            971,

        "active_prefixes":
            2440,

        "edges_examined":
            341526,

        "candidate_branches":
            7983,

        "reachable_plans":
            345,

        "reachable_questions":
            205,
    },

    "cwq": {
        "questions":
            3519,

        "total_predicted_plans":
            10536,

        "nonempty_plans":
            10529,

        "empty_plans":
            7,

        "active_hop_rows":
            16564,

        "active_prefixes":
            57841,

        "edges_examined":
            5257272,

        "candidate_branches":
            247161,

        "reachable_plans":
            3971,

        "reachable_questions":
            2425,
    },
}


# ======================================================================
# 10. CORRECTED RoG FIDELITY GATE
# ======================================================================

def rog_fidelity_gate(
    dataset_name,
    planning_rows
):

    expected = EXPECTED_ROG[
        dataset_name
    ]


    assert len(
        planning_rows
    ) == expected[
        "questions"
    ]


    totals = {
        "questions":
            len(
                planning_rows
            ),

        "total_predicted_plans":
            0,

        "nonempty_plans":
            0,

        "empty_plans":
            0,

        "active_hop_rows":
            0,

        "active_prefixes":
            0,

        "edges_examined":
            0,

        "candidate_branches":
            0,

        "reachable_plans":
            0,

        "reachable_questions":
            0,
    }


    for rec in tqdm(
        planning_rows,
        desc=(
            f"{dataset_name} RoG fidelity"
        )
    ):

        adjacency = (
            build_exact_rog_adjacency(
                rec[
                    "graph"
                ]
            )
        )


        topic_entities = (
            as_entity_list(
                rec[
                    "q_entity"
                ]
            )
        )


        gold_answers = (
            as_entity_list(
                rec[
                    "a_entity"
                ]
            )
        )


        plans = (
            normalize_relation_plans(
                rec[
                    "predicted_paths"
                ]
            )
        )


        totals[
            "total_predicted_plans"
        ] += len(
            plans
        )


        question_reachable = False


        for plan in plans:

            if len(
                plan
            ) == 0:

                totals[
                    "empty_plans"
                ] += 1

                continue


            totals[
                "nonempty_plans"
            ] += 1


            result = (
                traverse_rog_unpruned(
                    adjacency=
                        adjacency,

                    topic_entities=
                        topic_entities,

                    plan=
                        plan,

                    gold_answers=
                        gold_answers
                )
            )


            totals[
                "active_hop_rows"
            ] += result[
                "active_hop_rows"
            ]


            totals[
                "active_prefixes"
            ] += result[
                "active_prefixes"
            ]


            totals[
                "edges_examined"
            ] += result[
                "edges_examined"
            ]


            totals[
                "candidate_branches"
            ] += result[
                "candidate_branches"
            ]


            if result[
                "reachable"
            ]:

                totals[
                    "reachable_plans"
                ] += 1

                question_reachable = True


        if question_reachable:

            totals[
                "reachable_questions"
            ] += 1


    print(
        "\n"
        + "=" * 100
    )

    print(
        f"{dataset_name.upper()} RoG FIDELITY"
    )

    print(
        "=" * 100
    )


    for key, value in (
        totals.items()
    ):

        print(
            f"{key:<25}",
            value
        )


    fidelity_keys = [
        "questions",
        "total_predicted_plans",
        "nonempty_plans",
        "empty_plans",
        "active_hop_rows",
        "active_prefixes",
        "edges_examined",
        "candidate_branches",
        "reachable_plans",
        "reachable_questions",
    ]


    for key in fidelity_keys:

        assert (
            totals[
                key
            ]
            ==
            expected[
                key
            ]
        ), (
            f"{dataset_name}: RoG fidelity mismatch "
            f"for {key}: "
            f"{totals[key]} != {expected[key]}"
        )


    print(
        f"{dataset_name.upper()} RoG fidelity: PASSED"
    )


    return totals


# ======================================================================
# 11. RUN RoG FIDELITY GATES
# ======================================================================

webqsp_rog_reference = (
    rog_fidelity_gate(
        dataset_name=
            "webqsp",

        planning_rows=
            webqsp_val_plan_rows
    )
)


cwq_rog_reference = (
    rog_fidelity_gate(
        dataset_name=
            "cwq",

        planning_rows=
            cwq_val_plan_rows
    )
)


print(
    "\nExact RoG validation fidelity gate: PASSED"
)


# ======================================================================
# 12. BATCHED ONLINE VALIDATION SEMANTIC PREFILL
# ======================================================================
#
# All dynamic pruned candidate groups are subsets of the unpruned RoG
# candidate universe.
#
# We therefore collect validation semantic texts from the UNPRUNED
# traversal and batch-prefill MiniLM before tuning.
#
# This is only a speed optimization.
# ======================================================================

def collect_online_semantic_inventory(
    dataset_name,
    planning_rows,
    question_rows
):

    questions = {}
    entities = {}
    relations = {}
    plans_mapping = {}
    suffixes = {}


    assert len(
        planning_rows
    ) == len(
        question_rows
    )


    for source_index in tqdm(
        range(
            len(
                planning_rows
            )
        ),
        desc=(
            f"{dataset_name} semantic inventory"
        )
    ):

        plan_rec = (
            planning_rows[
                source_index
            ]
        )

        question_rec = (
            question_rows[
                source_index
            ]
        )


        assert str(
            plan_rec[
                "id"
            ]
        ) == str(
            question_rec[
                "id"
            ]
        )


        question_id = str(
            plan_rec[
                "id"
            ]
        )


        questions[
            question_id
        ] = str(
            question_rec[
                "question"
            ]
        )


        adjacency = (
            build_exact_rog_adjacency(
                plan_rec[
                    "graph"
                ]
            )
        )


        topic_entities = (
            as_entity_list(
                plan_rec[
                    "q_entity"
                ]
            )
        )


        for entity in topic_entities:

            if (
                has_readable_entity_surface_runtime(
                    entity
                )
            ):

                entities[
                    str(
                        entity
                    )
                ] = (
                    entity_surface_text_runtime(
                        entity
                    )
                )


        relation_plans = (
            normalize_relation_plans(
                plan_rec[
                    "predicted_paths"
                ]
            )
        )


        for plan in relation_plans:

            if len(
                plan
            ) == 0:

                continue


            # ----------------------------------------------------------
            # Relations
            # ----------------------------------------------------------

            for relation in plan:

                relations[
                    str(
                        relation
                    )
                ] = (
                    relation_surface_text_runtime(
                        relation
                    )
                )


            # ----------------------------------------------------------
            # Full plan
            # ----------------------------------------------------------

            plan_key = "||".join(
                str(
                    x
                )
                for x in plan
            )


            plans_mapping[
                plan_key
            ] = (
                relation_sequence_text_runtime(
                    plan
                )
            )


            # ----------------------------------------------------------
            # Every possible intermediate-hop remaining suffix
            # ----------------------------------------------------------

            for hop in range(
                max(
                    0,
                    len(
                        plan
                    )
                    - 1
                )
            ):

                suffix = plan[
                    hop + 1:
                ]


                suffix_key = "||".join(
                    str(
                        x
                    )
                    for x in suffix
                )


                suffixes[
                    suffix_key
                ] = (
                    relation_sequence_text_runtime(
                        suffix
                    )
                )


            # ----------------------------------------------------------
            # Unpruned relation-constrained entity universe
            # ----------------------------------------------------------

            active = [
                (
                    str(
                        entity
                    ),
                )
                for entity in topic_entities
            ]


            for target_relation in plan:

                if not active:

                    break


                candidates = []


                for prefix in active:

                    endpoint = prefix[
                        -1
                    ]


                    neighbors = adjacency.get(
                        endpoint,
                        {}
                    )


                    for neighbor, relation in (
                        neighbors.items()
                    ):

                        if relation != target_relation:

                            continue


                        candidates.append(
                            prefix
                            + (
                                neighbor,
                            )
                        )


                        if (
                            has_readable_entity_surface_runtime(
                                neighbor
                            )
                        ):

                            entities[
                                str(
                                    neighbor
                                )
                            ] = (
                                entity_surface_text_runtime(
                                    neighbor
                                )
                            )


                active = candidates


    return {
        "questions":
            questions,

        "entities":
            entities,

        "relations":
            relations,

        "plans":
            plans_mapping,

        "suffixes":
            suffixes,
    }


print(
    "\nCollecting complete validation runtime semantic universe..."
)


webqsp_online_inventory = (
    collect_online_semantic_inventory(
        dataset_name=
            "webqsp",

        planning_rows=
            webqsp_val_plan_rows,

        question_rows=
            webqsp_val_runtime
    )
)


cwq_online_inventory = (
    collect_online_semantic_inventory(
        dataset_name=
            "cwq",

        planning_rows=
            cwq_val_plan_rows,

        question_rows=
            cwq_val_runtime
    )
)


def merge_dicts_in_order(
    *dicts
):

    result = {}


    for mapping in dicts:

        result.update(
            mapping
        )


    return result


ALL_VALIDATION_QUESTIONS = (
    merge_dicts_in_order(
        webqsp_online_inventory[
            "questions"
        ],

        cwq_online_inventory[
            "questions"
        ]
    )
)


ALL_VALIDATION_ENTITIES = (
    merge_dicts_in_order(
        webqsp_online_inventory[
            "entities"
        ],

        cwq_online_inventory[
            "entities"
        ]
    )
)


ALL_VALIDATION_RELATIONS = (
    merge_dicts_in_order(
        webqsp_online_inventory[
            "relations"
        ],

        cwq_online_inventory[
            "relations"
        ]
    )
)


ALL_VALIDATION_PLANS = (
    merge_dicts_in_order(
        webqsp_online_inventory[
            "plans"
        ],

        cwq_online_inventory[
            "plans"
        ]
    )
)


ALL_VALIDATION_SUFFIXES = (
    merge_dicts_in_order(
        webqsp_online_inventory[
            "suffixes"
        ],

        cwq_online_inventory[
            "suffixes"
        ]
    )
)


print(
    "\nValidation online semantic inventory"
)

print(
    "  Questions:         ",
    len(
        ALL_VALIDATION_QUESTIONS
    )
)

print(
    "  Readable entities: ",
    len(
        ALL_VALIDATION_ENTITIES
    )
)

print(
    "  Relations:         ",
    len(
        ALL_VALIDATION_RELATIONS
    )
)

print(
    "  Plans:             ",
    len(
        ALL_VALIDATION_PLANS
    )
)

print(
    "  Suffixes:          ",
    len(
        ALL_VALIDATION_SUFFIXES
    )
)


print(
    "\nBatch-prefilling semantic runtime..."
)


for namespace, mapping in [
    (
        "question",
        ALL_VALIDATION_QUESTIONS
    ),

    (
        "entity",
        ALL_VALIDATION_ENTITIES
    ),

    (
        "relation",
        ALL_VALIDATION_RELATIONS
    ),

    (
        "plan",
        ALL_VALIDATION_PLANS
    ),

    (
        "suffix",
        ALL_VALIDATION_SUFFIXES
    ),
]:

    print(
        f"  {namespace:<10} "
        f"{len(mapping):>7}"
    )


    AFP_RUNTIME_SEMANTIC_ENCODER.prefill(
        namespace,
        mapping
    )


print(
    "Validation semantic prefill: READY"
)


# ======================================================================
# 13. NUMERICAL HELPERS
# ======================================================================

def stable_sigmoid(
    logits
):

    logits = np.asarray(
        logits,
        dtype=np.float64
    )


    output = np.empty_like(
        logits
    )


    positive = (
        logits >= 0
    )


    output[
        positive
    ] = (
        1.0
        /
        (
            1.0
            +
            np.exp(
                -logits[
                    positive
                ]
            )
        )
    )


    exp_values = np.exp(
        logits[
            ~positive
        ]
    )


    output[
        ~positive
    ] = (
        exp_values
        /
        (
            1.0
            +
            exp_values
        )
    )


    return output


def temperature_softmax(
    logits,
    temperature
):

    temperature = float(
        temperature
    )


    assert temperature > 0


    values = (
        np.asarray(
            logits,
            dtype=np.float64
        )
        /
        temperature
    )


    values = (
        values
        -
        np.max(
            values
        )
    )


    exp_values = np.exp(
        values
    )


    denominator = float(
        np.sum(
            exp_values
        )
    )


    assert denominator > 0
    assert np.isfinite(
        denominator
    )


    return (
        exp_values
        /
        denominator
    )


def normalized_entropy(
    probabilities
):

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64
    )


    n = len(
        probabilities
    )


    if n <= 1:

        return 0.0


    safe_probabilities = np.clip(
        probabilities,
        1e-12,
        1.0
    )


    entropy = -float(
        np.sum(
            safe_probabilities
            * np.log(
                safe_probabilities
            )
        )
    )


    denominator = math.log(
        n
    )


    if denominator <= 0:

        return 0.0


    return float(
        np.clip(
            entropy
            /
            denominator,
            0.0,
            1.0
        )
    )


# ======================================================================
# 14. TIE-AWARE HELPERS
# ======================================================================

def all_logits_tied(
    logits
):

    logits = np.asarray(
        logits,
        dtype=np.float64
    )


    if len(
        logits
    ) <= 1:

        return True


    return bool(
        (
            np.max(
                logits
            )
            -
            np.min(
                logits
            )
        )
        <= TIE_ATOL
    )


def stable_descending_order(
    logits
):

    return np.argsort(
        -np.asarray(
            logits,
            dtype=np.float64
        ),
        kind="stable"
    )


def tie_expanded_top_k_indices(
    logits,
    requested_k
):

    logits = np.asarray(
        logits,
        dtype=np.float64
    )


    n = len(
        logits
    )


    requested_k = int(
        requested_k
    )


    if requested_k >= n:

        return list(
            range(
                n
            )
        )


    assert requested_k >= 1


    order = (
        stable_descending_order(
            logits
        )
    )


    cutoff_index = order[
        requested_k
        - 1
    ]


    cutoff_score = logits[
        cutoff_index
    ]


    selected = []


    for index, score in enumerate(
        logits
    ):

        if (
            score
            >
            cutoff_score
            or
            np.isclose(
                score,
                cutoff_score,
                atol=TIE_ATOL,
                rtol=0.0
            )
        ):

            selected.append(
                index
            )


    return selected


# ======================================================================
# 15. POLICY SELECTION
# ======================================================================

def select_policy_indices(
    config,
    logits
):

    logits = np.asarray(
        logits,
        dtype=np.float64
    )


    n = len(
        logits
    )


    assert n > 1


    family = config[
        "family"
    ]


    # ------------------------------------------------------------------
    # Fixed Top-B
    # ------------------------------------------------------------------

    if family == "fixed_top_b":

        B = int(
            config[
                "B"
            ]
        )


        requested_budget = min(
            B,
            n
        )


        selected = (
            tie_expanded_top_k_indices(
                logits=
                    logits,

                requested_k=
                    requested_budget
            )
        )


        return {
            "selected_indices":
                selected,

            "requested_budget":
                requested_budget,

            "retained_count":
                len(
                    selected
                ),

            "uncertainty":
                None,

            "gamma":
                None,

            "fully_tied":
                all_logits_tied(
                    logits
                ),
        }


    # ------------------------------------------------------------------
    # Fixed Threshold
    # ------------------------------------------------------------------

    if family == "fixed_threshold":

        tau = float(
            config[
                "tau"
            ]
        )


        probabilities = (
            stable_sigmoid(
                logits
            )
        )


        selected = [
            index
            for index, probability
            in enumerate(
                probabilities
            )
            if probability
            >= tau
        ]


        # No forced top-1.
        return {
            "selected_indices":
                selected,

            "requested_budget":
                None,

            "retained_count":
                len(
                    selected
                ),

            "uncertainty":
                None,

            "gamma":
                None,

            "fully_tied":
                all_logits_tied(
                    logits
                ),
        }


    # ------------------------------------------------------------------
    # AFP
    # ------------------------------------------------------------------

    assert family == "afp"


    T = float(
        config[
            "T"
        ]
    )


    gamma_min = float(
        config[
            "gamma_min"
        ]
    )


    assert 0.0 <= gamma_min <= 1.0


    # Explicit abstention under complete score tie.
    if all_logits_tied(
        logits
    ):

        return {
            "selected_indices":
                list(
                    range(
                        n
                    )
                ),

            "requested_budget":
                n,

            "retained_count":
                n,

            "uncertainty":
                1.0,

            "gamma":
                1.0,

            "fully_tied":
                True,
        }


    probabilities = (
        temperature_softmax(
            logits=
                logits,

            temperature=
                T
        )
    )


    uncertainty = (
        normalized_entropy(
            probabilities
        )
    )


    gamma = (
        gamma_min
        +
        uncertainty
        * (
            1.0
            -
            gamma_min
        )
    )


    order = (
        stable_descending_order(
            logits
        )
    )


    sorted_probabilities = (
        probabilities[
            order
        ]
    )


    cumulative = np.cumsum(
        sorted_probabilities
    )


    requested_budget = int(
        np.searchsorted(
            cumulative,
            gamma,
            side="left"
        )
        + 1
    )


    requested_budget = min(
        max(
            requested_budget,
            1
        ),
        n
    )


    selected = (
        tie_expanded_top_k_indices(
            logits=
                logits,

            requested_k=
                requested_budget
        )
    )


    return {
        "selected_indices":
            selected,

        "requested_budget":
            requested_budget,

        "retained_count":
            len(
                selected
            ),

        "uncertainty":
            float(
                uncertainty
            ),

        "gamma":
            float(
                gamma
            ),

        "fully_tied":
            False,
    }


# ======================================================================
# 16. RELATION-EXPANSION CACHE
# ======================================================================
#
# The same dynamic frontier may be encountered by multiple tuning
# configurations.
#
# We cache the actual candidate construction once.
#
# IMPORTANT:
# edges_cost is still added independently to EVERY method, so measured
# graph-search cost remains method-correct.
# ======================================================================

def frontier_cache_key(
    plan_index,
    hop,
    active_prefixes
):

    return (
        int(
            plan_index
        ),

        int(
            hop
        ),

        tuple(
            tuple(
                prefix
            )
            for prefix in active_prefixes
        ),
    )


def get_cached_relation_expansion(
    adjacency,
    plan_index,
    hop,
    target_relation,
    active_prefixes,
    expansion_cache
):

    key = (
        frontier_cache_key(
            plan_index=
                plan_index,

            hop=
                hop,

            active_prefixes=
                active_prefixes
        )
    )


    if key in expansion_cache:

        return expansion_cache[
            key
        ]


    candidates = []
    candidate_rows = []

    edges_cost = 0


    for parent_index, prefix in enumerate(
        active_prefixes
    ):

        endpoint = prefix[
            -1
        ]


        neighbors = adjacency.get(
            endpoint,
            {}
        )


        edges_cost += len(
            neighbors
        )


        for neighbor, relation in (
            neighbors.items()
        ):

            if relation != target_relation:

                continue


            candidate_prefix = (
                prefix
                + (
                    neighbor,
                )
            )


            candidates.append(
                candidate_prefix
            )


            candidate_rows.append(
                {
                    "prefix_entities":
                        list(
                            prefix
                        ),

                    "candidate_entity":
                        neighbor,

                    "parent_prefix_index":
                        int(
                            parent_index
                        ),
                }
            )


    result = {
        "candidates":
            candidates,

        "candidate_rows":
            candidate_rows,

        "edges_cost":
            int(
                edges_cost
            ),
    }


    expansion_cache[
        key
    ] = result


    return result


# ======================================================================
# 17. ONLINE SCORER CACHE
# ======================================================================

def get_group_logits(
    dataset_name,
    question_id,
    question,
    plan_index,
    plan,
    hop,
    active_prefixes,
    candidate_rows,
    score_cache
):

    key = (
        frontier_cache_key(
            plan_index=
                plan_index,

            hop=
                hop,

            active_prefixes=
                active_prefixes
        )
    )


    if key in score_cache:

        cached = score_cache[
            key
        ]


        assert (
            cached[
                "candidate_count"
            ]
            ==
            len(
                candidate_rows
            )
        )


        return cached[
            "logits"
        ]


    output = (
        AFP_RUNTIME_SCORE_GROUP(
            dataset_name=
                dataset_name,

            question_id=
                question_id,

            question=
                question,

            plan=
                plan,

            hop=
                hop,

            candidate_rows=
                candidate_rows
        )
    )


    logits = np.asarray(
        output[
            "logits"
        ],
        dtype=np.float32
    )


    assert logits.shape == (
        len(
            candidate_rows
        ),
    )


    score_cache[
        key
    ] = {
        "candidate_count":
            len(
                candidate_rows
            ),

        "logits":
            logits,
    }


    return logits


# ======================================================================
# 18. POLICY-AWARE ONLINE TRAVERSAL
# ======================================================================

def traverse_with_policy(
    dataset_name,
    question_id,
    question,
    adjacency,
    topic_entities,
    plan_index,
    plan,
    config,
    expansion_cache,
    score_cache
):

    assert len(
        plan
    ) > 0


    active = [
        (
            str(
                entity
            ),
        )
        for entity in topic_entities
    ]


    edges_examined = 0
    candidate_branches = 0

    active_hop_rows = 0
    active_prefixes_total = 0

    decision_hops = 0

    retained_after_decision = 0

    requested_budget_total = 0
    requested_budget_observations = 0

    uncertainty_total = 0.0
    uncertainty_observations = 0

    fully_tied_decisions = 0

    peak_frontier = len(
        active
    )


    L = len(
        plan
    )


    for hop, target_relation in enumerate(
        plan
    ):

        if not active:

            break


        active_hop_rows += 1

        active_prefixes_total += len(
            active
        )


        expansion = (
            get_cached_relation_expansion(
                adjacency=
                    adjacency,

                plan_index=
                    plan_index,

                hop=
                    hop,

                target_relation=
                    target_relation,

                active_prefixes=
                    active,

                expansion_cache=
                    expansion_cache
            )
        )


        candidates = expansion[
            "candidates"
        ]


        candidate_rows = expansion[
            "candidate_rows"
        ]


        # Even though expansion computation is cached, this is the
        # graph work THIS method would perform.
        edges_examined += expansion[
            "edges_cost"
        ]


        candidate_branches += len(
            candidates
        )


        if not candidates:

            active = []

            break


        # --------------------------------------------------------------
        # FINAL-HOP PROTECTION
        # --------------------------------------------------------------

        is_final_hop = (
            hop
            ==
            L - 1
        )


        if is_final_hop:

            active = candidates


            peak_frontier = max(
                peak_frontier,
                len(
                    active
                )
            )


            continue


        # --------------------------------------------------------------
        # SINGLETON BYPASS
        # --------------------------------------------------------------

        if len(
            candidates
        ) <= 1:

            active = candidates


            peak_frontier = max(
                peak_frontier,
                len(
                    active
                )
            )


            continue


        # --------------------------------------------------------------
        # INTERMEDIATE DECISION
        # --------------------------------------------------------------

        decision_hops += 1


        logits = (
            get_group_logits(
                dataset_name=
                    dataset_name,

                question_id=
                    question_id,

                question=
                    question,

                plan_index=
                    plan_index,

                plan=
                    plan,

                hop=
                    hop,

                active_prefixes=
                    active,

                candidate_rows=
                    candidate_rows,

                score_cache=
                    score_cache
            )
        )


        selection = (
            select_policy_indices(
                config=
                    config,

                logits=
                    logits
            )
        )


        selected_indices = (
            selection[
                "selected_indices"
            ]
        )


        active = [
            candidates[
                index
            ]
            for index in selected_indices
        ]


        retained_after_decision += len(
            active
        )


        if (
            selection[
                "requested_budget"
            ]
            is not None
        ):

            requested_budget_total += int(
                selection[
                    "requested_budget"
                ]
            )


            requested_budget_observations += 1


        if (
            selection[
                "uncertainty"
            ]
            is not None
        ):

            uncertainty_total += float(
                selection[
                    "uncertainty"
                ]
            )


            uncertainty_observations += 1


        if selection[
            "fully_tied"
        ]:

            fully_tied_decisions += 1


        peak_frontier = max(
            peak_frontier,
            len(
                active
            )
        )


    return {
        "final_prefixes":
            active,

        "active_hop_rows":
            int(
                active_hop_rows
            ),

        "active_prefixes":
            int(
                active_prefixes_total
            ),

        "edges_examined":
            int(
                edges_examined
            ),

        "candidate_branches":
            int(
                candidate_branches
            ),

        "decision_hops":
            int(
                decision_hops
            ),

        "retained_after_decision":
            int(
                retained_after_decision
            ),

        "requested_budget_total":
            int(
                requested_budget_total
            ),

        "requested_budget_observations":
            int(
                requested_budget_observations
            ),

        "uncertainty_total":
            float(
                uncertainty_total
            ),

        "uncertainty_observations":
            int(
                uncertainty_observations
            ),

        "fully_tied_decisions":
            int(
                fully_tied_decisions
            ),

        "peak_frontier":
            int(
                peak_frontier
            ),
    }


# ======================================================================
# 19. GOLD-ONLY POST-TRAVERSAL REACHABILITY
# ======================================================================
#
# Gold is deliberately kept OUT of traverse_with_policy().
# ======================================================================

def final_prefixes_reach_answer(
    final_prefixes,
    gold_answers
):

    if not final_prefixes:

        return False


    answer_set = {
        str(
            answer
        )
        for answer in gold_answers
    }


    return any(
        prefix[
            -1
        ]
        in answer_set

        for prefix in final_prefixes
    )


# ======================================================================
# 20. INITIAL CONFIG STATS
# ======================================================================

def fresh_config_stats():

    return {
        config[
            "config_id"
        ]: {
            "active_hop_rows":
                0,

            "active_prefixes":
                0,

            "edges_examined":
                0,

            "candidate_branches":
                0,

            "reachable_plans":
                0,

            "reachable_questions":
                0,

            "decision_hops":
                0,

            "retained_after_decision":
                0,

            "requested_budget_total":
                0,

            "requested_budget_observations":
                0,

            "uncertainty_total":
                0.0,

            "uncertainty_observations":
                0,

            "fully_tied_decisions":
                0,

            "peak_frontier":
                0,
        }

        for config in CONFIGS
    }


# ======================================================================
# 21. RESUMABLE VALIDATION SWEEP
# ======================================================================

def tune_dataset(
    dataset_name,
    planning_rows,
    question_rows,
    rog_reference
):

    assert len(
        planning_rows
    ) == len(
        question_rows
    )


    # Version-specific resume file prevents accidental reuse of any
    # older Cell-13 state.
    resume_path = (
        TUNING_DIR
        / (
            f"{dataset_name}_"
            "cell13_v2_metric_corrected_resume.pkl"
        )
    )


    if resume_path.exists():

        with open(
            resume_path,
            "rb"
        ) as f:

            state = pickle.load(
                f
            )


        assert (
            state[
                "tuning_spec_sha256"
            ]
            ==
            TUNING_SPEC_SHA
        ), (
            "Resume file belongs to a different tuning specification."
        )


        assert (
            state[
                "dataset"
            ]
            ==
            dataset_name
        )


        start_index = int(
            state[
                "next_index"
            ]
        )


        stats = state[
            "stats"
        ]


        print(
            f"\n{dataset_name.upper()} resume state detected."
        )


        print(
            "Resuming from question:",
            start_index,
            "/",
            len(
                planning_rows
            )
        )


    else:

        start_index = 0

        stats = (
            fresh_config_stats()
        )


    start_time = time.time()


    for source_index in tqdm(
        range(
            start_index,
            len(
                planning_rows
            )
        ),
        desc=(
            f"{dataset_name} tuning"
        )
    ):

        plan_rec = (
            planning_rows[
                source_index
            ]
        )


        question_rec = (
            question_rows[
                source_index
            ]
        )


        assert str(
            plan_rec[
                "id"
            ]
        ) == str(
            question_rec[
                "id"
            ]
        )


        question_id = str(
            plan_rec[
                "id"
            ]
        )


        question = str(
            question_rec[
                "question"
            ]
        )


        topic_entities = (
            as_entity_list(
                plan_rec[
                    "q_entity"
                ]
            )
        )


        gold_answers = (
            as_entity_list(
                plan_rec[
                    "a_entity"
                ]
            )
        )


        plans = (
            normalize_relation_plans(
                plan_rec[
                    "predicted_paths"
                ]
            )
        )


        adjacency = (
            build_exact_rog_adjacency(
                plan_rec[
                    "graph"
                ]
            )
        )


        # --------------------------------------------------------------
        # Question-local computational caches.
        # --------------------------------------------------------------

        expansion_cache = {}

        score_cache = {}


        question_reachability = {
            config[
                "config_id"
            ]:
                False

            for config in CONFIGS
        }


        for plan_index, plan in enumerate(
            plans
        ):

            if len(
                plan
            ) == 0:

                continue


            for config in CONFIGS:

                config_id = (
                    config[
                        "config_id"
                    ]
                )


                result = (
                    traverse_with_policy(
                        dataset_name=
                            dataset_name,

                        question_id=
                            question_id,

                        question=
                            question,

                        adjacency=
                            adjacency,

                        topic_entities=
                            topic_entities,

                        plan_index=
                            plan_index,

                        plan=
                            plan,

                        config=
                            config,

                        expansion_cache=
                            expansion_cache,

                        score_cache=
                            score_cache
                    )
                )


                s = stats[
                    config_id
                ]


                s[
                    "active_hop_rows"
                ] += result[
                    "active_hop_rows"
                ]


                s[
                    "active_prefixes"
                ] += result[
                    "active_prefixes"
                ]


                s[
                    "edges_examined"
                ] += result[
                    "edges_examined"
                ]


                s[
                    "candidate_branches"
                ] += result[
                    "candidate_branches"
                ]


                s[
                    "decision_hops"
                ] += result[
                    "decision_hops"
                ]


                s[
                    "retained_after_decision"
                ] += result[
                    "retained_after_decision"
                ]


                s[
                    "requested_budget_total"
                ] += result[
                    "requested_budget_total"
                ]


                s[
                    "requested_budget_observations"
                ] += result[
                    "requested_budget_observations"
                ]


                s[
                    "uncertainty_total"
                ] += result[
                    "uncertainty_total"
                ]


                s[
                    "uncertainty_observations"
                ] += result[
                    "uncertainty_observations"
                ]


                s[
                    "fully_tied_decisions"
                ] += result[
                    "fully_tied_decisions"
                ]


                s[
                    "peak_frontier"
                ] = max(
                    s[
                        "peak_frontier"
                    ],
                    result[
                        "peak_frontier"
                    ]
                )


                # ------------------------------------------------------
                # GOLD USED ONLY HERE, AFTER TRAVERSAL.
                # ------------------------------------------------------

                reachable = (
                    final_prefixes_reach_answer(
                        final_prefixes=
                            result[
                                "final_prefixes"
                            ],

                        gold_answers=
                            gold_answers
                    )
                )


                if reachable:

                    s[
                        "reachable_plans"
                    ] += 1


                    question_reachability[
                        config_id
                    ] = True


        # --------------------------------------------------------------
        # Question-level Answer Retention indicator.
        # --------------------------------------------------------------

        for config_id, reachable in (
            question_reachability.items()
        ):

            if reachable:

                stats[
                    config_id
                ][
                    "reachable_questions"
                ] += 1


        # --------------------------------------------------------------
        # RESUME CHECKPOINT EVERY 100 QUESTIONS
        # --------------------------------------------------------------

        if (
            (
                source_index
                + 1
            )
            % 100
            == 0
            or
            source_index
            ==
            (
                len(
                    planning_rows
                )
                - 1
            )
        ):

            resume_state = {
                "dataset":
                    dataset_name,

                "tuning_spec_sha256":
                    TUNING_SPEC_SHA,

                "next_index":
                    int(
                        source_index
                        + 1
                    ),

                "stats":
                    stats,
            }


            with open(
                resume_path,
                "wb"
            ) as f:

                pickle.dump(
                    resume_state,
                    f
                )


    elapsed = (
        time.time()
        -
        start_time
    )


    print(
        f"\n{dataset_name.upper()} tuning sweep completed."
    )


    print(
        f"Elapsed this run: {elapsed/60:.2f} min"
    )


    # ==================================================================
    # AGGREGATED RESULT TABLE
    # ==================================================================

    rows = []


    rog_edges = float(
        rog_reference[
            "edges_examined"
        ]
    )


    rog_reachable_questions = int(
        rog_reference[
            "reachable_questions"
        ]
    )


    assert rog_edges > 0
    assert rog_reachable_questions > 0


    for config in CONFIGS:

        config_id = (
            config[
                "config_id"
            ]
        )


        s = stats[
            config_id
        ]


        edges = int(
            s[
                "edges_examined"
            ]
        )


        reachable_questions = int(
            s[
                "reachable_questions"
            ]
        )


        ssr = (
            1.0
            -
            edges
            /
            rog_edges
        )


        answer_retention = (
            reachable_questions
            /
            rog_reachable_questions
        )


        avg_requested_budget = (
            s[
                "requested_budget_total"
            ]
            /
            s[
                "requested_budget_observations"
            ]

            if
            s[
                "requested_budget_observations"
            ]
            > 0

            else
            np.nan
        )


        avg_uncertainty = (
            s[
                "uncertainty_total"
            ]
            /
            s[
                "uncertainty_observations"
            ]

            if
            s[
                "uncertainty_observations"
            ]
            > 0

            else
            np.nan
        )


        rows.append(
            {
                "dataset":
                    dataset_name,

                "config_id":
                    config_id,

                "family":
                    config[
                        "family"
                    ],

                "T":
                    config.get(
                        "T",
                        np.nan
                    ),

                "gamma_min":
                    config.get(
                        "gamma_min",
                        np.nan
                    ),

                "B":
                    config.get(
                        "B",
                        np.nan
                    ),

                "tau":
                    config.get(
                        "tau",
                        np.nan
                    ),

                "active_hop_rows":
                    int(
                        s[
                            "active_hop_rows"
                        ]
                    ),

                "active_prefixes":
                    int(
                        s[
                            "active_prefixes"
                        ]
                    ),

                "edges_examined":
                    edges,

                "rog_edges_examined":
                    int(
                        rog_reference[
                            "edges_examined"
                        ]
                    ),

                "ssr":
                    float(
                        ssr
                    ),

                "reachable_questions":
                    reachable_questions,

                "rog_reachable_questions":
                    rog_reachable_questions,

                "answer_retention":
                    float(
                        answer_retention
                    ),

                "coverage_all_questions":
                    float(
                        reachable_questions
                        /
                        len(
                            planning_rows
                        )
                    ),

                "reachable_plans":
                    int(
                        s[
                            "reachable_plans"
                        ]
                    ),

                "candidate_branches":
                    int(
                        s[
                            "candidate_branches"
                        ]
                    ),

                "decision_hops":
                    int(
                        s[
                            "decision_hops"
                        ]
                    ),

                "retained_after_decision":
                    int(
                        s[
                            "retained_after_decision"
                        ]
                    ),

                "avg_requested_budget":
                    float(
                        avg_requested_budget
                    ),

                "avg_uncertainty":
                    float(
                        avg_uncertainty
                    ),

                "fully_tied_decisions":
                    int(
                        s[
                            "fully_tied_decisions"
                        ]
                    ),

                "peak_frontier":
                    int(
                        s[
                            "peak_frontier"
                        ]
                    ),

                "ar_floor_feasible":
                    bool(
                        answer_retention
                        >= AR_FLOOR
                    ),
            }
        )


    return (
        pd.DataFrame(
            rows
        ),
        stats
    )


# ======================================================================
# 22. START VALIDATION TUNING
# ======================================================================

print(
    "\n"
    + "=" * 112
)

print(
    "STARTING VALIDATION-ONLY CONFIGURATION SWEEP"
)

print(
    "=" * 112
)


webqsp_tuning_df, webqsp_tuning_stats = (
    tune_dataset(
        dataset_name=
            "webqsp",

        planning_rows=
            webqsp_val_plan_rows,

        question_rows=
            webqsp_val_runtime,

        rog_reference=
            webqsp_rog_reference
    )
)


cwq_tuning_df, cwq_tuning_stats = (
    tune_dataset(
        dataset_name=
            "cwq",

        planning_rows=
            cwq_val_plan_rows,

        question_rows=
            cwq_val_runtime,

        rog_reference=
            cwq_rog_reference
    )
)


# ======================================================================
# 23. PREDECLARED CONFIGURATION SELECTION
# ======================================================================

def select_family_config(
    result_df,
    family
):

    subset = (
        result_df[
            result_df[
                "family"
            ]
            ==
            family
        ]
        .copy()
    )


    assert len(
        subset
    ) > 0


    feasible = (
        subset[
            subset[
                "answer_retention"
            ]
            >= AR_FLOOR
        ]
        .copy()
    )


    if len(
        feasible
    ) > 0:

        selection_mode = (
            "maximize_ssr_subject_to_ar_floor"
        )


        ranked = feasible.sort_values(
            by=[
                "ssr",
                "answer_retention",
                "config_id",
            ],

            ascending=[
                False,
                False,
                True,
            ],

            kind="stable"
        )


    else:

        selection_mode = (
            "fallback_maximize_ar_then_ssr"
        )


        ranked = subset.sort_values(
            by=[
                "answer_retention",
                "ssr",
                "config_id",
            ],

            ascending=[
                False,
                False,
                True,
            ],

            kind="stable"
        )


    selected = (
        ranked.iloc[
            0
        ].to_dict()
    )


    selected[
        "selection_mode"
    ] = selection_mode


    return (
        selected,
        ranked
    )


def select_dataset_configs(
    result_df
):

    selected = {}
    ranked = {}


    for family in [
        "afp",
        "fixed_top_b",
        "fixed_threshold",
    ]:

        (
            selected_family,
            ranked_family
        ) = (
            select_family_config(
                result_df=
                    result_df,

                family=
                    family
            )
        )


        selected[
            family
        ] = selected_family


        ranked[
            family
        ] = ranked_family


    return (
        selected,
        ranked
    )


(
    webqsp_selected,
    webqsp_ranked
) = (
    select_dataset_configs(
        webqsp_tuning_df
    )
)


(
    cwq_selected,
    cwq_ranked
) = (
    select_dataset_configs(
        cwq_tuning_df
    )
)


# ======================================================================
# 24. DISPLAY RESULTS
# ======================================================================

DISPLAY_COLUMNS = [
    "config_id",
    "family",
    "T",
    "gamma_min",
    "B",
    "tau",
    "ssr",
    "answer_retention",
    "reachable_questions",
    "edges_examined",
    "active_hop_rows",
    "active_prefixes",
    "decision_hops",
    "avg_requested_budget",
    "avg_uncertainty",
    "fully_tied_decisions",
    "ar_floor_feasible",
]


def show_family_results(
    dataset_name,
    result_df
):

    print(
        "\n"
        + "=" * 112
    )

    print(
        f"{dataset_name.upper()} VALIDATION TUNING RESULTS"
    )

    print(
        "=" * 112
    )


    for family in [
        "afp",
        "fixed_top_b",
        "fixed_threshold",
    ]:

        subset = (
            result_df[
                result_df[
                    "family"
                ]
                ==
                family
            ]
            .sort_values(
                [
                    "answer_retention",
                    "ssr",
                ],

                ascending=[
                    False,
                    False,
                ]
            )
        )


        print(
            f"\n--- {family} ---"
        )


        print(
            subset[
                DISPLAY_COLUMNS
            ].to_string(
                index=False,

                float_format=lambda x:
                    f"{x:.6f}"
            )
        )


show_family_results(
    "webqsp",
    webqsp_tuning_df
)


show_family_results(
    "cwq",
    cwq_tuning_df
)


# ======================================================================
# 25. DISPLAY SELECTED CONFIGURATIONS
# ======================================================================

def print_selected(
    dataset_name,
    selected
):

    print(
        "\n"
        + "=" * 112
    )

    print(
        f"{dataset_name.upper()} "
        "SELECTED DEVELOPMENT CONFIGURATIONS"
    )

    print(
        "=" * 112
    )


    for family in [
        "afp",
        "fixed_top_b",
        "fixed_threshold",
    ]:

        row = selected[
            family
        ]


        print(
            f"\n{family}"
        )


        print(
            "  config:",
            row[
                "config_id"
            ]
        )


        print(
            "  selection mode:",
            row[
                "selection_mode"
            ]
        )


        print(
            "  AR:",
            f"{row['answer_retention']:.6f}"
        )


        print(
            "  SSR:",
            f"{row['ssr']:.6f}"
        )


        print(
            "  edges:",
            int(
                row[
                    "edges_examined"
                ]
            )
        )


        if family == "afp":

            print(
                "  T:",
                row[
                    "T"
                ]
            )


            print(
                "  gamma_min:",
                row[
                    "gamma_min"
                ]
            )


        elif family == "fixed_top_b":

            print(
                "  B:",
                int(
                    row[
                        "B"
                    ]
                )
            )


        elif family == "fixed_threshold":

            print(
                "  tau:",
                row[
                    "tau"
                ]
            )


print_selected(
    "webqsp",
    webqsp_selected
)


print_selected(
    "cwq",
    cwq_selected
)


# ======================================================================
# 26. SELECTED-CONFIG GRID SANITY
# ======================================================================

for dataset_name, selected in [
    (
        "WebQSP",
        webqsp_selected
    ),

    (
        "CWQ",
        cwq_selected
    ),
]:

    selected_afp = selected[
        "afp"
    ]


    assert float(
        selected_afp[
            "T"
        ]
    ) in T_GRID


    assert float(
        selected_afp[
            "gamma_min"
        ]
    ) in GAMMA_MIN_GRID


    assert int(
        selected[
            "fixed_top_b"
        ][
            "B"
        ]
    ) in TOP_B_GRID


    assert float(
        selected[
            "fixed_threshold"
        ][
            "tau"
        ]
    ) in THRESHOLD_GRID


print(
    "\nSelected configurations belong to "
    "predeclared grids: PASSED"
)


# ======================================================================
# 27. EXPORT FULL TUNING CSV
# ======================================================================

WEBQSP_TUNING_CSV = (
    TUNING_DIR
    / "webqsp_validation_tuning_v2.csv"
)


CWQ_TUNING_CSV = (
    TUNING_DIR
    / "cwq_validation_tuning_v2.csv"
)


webqsp_tuning_df.to_csv(
    WEBQSP_TUNING_CSV,
    index=False
)


cwq_tuning_df.to_csv(
    CWQ_TUNING_CSV,
    index=False
)


# ======================================================================
# 28. JSON-SAFE SELECTED RESULT
# ======================================================================

def json_safe_selected(
    selected
):

    output = {}


    for family, row in (
        selected.items()
    ):

        clean = {}


        for key, value in (
            row.items()
        ):

            if isinstance(
                value,
                np.integer
            ):

                clean[
                    key
                ] = int(
                    value
                )


            elif isinstance(
                value,
                np.floating
            ):

                if np.isnan(
                    value
                ):

                    clean[
                        key
                    ] = None

                else:

                    clean[
                        key
                    ] = float(
                        value
                    )


            elif isinstance(
                value,
                np.bool_
            ):

                clean[
                    key
                ] = bool(
                    value
                )


            elif (
                isinstance(
                    value,
                    float
                )
                and
                math.isnan(
                    value
                )
            ):

                clean[
                    key
                ] = None


            else:

                clean[
                    key
                ] = value


        output[
            family
        ] = clean


    return output


# ======================================================================
# 29. DEVELOPMENT-SELECTION MANIFEST
# ======================================================================

CELL13_MANIFEST = {
    "cell":
        "RQ2_CELL13_VALIDATION_TUNING_REVISED",

    "version":
        "rq2_cell13_validation_tuning_v2_metric_corrected",

    "tuning_spec_sha256":
        TUNING_SPEC_SHA,

    "selection_rule":
        TUNING_SPEC[
            "selection_rule"
        ],

    "metric_definitions":
        TUNING_SPEC[
            "metric_definitions"
        ],

    "grids": {
        "T":
            T_GRID,

        "gamma_min":
            GAMMA_MIN_GRID,

        "top_B":
            TOP_B_GRID,

        "threshold":
            THRESHOLD_GRID,
    },

    "rog_validation_fidelity": {
        "webqsp":
            webqsp_rog_reference,

        "cwq":
            cwq_rog_reference,
    },

    "selected": {
        "webqsp":
            json_safe_selected(
                webqsp_selected
            ),

        "cwq":
            json_safe_selected(
                cwq_selected
            ),
    },

    "random_b_policy": {
        "independently_tuned":
            False,

        "budget_source":
            "selected_fixed_top_b",
    },

    "adaptive_budget_random_policy": {
        "independently_tuned":
            False,

        "budget_source":
            "selected_afp_dynamic_budget",
    },

    "selected_scorer_checkpoint_sha256": {
        "webqsp":
            webqsp_ckpt_sha,

        "cwq":
            cwq_ckpt_sha,
    },

    "feature_version":
        AFP_RUNTIME_FEATURE_VERSION,

    "final_hop_protection":
        True,

    "singleton_bypass":
        True,

    "gold_used_for": [
        "validation_answer_retention_after_traversal_only"
    ],

    "gold_used_by_scorer":
        False,

    "gold_used_by_selector":
        False,

    "test_examples_accessed":
        False,

    "complete_afp_frozen":
        False,

    "next_step":
        "RQ2_Cell14_validation_controlled_comparison",
}


CELL13_MANIFEST_PATH = (
    TUNING_DIR
    / "cell13_validation_tuning_v2_manifest.json"
)


with open(
    CELL13_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        CELL13_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False
    )


# ======================================================================
# 30. EXPOSE SELECTED PARAMETERS FOR CELL 14
# ======================================================================

AFP_VALIDATION_SELECTED = {
    "webqsp": {
        "T":
            float(
                webqsp_selected[
                    "afp"
                ][
                    "T"
                ]
            ),

        "gamma_min":
            float(
                webqsp_selected[
                    "afp"
                ][
                    "gamma_min"
                ]
            ),
    },

    "cwq": {
        "T":
            float(
                cwq_selected[
                    "afp"
                ][
                    "T"
                ]
            ),

        "gamma_min":
            float(
                cwq_selected[
                    "afp"
                ][
                    "gamma_min"
                ]
            ),
    },
}


FIXED_TOP_B_VALIDATION_SELECTED = {
    "webqsp":
        int(
            webqsp_selected[
                "fixed_top_b"
            ][
                "B"
            ]
        ),

    "cwq":
        int(
            cwq_selected[
                "fixed_top_b"
            ][
                "B"
            ]
        ),
}


FIXED_THRESHOLD_VALIDATION_SELECTED = {
    "webqsp":
        float(
            webqsp_selected[
                "fixed_threshold"
            ][
                "tau"
            ]
        ),

    "cwq":
        float(
            cwq_selected[
                "fixed_threshold"
            ][
                "tau"
            ]
        ),
}


CELL13_TUNING_COMPLETE = True


# ======================================================================
# 31. FINAL REPORT
# ======================================================================

print(
    "\n"
    + "=" * 118
)

print(
    "=== RQ2 CELL 13: "
    "VALIDATION-ONLY HYPERPARAMETER TUNING COMPLETE ==="
)

print(
    "=" * 118
)


print(
    "\nMetric correction:"
)

print(
    "  active_hop_rows != active_prefixes"
)

print(
    "  WebQSP reference: 971 hop rows / 2440 prefixes"
)

print(
    "  CWQ reference:    16564 hop rows / 57841 prefixes"
)


print(
    "\nSelection criterion:"
)

print(
    f"  maximize SSR subject to AR >= {AR_FLOOR}"
)

print(
    "  fallback: maximize AR, then SSR"
)


print(
    "\nSelected AFP:"
)

print(
    "  WebQSP:",
    AFP_VALIDATION_SELECTED[
        "webqsp"
    ]
)

print(
    "  CWQ:   ",
    AFP_VALIDATION_SELECTED[
        "cwq"
    ]
)


print(
    "\nSelected Fixed Top-B:"
)

print(
    "  WebQSP:",
    FIXED_TOP_B_VALIDATION_SELECTED[
        "webqsp"
    ]
)

print(
    "  CWQ:   ",
    FIXED_TOP_B_VALIDATION_SELECTED[
        "cwq"
    ]
)


print(
    "\nSelected Fixed Threshold:"
)

print(
    "  WebQSP:",
    FIXED_THRESHOLD_VALIDATION_SELECTED[
        "webqsp"
    ]
)

print(
    "  CWQ:   ",
    FIXED_THRESHOLD_VALIDATION_SELECTED[
        "cwq"
    ]
)


print(
    "\nRandom baselines:"
)

print(
    "  Random-B inherits selected Fixed Top-B B."
)

print(
    "  Adaptive-Budget Random inherits selected AFP budgets."
)


print(
    "\nDevelopment status:"
)

print(
    "  Feature-v2:                   FROZEN"
)

print(
    "  Scorer architecture/weights: FROZEN"
)

print(
    "  Selector hyperparameters:     VALIDATION-SELECTED"
)

print(
    "  Controlled comparison:        NEXT"
)

print(
    "  Ablations:                    NOT YET"
)

print(
    "  TEST examples accessed:       NO"
)

print(
    "  Complete AFP frozen:          NO"
)


print(
    "\nOutputs:"
)

print(
    " ",
    WEBQSP_TUNING_CSV
)

print(
    " ",
    CWQ_TUNING_CSV
)

print(
    " ",
    CELL13_MANIFEST_PATH
)


print(
    "\nNEXT STEP:"
)

print(
    "Cell 14 — validation controlled comparison:"
)

print(
    "RoG vs Fixed Top-B vs Fixed Threshold "
    "vs Random-B vs Adaptive-Budget Random vs AFP."
)

Cell 13 prerequisites: PASSED
Runtime scorer: READY
Runtime device: CPU
Feature version: afp_features_v2_masked_entity_semantics

PREDECLARED VALIDATION MODEL-SELECTION RULE
Primary objective: maximize SSR subject to AR >= 0.99
Fallback: maximize AR, then SSR, if no configuration satisfies AR floor.

AFP grid
  T:          [0.5, 1.0, 2.0]
  gamma_min:  [0.5, 0.7, 0.8, 0.9, 0.95]

Fixed Top-B grid:
  [1, 2, 4, 8, 16, 32]

Fixed Threshold grid:
  [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

Random-B: inherits selected Fixed Top-B B; NOT independently tuned.
Adaptive-Budget Random: inherits selected AFP adaptive budgets; NOT independently tuned.

Total validation configurations: 30
Tuning specification SHA256: d2d990a09c9d9494e8941ad3f9fb561768a8a891467d23a2e55ed993e35f4510


webqsp RoG fidelity:   0%|          | 0/246 [00:00<?, ?it/s]


WEBQSP RoG FIDELITY
questions                 246
total_predicted_plans     721
nonempty_plans            721
empty_plans               0
active_hop_rows           971
active_prefixes           2440
edges_examined            341526
candidate_branches        7983
reachable_plans           345
reachable_questions       205
WEBQSP RoG fidelity: PASSED


cwq RoG fidelity:   0%|          | 0/3519 [00:00<?, ?it/s]


CWQ RoG FIDELITY
questions                 3519
total_predicted_plans     10536
nonempty_plans            10529
empty_plans               7
active_hop_rows           16564
active_prefixes           57841
edges_examined            5257272
candidate_branches        247161
reachable_plans           3971
reachable_questions       2425
CWQ RoG fidelity: PASSED

Exact RoG validation fidelity gate: PASSED



...


Validation online semantic inventory
  Questions:          3765
  Readable entities:  25248
  Relations:          762
  Plans:              2720
  Suffixes:           724

Batch-prefilling semantic runtime...
  question      3765
  entity       25248
  relation       762
  plan          2720
  suffix         724
Validation semantic prefill: READY

STARTING VALIDATION-ONLY CONFIGURATION SWEEP


webqsp tuning:   0%|          | 0/246 [00:00<?, ?it/s]


WEBQSP tuning sweep completed.
Elapsed this run: 0.04 min


cwq tuning:   0%|          | 0/3519 [00:00<?, ?it/s]


CWQ tuning sweep completed.
Elapsed this run: 2.23 min

WEBQSP VALIDATION TUNING RESULTS

--- afp ---
     config_id family        T  gamma_min   B  tau      ssr  answer_retention  reachable_questions  edges_examined  active_hop_rows  active_prefixes  decision_hops  avg_requested_budget  avg_uncertainty  fully_tied_decisions  ar_floor_feasible
 afp_T0.5_g0.5    afp 0.500000   0.500000 NaN  NaN 0.001403          1.000000                  205          341047              971             2425            147             10.789116         0.988268                    97               True
 afp_T0.5_g0.7    afp 0.500000   0.700000 NaN  NaN 0.000820          1.000000                  205          341246              971             2428            147             10.809524         0.988268                    97               True
 afp_T0.5_g0.8    afp 0.500000   0.800000 NaN  NaN 0.000217          1.000000                  205          341452              971             2432            147  

## RQ2 validation comparison, SSR(Search Space Reduction) and Answer Retentation 

In [28]:
# ======================================================================
# RQ2 CELL 14
# VALIDATION CONTROLLED COMPARISON
# ======================================================================
#
# METHODS
# -------
# 1. RoG
# 2. Fixed Top-B
# 3. Fixed Threshold
# 4. Random-B                     seeds 42,43,44
# 5. Adaptive-Budget Random       seeds 42,43,44
# 6. AFP
#
# CONTROL
# -------
# Same:
#   questions
#   topic entities
#   per-question graph
#   relation plans
#   exact RoG graph semantics
#   Feature-v2
#   selected scorer
#   final-hop protection
#
# Random-B:
#   inherits selected Fixed Top-B B.
#
# Adaptive-Budget Random:
#   uses scorer ONLY to determine AFP's adaptive retained COUNT;
#   branch identity is then selected uniformly at random.
#
# This directly tests whether AFP's learned branch ranking adds value
# beyond its adaptive budget.
#
# VALIDATION ONLY.
# NO TEST examples accessed.
# ======================================================================

import hashlib
import json
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


# ======================================================================
# 1. HARD PREREQUISITES
# ======================================================================

required = [
    "CELL13_TUNING_COMPLETE",

    "AFP_VALIDATION_SELECTED",
    "FIXED_TOP_B_VALIDATION_SELECTED",
    "FIXED_THRESHOLD_VALIDATION_SELECTED",

    "AFP_RUNTIME_SCORE_GROUP",

    "webqsp_val_plan_rows",
    "cwq_val_plan_rows",

    "webqsp_val_runtime",
    "cwq_val_runtime",

    "webqsp_rog_reference",
    "cwq_rog_reference",

    "build_exact_rog_adjacency",
    "as_entity_list",
    "normalize_relation_plans",

    "select_policy_indices",
    "get_cached_relation_expansion",
    "get_group_logits",
    "final_prefixes_reach_answer",

    "webqsp_ckpt_sha",
    "cwq_ckpt_sha",
]

missing = [
    name
    for name in required
    if name not in globals()
]

assert not missing, (
    "Missing Cell-13/B4 objects:\n  "
    + "\n  ".join(missing)
)

assert CELL13_TUNING_COMPLETE is True

print("Cell 14 prerequisites: PASSED")


# ======================================================================
# 2. OUTPUT DIRECTORY
# ======================================================================

ROOT = Path(
    "/kaggle/working/step3_rq2_dev_v1"
)

COMPARE_DIR = (
    ROOT
    / "12_validation_comparison"
)

COMPARE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ======================================================================
# 3. FIXED RANDOM SEEDS
# ======================================================================

RANDOM_SEEDS = [
    42,
    43,
    44,
]

print(
    "Random baseline seeds:",
    RANDOM_SEEDS
)


# ======================================================================
# 4. DISPLAY FROZEN DEVELOPMENT CONFIGURATIONS
# ======================================================================

print(
    "\n"
    + "=" * 108
)

print(
    "FROZEN VALIDATION-SELECTED CONFIGURATIONS"
)

print(
    "=" * 108
)


for dataset in [
    "webqsp",
    "cwq",
]:

    print(
        f"\n{dataset.upper()}"
    )

    print(
        "  AFP:",
        AFP_VALIDATION_SELECTED[
            dataset
        ]
    )

    print(
        "  Fixed Top-B:",
        FIXED_TOP_B_VALIDATION_SELECTED[
            dataset
        ]
    )

    print(
        "  Fixed Threshold:",
        FIXED_THRESHOLD_VALIDATION_SELECTED[
            dataset
        ]
    )


# ======================================================================
# 5. METHOD SPECIFICATION
# ======================================================================

def build_method_specs(
    dataset_name
):

    afp_params = (
        AFP_VALIDATION_SELECTED[
            dataset_name
        ]
    )

    selected_B = int(
        FIXED_TOP_B_VALIDATION_SELECTED[
            dataset_name
        ]
    )

    selected_tau = float(
        FIXED_THRESHOLD_VALIDATION_SELECTED[
            dataset_name
        ]
    )


    methods = [
        {
            "method":
                "RoG",

            "family":
                "rog",

            "seed":
                None,
        },

        {
            "method":
                "Fixed-Top-B",

            "family":
                "fixed_top_b",

            "B":
                selected_B,

            "seed":
                None,
        },

        {
            "method":
                "Fixed-Threshold",

            "family":
                "fixed_threshold",

            "tau":
                selected_tau,

            "seed":
                None,
        },

        {
            "method":
                "AFP",

            "family":
                "afp",

            "T":
                float(
                    afp_params[
                        "T"
                    ]
                ),

            "gamma_min":
                float(
                    afp_params[
                        "gamma_min"
                    ]
                ),

            "seed":
                None,
        },
    ]


    for seed in RANDOM_SEEDS:

        methods.append(
            {
                "method":
                    "Random-B",

                "family":
                    "random_b",

                "B":
                    selected_B,

                "seed":
                    int(
                        seed
                    ),
            }
        )


    for seed in RANDOM_SEEDS:

        methods.append(
            {
                "method":
                    "Adaptive-Budget-Random",

                "family":
                    "adaptive_budget_random",

                "T":
                    float(
                        afp_params[
                            "T"
                        ]
                    ),

                "gamma_min":
                    float(
                        afp_params[
                            "gamma_min"
                        ]
                    ),

                "seed":
                    int(
                        seed
                    ),
            }
        )


    return methods


WEBQSP_METHODS = (
    build_method_specs(
        "webqsp"
    )
)

CWQ_METHODS = (
    build_method_specs(
        "cwq"
    )
)


# ======================================================================
# 6. DETERMINISTIC RANDOM GENERATOR
# ======================================================================
#
# Do NOT use Python hash().
#
# RNG seed derives from:
#
#   fixed seed
#   dataset
#   question ID
#   plan index
#   hop
#   active frontier
#
# Therefore runs are reproducible independent of process/hash state.
# ======================================================================

def deterministic_group_rng(
    seed,
    dataset_name,
    question_id,
    plan_index,
    hop,
    active_prefixes
):

    payload = {
        "seed":
            int(
                seed
            ),

        "dataset":
            str(
                dataset_name
            ),

        "question_id":
            str(
                question_id
            ),

        "plan_index":
            int(
                plan_index
            ),

        "hop":
            int(
                hop
            ),

        "active_prefixes":
            [
                list(
                    prefix
                )
                for prefix in active_prefixes
            ],
    }


    canonical = json.dumps(
        payload,
        sort_keys=True,
        separators=(
            ",",
            ":"
        ),
        ensure_ascii=False
    )


    digest = hashlib.sha256(
        canonical.encode(
            "utf-8"
        )
    ).digest()


    rng_seed = int.from_bytes(
        digest[
            :8
        ],
        byteorder="big",
        signed=False
    )


    return np.random.default_rng(
        rng_seed
    )


# ======================================================================
# 7. RANDOM SELECTION
# ======================================================================

def uniform_random_indices(
    n,
    k,
    rng
):

    n = int(
        n
    )

    k = int(
        k
    )


    assert n >= 1
    assert 0 <= k <= n


    if k == 0:

        return []


    if k >= n:

        return list(
            range(
                n
            )
        )


    selected = rng.choice(
        n,
        size=k,
        replace=False
    )


    # Preserve original candidate order after sampling.
    return sorted(
        int(
            x
        )
        for x in selected
    )


# ======================================================================
# 8. METHOD-SPECIFIC SELECTION
# ======================================================================

def controlled_selection(
    dataset_name,
    method_spec,
    question_id,
    plan_index,
    hop,
    active_prefixes,
    logits
):

    family = method_spec[
        "family"
    ]


    n = len(
        logits
    )


    assert n > 1


    # ------------------------------------------------------------------
    # Fixed Top-B
    # ------------------------------------------------------------------

    if family == "fixed_top_b":

        config = {
            "family":
                "fixed_top_b",

            "B":
                int(
                    method_spec[
                        "B"
                    ]
                ),
        }


        output = (
            select_policy_indices(
                config,
                logits
            )
        )


        return {
            **output,

            "selection_type":
                "scorer_ranked_fixed_budget",
        }


    # ------------------------------------------------------------------
    # Fixed Threshold
    # ------------------------------------------------------------------

    if family == "fixed_threshold":

        config = {
            "family":
                "fixed_threshold",

            "tau":
                float(
                    method_spec[
                        "tau"
                    ]
                ),
        }


        output = (
            select_policy_indices(
                config,
                logits
            )
        )


        return {
            **output,

            "selection_type":
                "scorer_threshold",
        }


    # ------------------------------------------------------------------
    # AFP
    # ------------------------------------------------------------------

    if family == "afp":

        config = {
            "family":
                "afp",

            "T":
                float(
                    method_spec[
                        "T"
                    ]
                ),

            "gamma_min":
                float(
                    method_spec[
                        "gamma_min"
                    ]
                ),
        }


        output = (
            select_policy_indices(
                config,
                logits
            )
        )


        return {
            **output,

            "selection_type":
                "adaptive_scorer_ranked",
        }


    # ------------------------------------------------------------------
    # Random-B
    # ------------------------------------------------------------------

    if family == "random_b":

        B = int(
            method_spec[
                "B"
            ]
        )


        retained_count = min(
            B,
            n
        )


        rng = (
            deterministic_group_rng(
                seed=
                    method_spec[
                        "seed"
                    ],

                dataset_name=
                    dataset_name,

                question_id=
                    question_id,

                plan_index=
                    plan_index,

                hop=
                    hop,

                active_prefixes=
                    active_prefixes
            )
        )


        selected = (
            uniform_random_indices(
                n=
                    n,

                k=
                    retained_count,

                rng=
                    rng
            )
        )


        return {
            "selected_indices":
                selected,

            "requested_budget":
                retained_count,

            "retained_count":
                len(
                    selected
                ),

            "uncertainty":
                None,

            "gamma":
                None,

            "fully_tied":
                False,

            "selection_type":
                "uniform_random_fixed_budget",
        }


    # ------------------------------------------------------------------
    # Adaptive-Budget Random
    # ------------------------------------------------------------------
    #
    # IMPORTANT:
    #
    # First compute what AFP would do on THIS encountered group.
    #
    # We use AFP's ACTUAL retained count after tie expansion as the
    # matched local budget.
    #
    # Then branch identities are selected uniformly at random.
    # ------------------------------------------------------------------

    assert family == "adaptive_budget_random"


    afp_config = {
        "family":
            "afp",

        "T":
            float(
                method_spec[
                    "T"
                ]
            ),

        "gamma_min":
            float(
                method_spec[
                    "gamma_min"
                ]
            ),
    }


    afp_budget_output = (
        select_policy_indices(
            afp_config,
            logits
        )
    )


    matched_count = int(
        afp_budget_output[
            "retained_count"
        ]
    )


    assert (
        1
        <= matched_count
        <= n
    )


    rng = (
        deterministic_group_rng(
            seed=
                method_spec[
                    "seed"
                ],

            dataset_name=
                dataset_name,

            question_id=
                question_id,

            plan_index=
                plan_index,

            hop=
                hop,

            active_prefixes=
                active_prefixes
        )
    )


    selected = (
        uniform_random_indices(
            n=
                n,

            k=
                matched_count,

            rng=
                rng
        )
    )


    return {
        "selected_indices":
            selected,

        "requested_budget":
            int(
                afp_budget_output[
                    "requested_budget"
                ]
            ),

        "retained_count":
            len(
                selected
            ),

        "uncertainty":
            afp_budget_output[
                "uncertainty"
            ],

        "gamma":
            afp_budget_output[
                "gamma"
            ],

        "fully_tied":
            afp_budget_output[
                "fully_tied"
            ],

        "selection_type":
            "uniform_random_afp_matched_retained_count",
    }


# ======================================================================
# 9. CONTROLLED METHOD TRAVERSAL
# ======================================================================

def traverse_controlled_method(
    dataset_name,
    method_spec,
    question_id,
    question,
    adjacency,
    topic_entities,
    plan_index,
    plan,
    expansion_cache,
    score_cache
):

    family = method_spec[
        "family"
    ]


    active = [
        (
            str(
                entity
            ),
        )
        for entity in topic_entities
    ]


    L = len(
        plan
    )


    active_hop_rows = 0
    active_prefixes_total = 0

    edges_examined = 0
    candidate_branches = 0

    decision_hops = 0
    retained_after_decision = 0

    requested_budget_total = 0
    requested_budget_observations = 0

    uncertainty_total = 0.0
    uncertainty_observations = 0

    fully_tied_decisions = 0

    peak_frontier = len(
        active
    )


    for hop, target_relation in enumerate(
        plan
    ):

        if not active:

            break


        active_hop_rows += 1

        active_prefixes_total += len(
            active
        )


        expansion = (
            get_cached_relation_expansion(
                adjacency=
                    adjacency,

                plan_index=
                    plan_index,

                hop=
                    hop,

                target_relation=
                    target_relation,

                active_prefixes=
                    active,

                expansion_cache=
                    expansion_cache
            )
        )


        candidates = expansion[
            "candidates"
        ]


        candidate_rows = expansion[
            "candidate_rows"
        ]


        # Search cost THIS method would pay.
        edges_examined += int(
            expansion[
                "edges_cost"
            ]
        )


        candidate_branches += len(
            candidates
        )


        if not candidates:

            active = []

            break


        # --------------------------------------------------------------
        # RoG never prunes.
        # --------------------------------------------------------------

        if family == "rog":

            active = candidates


            peak_frontier = max(
                peak_frontier,
                len(
                    active
                )
            )


            continue


        # --------------------------------------------------------------
        # Final-hop protection for ALL pruning methods.
        # --------------------------------------------------------------

        is_final_hop = (
            hop
            ==
            L - 1
        )


        if is_final_hop:

            active = candidates


            peak_frontier = max(
                peak_frontier,
                len(
                    active
                )
            )


            continue


        # --------------------------------------------------------------
        # Singleton bypass for ALL pruning methods.
        # --------------------------------------------------------------

        if len(
            candidates
        ) <= 1:

            active = candidates


            peak_frontier = max(
                peak_frontier,
                len(
                    active
                )
            )


            continue


        decision_hops += 1


        logits = (
            get_group_logits(
                dataset_name=
                    dataset_name,

                question_id=
                    question_id,

                question=
                    question,

                plan_index=
                    plan_index,

                plan=
                    plan,

                hop=
                    hop,

                active_prefixes=
                    active,

                candidate_rows=
                    candidate_rows,

                score_cache=
                    score_cache
            )
        )


        selection = (
            controlled_selection(
                dataset_name=
                    dataset_name,

                method_spec=
                    method_spec,

                question_id=
                    question_id,

                plan_index=
                    plan_index,

                hop=
                    hop,

                active_prefixes=
                    active,

                logits=
                    logits
            )
        )


        selected_indices = (
            selection[
                "selected_indices"
            ]
        )


        active = [
            candidates[
                index
            ]
            for index in selected_indices
        ]


        retained_after_decision += len(
            active
        )


        if (
            selection[
                "requested_budget"
            ]
            is not None
        ):

            requested_budget_total += int(
                selection[
                    "requested_budget"
                ]
            )

            requested_budget_observations += 1


        if (
            selection[
                "uncertainty"
            ]
            is not None
        ):

            uncertainty_total += float(
                selection[
                    "uncertainty"
                ]
            )

            uncertainty_observations += 1


        if selection[
            "fully_tied"
        ]:

            fully_tied_decisions += 1


        peak_frontier = max(
            peak_frontier,
            len(
                active
            )
        )


    return {
        "final_prefixes":
            active,

        "active_hop_rows":
            int(
                active_hop_rows
            ),

        "active_prefixes":
            int(
                active_prefixes_total
            ),

        "edges_examined":
            int(
                edges_examined
            ),

        "candidate_branches":
            int(
                candidate_branches
            ),

        "decision_hops":
            int(
                decision_hops
            ),

        "retained_after_decision":
            int(
                retained_after_decision
            ),

        "requested_budget_total":
            int(
                requested_budget_total
            ),

        "requested_budget_observations":
            int(
                requested_budget_observations
            ),

        "uncertainty_total":
            float(
                uncertainty_total
            ),

        "uncertainty_observations":
            int(
                uncertainty_observations
            ),

        "fully_tied_decisions":
            int(
                fully_tied_decisions
            ),

        "peak_frontier":
            int(
                peak_frontier
            ),
    }


# ======================================================================
# 10. RUN CONTROLLED COMPARISON
# ======================================================================

def run_controlled_dataset(
    dataset_name,
    planning_rows,
    question_rows,
    rog_reference,
    method_specs
):

    assert len(
        planning_rows
    ) == len(
        question_rows
    )


    aggregate = {}


    per_question_rows = []


    for spec_index, spec in enumerate(
        method_specs
    ):

        run_id = (
            f"{spec['method']}"
            if spec[
                "seed"
            ]
            is None
            else
            f"{spec['method']}_seed{spec['seed']}"
        )


        aggregate[
            run_id
        ] = {
            "method":
                spec[
                    "method"
                ],

            "family":
                spec[
                    "family"
                ],

            "seed":
                spec[
                    "seed"
                ],

            "active_hop_rows":
                0,

            "active_prefixes":
                0,

            "edges_examined":
                0,

            "candidate_branches":
                0,

            "reachable_plans":
                0,

            "reachable_questions":
                0,

            "decision_hops":
                0,

            "retained_after_decision":
                0,

            "requested_budget_total":
                0,

            "requested_budget_observations":
                0,

            "uncertainty_total":
                0.0,

            "uncertainty_observations":
                0,

            "fully_tied_decisions":
                0,

            "peak_frontier":
                0,
        }


    start_time = time.time()


    for source_index in tqdm(
        range(
            len(
                planning_rows
            )
        ),
        desc=(
            f"{dataset_name} controlled comparison"
        )
    ):

        plan_rec = planning_rows[
            source_index
        ]


        question_rec = question_rows[
            source_index
        ]


        assert str(
            plan_rec[
                "id"
            ]
        ) == str(
            question_rec[
                "id"
            ]
        )


        question_id = str(
            plan_rec[
                "id"
            ]
        )


        question = str(
            question_rec[
                "question"
            ]
        )


        topic_entities = (
            as_entity_list(
                plan_rec[
                    "q_entity"
                ]
            )
        )


        gold_answers = (
            as_entity_list(
                plan_rec[
                    "a_entity"
                ]
            )
        )


        plans = (
            normalize_relation_plans(
                plan_rec[
                    "predicted_paths"
                ]
            )
        )


        adjacency = (
            build_exact_rog_adjacency(
                plan_rec[
                    "graph"
                ]
            )
        )


        # Shared computational caches only.
        expansion_cache = {}
        score_cache = {}


        question_metrics = {}


        for spec in method_specs:

            run_id = (
                f"{spec['method']}"
                if spec[
                    "seed"
                ]
                is None
                else
                f"{spec['method']}_seed{spec['seed']}"
            )


            question_metrics[
                run_id
            ] = {
                "edges_examined":
                    0,

                "active_prefixes":
                    0,

                "candidate_branches":
                    0,

                "reachable":
                    False,
            }


        # ==============================================================
        # PLAN LOOP
        # ==============================================================

        for plan_index, plan in enumerate(
            plans
        ):

            if len(
                plan
            ) == 0:

                continue


            for spec in method_specs:

                run_id = (
                    f"{spec['method']}"
                    if spec[
                        "seed"
                    ]
                    is None
                    else
                    f"{spec['method']}_seed{spec['seed']}"
                )


                result = (
                    traverse_controlled_method(
                        dataset_name=
                            dataset_name,

                        method_spec=
                            spec,

                        question_id=
                            question_id,

                        question=
                            question,

                        adjacency=
                            adjacency,

                        topic_entities=
                            topic_entities,

                        plan_index=
                            plan_index,

                        plan=
                            plan,

                        expansion_cache=
                            expansion_cache,

                        score_cache=
                            score_cache
                    )
                )


                stats = aggregate[
                    run_id
                ]


                stats[
                    "active_hop_rows"
                ] += result[
                    "active_hop_rows"
                ]


                stats[
                    "active_prefixes"
                ] += result[
                    "active_prefixes"
                ]


                stats[
                    "edges_examined"
                ] += result[
                    "edges_examined"
                ]


                stats[
                    "candidate_branches"
                ] += result[
                    "candidate_branches"
                ]


                stats[
                    "decision_hops"
                ] += result[
                    "decision_hops"
                ]


                stats[
                    "retained_after_decision"
                ] += result[
                    "retained_after_decision"
                ]


                stats[
                    "requested_budget_total"
                ] += result[
                    "requested_budget_total"
                ]


                stats[
                    "requested_budget_observations"
                ] += result[
                    "requested_budget_observations"
                ]


                stats[
                    "uncertainty_total"
                ] += result[
                    "uncertainty_total"
                ]


                stats[
                    "uncertainty_observations"
                ] += result[
                    "uncertainty_observations"
                ]


                stats[
                    "fully_tied_decisions"
                ] += result[
                    "fully_tied_decisions"
                ]


                stats[
                    "peak_frontier"
                ] = max(
                    stats[
                        "peak_frontier"
                    ],
                    result[
                        "peak_frontier"
                    ]
                )


                reachable = (
                    final_prefixes_reach_answer(
                        result[
                            "final_prefixes"
                        ],
                        gold_answers
                    )
                )


                if reachable:

                    stats[
                        "reachable_plans"
                    ] += 1


                    question_metrics[
                        run_id
                    ][
                        "reachable"
                    ] = True


                question_metrics[
                    run_id
                ][
                    "edges_examined"
                ] += result[
                    "edges_examined"
                ]


                question_metrics[
                    run_id
                ][
                    "active_prefixes"
                ] += result[
                    "active_prefixes"
                ]


                question_metrics[
                    run_id
                ][
                    "candidate_branches"
                ] += result[
                    "candidate_branches"
                ]


        # ==============================================================
        # QUESTION-LEVEL AGGREGATION
        # ==============================================================

        for run_id, qstats in (
            question_metrics.items()
        ):

            if qstats[
                "reachable"
            ]:

                aggregate[
                    run_id
                ][
                    "reachable_questions"
                ] += 1


            per_question_rows.append(
                {
                    "dataset":
                        dataset_name,

                    "question_index":
                        int(
                            source_index
                        ),

                    "question_id":
                        question_id,

                    "run_id":
                        run_id,

                    "method":
                        aggregate[
                            run_id
                        ][
                            "method"
                        ],

                    "seed":
                        aggregate[
                            run_id
                        ][
                            "seed"
                        ],

                    "edges_examined":
                        int(
                            qstats[
                                "edges_examined"
                            ]
                        ),

                    "active_prefixes":
                        int(
                            qstats[
                                "active_prefixes"
                            ]
                        ),

                    "candidate_branches":
                        int(
                            qstats[
                                "candidate_branches"
                            ]
                        ),

                    "reachable":
                        bool(
                            qstats[
                                "reachable"
                            ]
                        ),
                }
            )


    elapsed = (
        time.time()
        -
        start_time
    )


    print(
        f"\n{dataset_name.upper()} controlled comparison completed."
    )

    print(
        f"Elapsed: {elapsed/60:.2f} min"
    )


    # ==================================================================
    # AGGREGATE TABLE
    # ==================================================================

    rows = []


    rog_edges = float(
        rog_reference[
            "edges_examined"
        ]
    )


    rog_reachable = int(
        rog_reference[
            "reachable_questions"
        ]
    )


    for run_id, stats in (
        aggregate.items()
    ):

        edges = int(
            stats[
                "edges_examined"
            ]
        )


        reachable_q = int(
            stats[
                "reachable_questions"
            ]
        )


        avg_budget = (
            stats[
                "requested_budget_total"
            ]
            /
            stats[
                "requested_budget_observations"
            ]

            if
            stats[
                "requested_budget_observations"
            ]
            > 0

            else
            np.nan
        )


        avg_uncertainty = (
            stats[
                "uncertainty_total"
            ]
            /
            stats[
                "uncertainty_observations"
            ]

            if
            stats[
                "uncertainty_observations"
            ]
            > 0

            else
            np.nan
        )


        rows.append(
            {
                "dataset":
                    dataset_name,

                "run_id":
                    run_id,

                "method":
                    stats[
                        "method"
                    ],

                "family":
                    stats[
                        "family"
                    ],

                "seed":
                    stats[
                        "seed"
                    ],

                "edges_examined":
                    edges,

                "ssr":
                    float(
                        1.0
                        -
                        edges
                        /
                        rog_edges
                    ),

                "reachable_questions":
                    reachable_q,

                "rog_reachable_questions":
                    rog_reachable,

                "answer_retention":
                    float(
                        reachable_q
                        /
                        rog_reachable
                    ),

                "coverage_all_questions":
                    float(
                        reachable_q
                        /
                        len(
                            planning_rows
                        )
                    ),

                "reachable_plans":
                    int(
                        stats[
                            "reachable_plans"
                        ]
                    ),

                "active_hop_rows":
                    int(
                        stats[
                            "active_hop_rows"
                        ]
                    ),

                "active_prefixes":
                    int(
                        stats[
                            "active_prefixes"
                        ]
                    ),

                "candidate_branches":
                    int(
                        stats[
                            "candidate_branches"
                        ]
                    ),

                "decision_hops":
                    int(
                        stats[
                            "decision_hops"
                        ]
                    ),

                "retained_after_decision":
                    int(
                        stats[
                            "retained_after_decision"
                        ]
                    ),

                "avg_requested_budget":
                    float(
                        avg_budget
                    ),

                "avg_uncertainty":
                    float(
                        avg_uncertainty
                    ),

                "fully_tied_decisions":
                    int(
                        stats[
                            "fully_tied_decisions"
                        ]
                    ),

                "peak_frontier":
                    int(
                        stats[
                            "peak_frontier"
                        ]
                    ),
            }
        )


    aggregate_df = pd.DataFrame(
        rows
    )


    per_question_df = pd.DataFrame(
        per_question_rows
    )


    return (
        aggregate_df,
        per_question_df
    )


# ======================================================================
# 11. RUN BOTH DATASETS
# ======================================================================

webqsp_comparison_df, webqsp_question_df = (
    run_controlled_dataset(
        dataset_name=
            "webqsp",

        planning_rows=
            webqsp_val_plan_rows,

        question_rows=
            webqsp_val_runtime,

        rog_reference=
            webqsp_rog_reference,

        method_specs=
            WEBQSP_METHODS
    )
)


cwq_comparison_df, cwq_question_df = (
    run_controlled_dataset(
        dataset_name=
            "cwq",

        planning_rows=
            cwq_val_plan_rows,

        question_rows=
            cwq_val_runtime,

        rog_reference=
            cwq_rog_reference,

        method_specs=
            CWQ_METHODS
    )
)


# ======================================================================
# 12. RoG CONTROL FIDELITY
# ======================================================================

def assert_rog_control(
    dataset_name,
    df,
    reference
):

    rog_row = (
        df[
            df[
                "method"
            ]
            ==
            "RoG"
        ]
        .iloc[
            0
        ]
    )


    assert int(
        rog_row[
            "edges_examined"
        ]
    ) == int(
        reference[
            "edges_examined"
        ]
    )


    assert int(
        rog_row[
            "reachable_questions"
        ]
    ) == int(
        reference[
            "reachable_questions"
        ]
    )


    assert int(
        rog_row[
            "active_prefixes"
        ]
    ) == int(
        reference[
            "active_prefixes"
        ]
    )


    assert abs(
        float(
            rog_row[
                "ssr"
            ]
        )
    ) <= 1e-12


    assert abs(
        float(
            rog_row[
                "answer_retention"
            ]
        )
        -
        1.0
    ) <= 1e-12


    print(
        f"{dataset_name} RoG control fidelity: PASSED"
    )


assert_rog_control(
    "WebQSP",
    webqsp_comparison_df,
    webqsp_rog_reference
)


assert_rog_control(
    "CWQ",
    cwq_comparison_df,
    cwq_rog_reference
)


# ======================================================================
# 13. VERIFY DETERMINISTIC METHODS MATCH CELL 13
# ======================================================================

def get_cell13_selected_row(
    tuning_df,
    family
):

    if family == "afp":

        selected_config = None

        # reconstruct from exposed selected params
        dataset = str(
            tuning_df[
                "dataset"
            ].iloc[
                0
            ]
        )


        params = (
            AFP_VALIDATION_SELECTED[
                dataset
            ]
        )


        mask = (
            (
                tuning_df[
                    "family"
                ]
                ==
                "afp"
            )
            &
            np.isclose(
                tuning_df[
                    "T"
                ],
                params[
                    "T"
                ]
            )
            &
            np.isclose(
                tuning_df[
                    "gamma_min"
                ],
                params[
                    "gamma_min"
                ]
            )
        )


    elif family == "fixed_top_b":

        dataset = str(
            tuning_df[
                "dataset"
            ].iloc[
                0
            ]
        )


        selected_B = (
            FIXED_TOP_B_VALIDATION_SELECTED[
                dataset
            ]
        )


        mask = (
            (
                tuning_df[
                    "family"
                ]
                ==
                family
            )
            &
            (
                tuning_df[
                    "B"
                ]
                ==
                selected_B
            )
        )


    else:

        dataset = str(
            tuning_df[
                "dataset"
            ].iloc[
                0
            ]
        )


        selected_tau = (
            FIXED_THRESHOLD_VALIDATION_SELECTED[
                dataset
            ]
        )


        mask = (
            (
                tuning_df[
                    "family"
                ]
                ==
                "fixed_threshold"
            )
            &
            np.isclose(
                tuning_df[
                    "tau"
                ],
                selected_tau
            )
        )


    rows = tuning_df[
        mask
    ]


    assert len(
        rows
    ) == 1


    return rows.iloc[
        0
    ]


def deterministic_method_fidelity(
    dataset_name,
    comparison_df,
    tuning_df
):

    mapping = {
        "AFP":
            "afp",

        "Fixed-Top-B":
            "fixed_top_b",

        "Fixed-Threshold":
            "fixed_threshold",
    }


    for method, family in (
        mapping.items()
    ):

        comparison = (
            comparison_df[
                comparison_df[
                    "method"
                ]
                ==
                method
            ]
            .iloc[
                0
            ]
        )


        tuning = (
            get_cell13_selected_row(
                tuning_df,
                family
            )
        )


        assert int(
            comparison[
                "edges_examined"
            ]
        ) == int(
            tuning[
                "edges_examined"
            ]
        ), (
            f"{dataset_name} {method} edge mismatch "
            "vs Cell 13."
        )


        assert int(
            comparison[
                "reachable_questions"
            ]
        ) == int(
            tuning[
                "reachable_questions"
            ]
        ), (
            f"{dataset_name} {method} AR mismatch "
            "vs Cell 13."
        )


    print(
        f"{dataset_name} deterministic-method "
        "Cell-13 fidelity: PASSED"
    )


deterministic_method_fidelity(
    "WebQSP",
    webqsp_comparison_df,
    webqsp_tuning_df
)


deterministic_method_fidelity(
    "CWQ",
    cwq_comparison_df,
    cwq_tuning_df
)


# ======================================================================
# 14. RANDOM BASELINE SUMMARY
# ======================================================================

def summarize_method_runs(
    comparison_df
):

    summary_rows = []


    method_order = [
        "RoG",
        "Fixed-Top-B",
        "Fixed-Threshold",
        "Random-B",
        "Adaptive-Budget-Random",
        "AFP",
    ]


    for method in method_order:

        rows = comparison_df[
            comparison_df[
                "method"
            ]
            ==
            method
        ]


        assert len(
            rows
        ) > 0


        summary_rows.append(
            {
                "method":
                    method,

                "n_runs":
                    int(
                        len(
                            rows
                        )
                    ),

                "edges_mean":
                    float(
                        rows[
                            "edges_examined"
                        ].mean()
                    ),

                "edges_sd":
                    float(
                        rows[
                            "edges_examined"
                        ].std(
                            ddof=1
                        )
                    )
                    if len(
                        rows
                    ) > 1
                    else
                    0.0,

                "ssr_mean":
                    float(
                        rows[
                            "ssr"
                        ].mean()
                    ),

                "ssr_sd":
                    float(
                        rows[
                            "ssr"
                        ].std(
                            ddof=1
                        )
                    )
                    if len(
                        rows
                    ) > 1
                    else
                    0.0,

                "ar_mean":
                    float(
                        rows[
                            "answer_retention"
                        ].mean()
                    ),

                "ar_sd":
                    float(
                        rows[
                            "answer_retention"
                        ].std(
                            ddof=1
                        )
                    )
                    if len(
                        rows
                    ) > 1
                    else
                    0.0,

                "reachable_q_mean":
                    float(
                        rows[
                            "reachable_questions"
                        ].mean()
                    ),

                "active_prefixes_mean":
                    float(
                        rows[
                            "active_prefixes"
                        ].mean()
                    ),

                "candidate_branches_mean":
                    float(
                        rows[
                            "candidate_branches"
                        ].mean()
                    ),

                "decision_hops_mean":
                    float(
                        rows[
                            "decision_hops"
                        ].mean()
                    ),

                "avg_requested_budget_mean":
                    float(
                        rows[
                            "avg_requested_budget"
                        ].mean()
                    )
                    if rows[
                        "avg_requested_budget"
                    ].notna().any()
                    else
                    np.nan,
            }
        )


    return pd.DataFrame(
        summary_rows
    )


webqsp_summary_df = (
    summarize_method_runs(
        webqsp_comparison_df
    )
)


cwq_summary_df = (
    summarize_method_runs(
        cwq_comparison_df
    )
)


# ======================================================================
# 15. DISPLAY RUN-LEVEL RESULTS
# ======================================================================

RUN_COLUMNS = [
    "method",
    "seed",
    "edges_examined",
    "ssr",
    "reachable_questions",
    "answer_retention",
    "active_prefixes",
    "candidate_branches",
    "decision_hops",
    "avg_requested_budget",
    "fully_tied_decisions",
    "peak_frontier",
]


def display_results(
    dataset_name,
    comparison_df,
    summary_df
):

    print(
        "\n"
        + "=" * 118
    )

    print(
        f"{dataset_name.upper()} "
        "CONTROLLED VALIDATION COMPARISON — RUN LEVEL"
    )

    print(
        "=" * 118
    )


    print(
        comparison_df[
            RUN_COLUMNS
        ].to_string(
            index=False,
            float_format=lambda x:
                f"{x:.6f}"
        )
    )


    print(
        "\n"
        + "=" * 118
    )

    print(
        f"{dataset_name.upper()} "
        "CONTROLLED VALIDATION COMPARISON — SUMMARY"
    )

    print(
        "=" * 118
    )


    print(
        summary_df.to_string(
            index=False,
            float_format=lambda x:
                f"{x:.6f}"
        )
    )


display_results(
    "webqsp",
    webqsp_comparison_df,
    webqsp_summary_df
)


display_results(
    "cwq",
    cwq_comparison_df,
    cwq_summary_df
)


# ======================================================================
# 16. KEY CAUSAL COMPARISONS
# ======================================================================

def method_summary_row(
    summary_df,
    method
):

    rows = summary_df[
        summary_df[
            "method"
        ]
        ==
        method
    ]


    assert len(
        rows
    ) == 1


    return rows.iloc[
        0
    ]


def print_causal_diagnostics(
    dataset_name,
    summary_df
):

    afp = method_summary_row(
        summary_df,
        "AFP"
    )


    fixed = method_summary_row(
        summary_df,
        "Fixed-Top-B"
    )


    random_b = method_summary_row(
        summary_df,
        "Random-B"
    )


    adaptive_random = (
        method_summary_row(
            summary_df,
            "Adaptive-Budget-Random"
        )
    )


    print(
        "\n"
        + "=" * 112
    )

    print(
        f"{dataset_name.upper()} CAUSAL DIAGNOSTICS"
    )

    print(
        "=" * 112
    )


    print(
        "\nAFP:"
    )

    print(
        f"  AR  = {afp['ar_mean']:.6f}"
    )

    print(
        f"  SSR = {afp['ssr_mean']:.6f}"
    )


    print(
        "\nFixed Top-B:"
    )

    print(
        f"  AR  = {fixed['ar_mean']:.6f}"
    )

    print(
        f"  SSR = {fixed['ssr_mean']:.6f}"
    )


    print(
        "\nRandom-B:"
    )

    print(
        f"  AR  = {random_b['ar_mean']:.6f} "
        f"± {random_b['ar_sd']:.6f}"
    )

    print(
        f"  SSR = {random_b['ssr_mean']:.6f} "
        f"± {random_b['ssr_sd']:.6f}"
    )


    print(
        "\nAdaptive-Budget Random:"
    )

    print(
        f"  AR  = {adaptive_random['ar_mean']:.6f} "
        f"± {adaptive_random['ar_sd']:.6f}"
    )

    print(
        f"  SSR = {adaptive_random['ssr_mean']:.6f} "
        f"± {adaptive_random['ssr_sd']:.6f}"
    )


    print(
        "\nRanking-value diagnostic:"
    )

    print(
        "  AFP AR - Adaptive-Random AR =",
        f"{afp['ar_mean'] - adaptive_random['ar_mean']:.6f}"
    )

    print(
        "  AFP SSR - Adaptive-Random SSR =",
        f"{afp['ssr_mean'] - adaptive_random['ssr_mean']:.6f}"
    )


    print(
        "\nFixed-ranking diagnostic:"
    )

    print(
        "  Fixed Top-B AR - Random-B AR =",
        f"{fixed['ar_mean'] - random_b['ar_mean']:.6f}"
    )

    print(
        "  Fixed Top-B SSR - Random-B SSR =",
        f"{fixed['ssr_mean'] - random_b['ssr_mean']:.6f}"
    )


print_causal_diagnostics(
    "webqsp",
    webqsp_summary_df
)


print_causal_diagnostics(
    "cwq",
    cwq_summary_df
)


# ======================================================================
# 17. EXPORT ARTIFACTS
# ======================================================================

WEBQSP_RUN_CSV = (
    COMPARE_DIR
    / "webqsp_validation_controlled_runs.csv"
)

CWQ_RUN_CSV = (
    COMPARE_DIR
    / "cwq_validation_controlled_runs.csv"
)

WEBQSP_SUMMARY_CSV = (
    COMPARE_DIR
    / "webqsp_validation_controlled_summary.csv"
)

CWQ_SUMMARY_CSV = (
    COMPARE_DIR
    / "cwq_validation_controlled_summary.csv"
)

WEBQSP_QUESTION_CSV = (
    COMPARE_DIR
    / "webqsp_validation_controlled_per_question.csv"
)

CWQ_QUESTION_CSV = (
    COMPARE_DIR
    / "cwq_validation_controlled_per_question.csv"
)


webqsp_comparison_df.to_csv(
    WEBQSP_RUN_CSV,
    index=False
)

cwq_comparison_df.to_csv(
    CWQ_RUN_CSV,
    index=False
)

webqsp_summary_df.to_csv(
    WEBQSP_SUMMARY_CSV,
    index=False
)

cwq_summary_df.to_csv(
    CWQ_SUMMARY_CSV,
    index=False
)

webqsp_question_df.to_csv(
    WEBQSP_QUESTION_CSV,
    index=False
)

cwq_question_df.to_csv(
    CWQ_QUESTION_CSV,
    index=False
)


# ======================================================================
# 18. MANIFEST
# ======================================================================

CELL14_MANIFEST = {
    "cell":
        "RQ2_CELL14_VALIDATION_CONTROLLED_COMPARISON",

    "version":
        "rq2_cell14_controlled_comparison_v1",

    "datasets": [
        "webqsp",
        "cwq",
    ],

    "methods": [
        "RoG",
        "Fixed-Top-B",
        "Fixed-Threshold",
        "Random-B",
        "Adaptive-Budget-Random",
        "AFP",
    ],

    "random_seeds":
        RANDOM_SEEDS,

    "selected_parameters": {
        "webqsp": {
            "afp":
                AFP_VALIDATION_SELECTED[
                    "webqsp"
                ],

            "fixed_top_b":
                FIXED_TOP_B_VALIDATION_SELECTED[
                    "webqsp"
                ],

            "fixed_threshold":
                FIXED_THRESHOLD_VALIDATION_SELECTED[
                    "webqsp"
                ],
        },

        "cwq": {
            "afp":
                AFP_VALIDATION_SELECTED[
                    "cwq"
                ],

            "fixed_top_b":
                FIXED_TOP_B_VALIDATION_SELECTED[
                    "cwq"
                ],

            "fixed_threshold":
                FIXED_THRESHOLD_VALIDATION_SELECTED[
                    "cwq"
                ],
        },
    },

    "random_b": {
        "budget":
            "selected_fixed_top_b",

        "selection":
            "uniform_without_replacement",

        "preserve_candidate_order_after_sampling":
            True,
    },

    "adaptive_budget_random": {
        "budget":
            "AFP_actual_retained_count_after_tie_expansion",

        "scorer_use":
            "budget_only",

        "selection":
            "uniform_without_replacement",

        "preserve_candidate_order_after_sampling":
            True,
    },

    "rng": {
        "type":
            "sha256_deterministic_group_rng",

        "python_hash_used":
            False,
    },

    "final_hop_protection":
        True,

    "singleton_bypass":
        True,

    "primary_search_cost":
        "edges_examined",

    "answer_retention_reference":
        "RoG_reachable_questions",

    "gold_used_by_scorer":
        False,

    "gold_used_by_selector":
        False,

    "gold_used_for":
        "post_traversal_validation_reachability_only",

    "selected_scorer_checkpoint_sha256": {
        "webqsp":
            webqsp_ckpt_sha,

        "cwq":
            cwq_ckpt_sha,
    },

    "test_examples_accessed":
        False,

    "complete_afp_frozen":
        False,

    "next_step":
        "RQ2_Cell15_ablations_and_final_development_freeze",
}


CELL14_MANIFEST_PATH = (
    COMPARE_DIR
    / "cell14_validation_controlled_comparison_manifest.json"
)


with open(
    CELL14_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        CELL14_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False
    )


# ======================================================================
# 19. FINAL REPORT
# ======================================================================

print(
    "\n"
    + "=" * 120
)

print(
    "=== RQ2 CELL 14: VALIDATION CONTROLLED COMPARISON COMPLETE ==="
)

print(
    "=" * 120
)


print(
    "\nControlled methods:"
)

print(
    "  RoG"
)

print(
    "  Fixed Top-B"
)

print(
    "  Fixed Threshold"
)

print(
    "  Random-B [42,43,44]"
)

print(
    "  Adaptive-Budget Random [42,43,44]"
)

print(
    "  AFP"
)


print(
    "\nControl gates:"
)

print(
    "  RoG fidelity:                 PASSED"
)

print(
    "  Cell-13 deterministic match:  PASSED"
)

print(
    "  Random selection reproducible: SHA256-based"
)

print(
    "  Gold in scorer/selector:      NO"
)

print(
    "  TEST examples accessed:       NO"
)


print(
    "\nDevelopment status:"
)

print(
    "  Controlled comparison: COMPLETE"
)

print(
    "  Complete AFP frozen:    NO"
)

print(
    "  Next: causal ablations + final development decision"
)


print(
    "\nOutputs:"
)

for path in [
    WEBQSP_RUN_CSV,
    CWQ_RUN_CSV,
    WEBQSP_SUMMARY_CSV,
    CWQ_SUMMARY_CSV,
    WEBQSP_QUESTION_CSV,
    CWQ_QUESTION_CSV,
    CELL14_MANIFEST_PATH,
]:

    print(
        " ",
        path
    )

Cell 14 prerequisites: PASSED
Random baseline seeds: [42, 43, 44]

FROZEN VALIDATION-SELECTED CONFIGURATIONS

WEBQSP
  AFP: {'T': 0.5, 'gamma_min': 0.5}
  Fixed Top-B: 1
  Fixed Threshold: 0.3

CWQ
  AFP: {'T': 0.5, 'gamma_min': 0.5}
  Fixed Top-B: 8
  Fixed Threshold: 0.1


webqsp controlled comparison:   0%|          | 0/246 [00:00<?, ?it/s]


WEBQSP controlled comparison completed.
Elapsed: 0.04 min


cwq controlled comparison:   0%|          | 0/3519 [00:00<?, ?it/s]


CWQ controlled comparison completed.
Elapsed: 2.16 min
WebQSP RoG control fidelity: PASSED
CWQ RoG control fidelity: PASSED
WebQSP deterministic-method Cell-13 fidelity: PASSED
CWQ deterministic-method Cell-13 fidelity: PASSED

WEBQSP CONTROLLED VALIDATION COMPARISON — RUN LEVEL
                method      seed  edges_examined      ssr  reachable_questions  answer_retention  active_prefixes  candidate_branches  decision_hops  avg_requested_budget  fully_tied_decisions  peak_frontier
                   RoG       NaN          341526 0.000000                  205          1.000000             2440                7983              0                   NaN                     0            533
           Fixed-Top-B       NaN          335061 0.018930                  205          1.000000             2011                5962            147              1.000000                    97            145
       Fixed-Threshold       NaN          338714 0.008234                  203          0.99024

In [29]:
# ======================================================================
# RQ2 CELL 13B
# ONE-TIME AFP SELECTOR BOUNDARY EXPANSION
# ======================================================================
#
# WHY THIS CELL EXISTS
# --------------------
# Original Cell 13 selected:
#
#       T = 0.5
#       gamma_min = 0.5
#
# on BOTH WebQSP and CWQ.
#
# Both values were the LOWER BOUNDARIES of the original grid.
#
# Cell 14 further showed that AFP remains extremely conservative:
#
#   WebQSP SSR = 0.001403 at AR = 1.000
#   CWQ    SSR = 0.002092 at AR = 1.000
#
# Therefore we perform ONE predeclared lower-bound expansion before
# freezing the method.
#
# THIS IS THE ONLY BOUNDARY-EXPANSION PASS.
# We will NOT iteratively expand the grid again based on its outcome.
#
# NEW GRID
# --------
# T:
#       0.05, 0.10, 0.20, 0.30, 0.50
#
# gamma_min:
#       0.10, 0.30, 0.50
#
# The original selected point (0.5, 0.5) is included as an anchor and
# MUST exactly reproduce Cell-13 results.
#
# SELECTION RULE IS UNCHANGED:
#
#       maximize SSR subject to AR >= 0.99
#
# fallback:
#
#       maximize AR, then SSR
#
# VALIDATION ONLY.
# NO TEST examples accessed.
# ======================================================================

import hashlib
import json
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


# ======================================================================
# 1. PREREQUISITES
# ======================================================================

required = [
    "CELL13_TUNING_COMPLETE",

    "webqsp_val_plan_rows",
    "cwq_val_plan_rows",

    "webqsp_val_runtime",
    "cwq_val_runtime",

    "webqsp_rog_reference",
    "cwq_rog_reference",

    "build_exact_rog_adjacency",
    "as_entity_list",
    "normalize_relation_plans",

    "traverse_with_policy",
    "final_prefixes_reach_answer",

    "webqsp_tuning_df",
    "cwq_tuning_df",

    "AFP_VALIDATION_SELECTED",
]

missing = [
    name
    for name in required
    if name not in globals()
]

assert not missing, (
    "Missing Cell-13 runtime objects:\n  "
    + "\n  ".join(missing)
)

assert CELL13_TUNING_COMPLETE is True

print("Cell 13B prerequisites: PASSED")


# ======================================================================
# 2. OUTPUT DIRECTORY
# ======================================================================

ROOT = Path(
    "/kaggle/working/step3_rq2_dev_v1"
)

BOUNDARY_DIR = (
    ROOT
    / "11_validation_tuning"
    / "selector_boundary_expansion"
)

BOUNDARY_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ======================================================================
# 3. PREDECLARE THE SINGLE EXPANDED GRID
# ======================================================================

AR_FLOOR_13B = 0.99


EXPANDED_T_GRID = [
    0.05,
    0.10,
    0.20,
    0.30,
    0.50,
]


EXPANDED_GAMMA_MIN_GRID = [
    0.10,
    0.30,
    0.50,
]


EXPANDED_CONFIGS = [
    {
        "config_id":
            f"afp_T{T:g}_g{gamma:g}",

        "family":
            "afp",

        "T":
            float(T),

        "gamma_min":
            float(gamma),
    }

    for T in EXPANDED_T_GRID
    for gamma in EXPANDED_GAMMA_MIN_GRID
]


assert len(
    EXPANDED_CONFIGS
) == 15


assert any(
    np.isclose(
        c["T"],
        0.5
    )
    and
    np.isclose(
        c["gamma_min"],
        0.5
    )
    for c in EXPANDED_CONFIGS
)


print(
    "\n"
    + "=" * 110
)

print(
    "ONE-TIME AFP SELECTOR BOUNDARY EXPANSION"
)

print(
    "=" * 110
)


print(
    "Selection rule: maximize SSR "
    "subject to AR >= 0.99"
)

print(
    "Fallback: maximize AR, then SSR"
)

print(
    "\nT grid:",
    EXPANDED_T_GRID
)

print(
    "gamma_min grid:",
    EXPANDED_GAMMA_MIN_GRID
)

print(
    "Configurations:",
    len(
        EXPANDED_CONFIGS
    )
)

print(
    "\nIMPORTANT: no further automatic "
    "grid expansion after this cell."
)


# ======================================================================
# 4. SPECIFICATION FINGERPRINT
# ======================================================================

CELL13B_SPEC = {
    "version":
        "rq2_cell13b_selector_boundary_expansion_v1",

    "reason":
        "original_selection_hit_both_lower_grid_boundaries",

    "original_selected": {
        "webqsp":
            AFP_VALIDATION_SELECTED[
                "webqsp"
            ],

        "cwq":
            AFP_VALIDATION_SELECTED[
                "cwq"
            ],
    },

    "expanded_T_grid":
        EXPANDED_T_GRID,

    "expanded_gamma_min_grid":
        EXPANDED_GAMMA_MIN_GRID,

    "selection_rule":
        "maximize_ssr_subject_to_ar_0.99",

    "fallback":
        "maximize_ar_then_ssr",

    "one_time_expansion_only":
        True,

    "test_examples_accessed":
        False,
}


CELL13B_SPEC_SHA = hashlib.sha256(
    json.dumps(
        CELL13B_SPEC,
        sort_keys=True,
        separators=(",", ":")
    ).encode(
        "utf-8"
    )
).hexdigest()


print(
    "\nCell 13B specification SHA256:",
    CELL13B_SPEC_SHA
)


# ======================================================================
# 5. FRESH AFP STATISTICS
# ======================================================================

def fresh_13b_stats():

    return {
        config[
            "config_id"
        ]: {
            "active_hop_rows":
                0,

            "active_prefixes":
                0,

            "edges_examined":
                0,

            "candidate_branches":
                0,

            "reachable_plans":
                0,

            "reachable_questions":
                0,

            "decision_hops":
                0,

            "retained_after_decision":
                0,

            "requested_budget_total":
                0,

            "requested_budget_observations":
                0,

            "uncertainty_total":
                0.0,

            "uncertainty_observations":
                0,

            "fully_tied_decisions":
                0,

            "peak_frontier":
                0,
        }

        for config in EXPANDED_CONFIGS
    }


# ======================================================================
# 6. RUN ONE DATASET
# ======================================================================

def run_13b_dataset(
    dataset_name,
    planning_rows,
    question_rows,
    rog_reference
):

    assert len(
        planning_rows
    ) == len(
        question_rows
    )


    stats = fresh_13b_stats()


    start_time = time.time()


    for source_index in tqdm(
        range(
            len(
                planning_rows
            )
        ),
        desc=(
            f"{dataset_name} Cell13B"
        )
    ):

        plan_rec = (
            planning_rows[
                source_index
            ]
        )


        question_rec = (
            question_rows[
                source_index
            ]
        )


        assert str(
            plan_rec[
                "id"
            ]
        ) == str(
            question_rec[
                "id"
            ]
        )


        question_id = str(
            plan_rec[
                "id"
            ]
        )


        question = str(
            question_rec[
                "question"
            ]
        )


        topic_entities = (
            as_entity_list(
                plan_rec[
                    "q_entity"
                ]
            )
        )


        gold_answers = (
            as_entity_list(
                plan_rec[
                    "a_entity"
                ]
            )
        )


        plans = (
            normalize_relation_plans(
                plan_rec[
                    "predicted_paths"
                ]
            )
        )


        adjacency = (
            build_exact_rog_adjacency(
                plan_rec[
                    "graph"
                ]
            )
        )


        # --------------------------------------------------------------
        # Computational caches only.
        # Method metrics are still accumulated independently.
        # --------------------------------------------------------------

        expansion_cache = {}

        score_cache = {}


        question_reachable = {
            config[
                "config_id"
            ]:
                False

            for config in EXPANDED_CONFIGS
        }


        for plan_index, plan in enumerate(
            plans
        ):

            if len(
                plan
            ) == 0:

                continue


            for config in EXPANDED_CONFIGS:

                config_id = (
                    config[
                        "config_id"
                    ]
                )


                result = (
                    traverse_with_policy(
                        dataset_name=
                            dataset_name,

                        question_id=
                            question_id,

                        question=
                            question,

                        adjacency=
                            adjacency,

                        topic_entities=
                            topic_entities,

                        plan_index=
                            plan_index,

                        plan=
                            plan,

                        config=
                            config,

                        expansion_cache=
                            expansion_cache,

                        score_cache=
                            score_cache
                    )
                )


                s = stats[
                    config_id
                ]


                for key in [
                    "active_hop_rows",
                    "active_prefixes",
                    "edges_examined",
                    "candidate_branches",
                    "decision_hops",
                    "retained_after_decision",
                    "requested_budget_total",
                    "requested_budget_observations",
                    "fully_tied_decisions",
                ]:

                    s[
                        key
                    ] += result[
                        key
                    ]


                s[
                    "uncertainty_total"
                ] += result[
                    "uncertainty_total"
                ]


                s[
                    "uncertainty_observations"
                ] += result[
                    "uncertainty_observations"
                ]


                s[
                    "peak_frontier"
                ] = max(
                    s[
                        "peak_frontier"
                    ],
                    result[
                        "peak_frontier"
                    ]
                )


                # ------------------------------------------------------
                # GOLD IS CONSULTED ONLY AFTER TRAVERSAL.
                # ------------------------------------------------------

                reachable = (
                    final_prefixes_reach_answer(
                        final_prefixes=
                            result[
                                "final_prefixes"
                            ],

                        gold_answers=
                            gold_answers
                    )
                )


                if reachable:

                    s[
                        "reachable_plans"
                    ] += 1


                    question_reachable[
                        config_id
                    ] = True


        for config_id, reachable in (
            question_reachable.items()
        ):

            if reachable:

                stats[
                    config_id
                ][
                    "reachable_questions"
                ] += 1


    elapsed = (
        time.time()
        -
        start_time
    )


    print(
        f"\n{dataset_name.upper()} Cell13B completed "
        f"in {elapsed/60:.2f} min"
    )


    # ==================================================================
    # BUILD RESULT TABLE
    # ==================================================================

    rows = []


    rog_edges = float(
        rog_reference[
            "edges_examined"
        ]
    )


    rog_reachable_questions = int(
        rog_reference[
            "reachable_questions"
        ]
    )


    for config in EXPANDED_CONFIGS:

        config_id = (
            config[
                "config_id"
            ]
        )


        s = stats[
            config_id
        ]


        edges = int(
            s[
                "edges_examined"
            ]
        )


        reachable_questions = int(
            s[
                "reachable_questions"
            ]
        )


        ssr = (
            1.0
            -
            edges
            /
            rog_edges
        )


        ar = (
            reachable_questions
            /
            rog_reachable_questions
        )


        avg_budget = (
            s[
                "requested_budget_total"
            ]
            /
            s[
                "requested_budget_observations"
            ]

            if
            s[
                "requested_budget_observations"
            ]
            > 0

            else
            np.nan
        )


        avg_uncertainty = (
            s[
                "uncertainty_total"
            ]
            /
            s[
                "uncertainty_observations"
            ]

            if
            s[
                "uncertainty_observations"
            ]
            > 0

            else
            np.nan
        )


        rows.append(
            {
                "dataset":
                    dataset_name,

                "config_id":
                    config_id,

                "T":
                    float(
                        config[
                            "T"
                        ]
                    ),

                "gamma_min":
                    float(
                        config[
                            "gamma_min"
                        ]
                    ),

                "edges_examined":
                    edges,

                "ssr":
                    float(
                        ssr
                    ),

                "reachable_questions":
                    reachable_questions,

                "rog_reachable_questions":
                    rog_reachable_questions,

                "answer_retention":
                    float(
                        ar
                    ),

                "active_hop_rows":
                    int(
                        s[
                            "active_hop_rows"
                        ]
                    ),

                "active_prefixes":
                    int(
                        s[
                            "active_prefixes"
                        ]
                    ),

                "candidate_branches":
                    int(
                        s[
                            "candidate_branches"
                        ]
                    ),

                "decision_hops":
                    int(
                        s[
                            "decision_hops"
                        ]
                    ),

                "avg_requested_budget":
                    float(
                        avg_budget
                    ),

                "avg_uncertainty":
                    float(
                        avg_uncertainty
                    ),

                "fully_tied_decisions":
                    int(
                        s[
                            "fully_tied_decisions"
                        ]
                    ),

                "peak_frontier":
                    int(
                        s[
                            "peak_frontier"
                        ]
                    ),

                "ar_floor_feasible":
                    bool(
                        ar
                        >=
                        AR_FLOOR_13B
                    ),
            }
        )


    return pd.DataFrame(
        rows
    )


# ======================================================================
# 7. RUN WEBQSP + CWQ
# ======================================================================

webqsp_13b_df = (
    run_13b_dataset(
        dataset_name=
            "webqsp",

        planning_rows=
            webqsp_val_plan_rows,

        question_rows=
            webqsp_val_runtime,

        rog_reference=
            webqsp_rog_reference
    )
)


cwq_13b_df = (
    run_13b_dataset(
        dataset_name=
            "cwq",

        planning_rows=
            cwq_val_plan_rows,

        question_rows=
            cwq_val_runtime,

        rog_reference=
            cwq_rog_reference
    )
)


# ======================================================================
# 8. ORIGINAL-POINT SOFTWARE FIDELITY GATE
# ======================================================================
#
# T=0.5, gamma=0.5 MUST reproduce original Cell 13 exactly.
# ======================================================================

def original_afp_row(
    tuning_df
):

    rows = tuning_df[
        (
            tuning_df[
                "family"
            ]
            ==
            "afp"
        )
        &
        np.isclose(
            tuning_df[
                "T"
            ],
            0.5
        )
        &
        np.isclose(
            tuning_df[
                "gamma_min"
            ],
            0.5
        )
    ]


    assert len(
        rows
    ) == 1


    return rows.iloc[
        0
    ]


def expanded_anchor_row(
    df
):

    rows = df[
        np.isclose(
            df[
                "T"
            ],
            0.5
        )
        &
        np.isclose(
            df[
                "gamma_min"
            ],
            0.5
        )
    ]


    assert len(
        rows
    ) == 1


    return rows.iloc[
        0
    ]


for dataset_name, old_df, new_df in [
    (
        "WebQSP",
        webqsp_tuning_df,
        webqsp_13b_df
    ),

    (
        "CWQ",
        cwq_tuning_df,
        cwq_13b_df
    ),
]:

    old_row = (
        original_afp_row(
            old_df
        )
    )


    new_row = (
        expanded_anchor_row(
            new_df
        )
    )


    assert int(
        old_row[
            "edges_examined"
        ]
    ) == int(
        new_row[
            "edges_examined"
        ]
    )


    assert int(
        old_row[
            "reachable_questions"
        ]
    ) == int(
        new_row[
            "reachable_questions"
        ]
    )


    assert np.isclose(
        float(
            old_row[
                "ssr"
            ]
        ),
        float(
            new_row[
                "ssr"
            ]
        ),
        atol=1e-12
    )


    assert np.isclose(
        float(
            old_row[
                "answer_retention"
            ]
        ),
        float(
            new_row[
                "answer_retention"
            ]
        ),
        atol=1e-12
    )


    print(
        f"{dataset_name} original AFP anchor fidelity: PASSED"
    )


# ======================================================================
# 9. SAME PREDECLARED SELECTION RULE
# ======================================================================

def select_13b(
    df
):

    feasible = df[
        df[
            "answer_retention"
        ]
        >=
        AR_FLOOR_13B
    ].copy()


    if len(
        feasible
    ) > 0:

        mode = (
            "maximize_ssr_subject_to_ar_floor"
        )


        ranked = feasible.sort_values(
            by=[
                "ssr",
                "answer_retention",
                "config_id",
            ],

            ascending=[
                False,
                False,
                True,
            ],

            kind="stable"
        )


    else:

        mode = (
            "fallback_maximize_ar_then_ssr"
        )


        ranked = df.sort_values(
            by=[
                "answer_retention",
                "ssr",
                "config_id",
            ],

            ascending=[
                False,
                False,
                True,
            ],

            kind="stable"
        )


    selected = (
        ranked.iloc[
            0
        ].to_dict()
    )


    selected[
        "selection_mode"
    ] = mode


    return (
        selected,
        ranked
    )


webqsp_13b_selected, webqsp_13b_ranked = (
    select_13b(
        webqsp_13b_df
    )
)


cwq_13b_selected, cwq_13b_ranked = (
    select_13b(
        cwq_13b_df
    )
)


# ======================================================================
# 10. DISPLAY ALL EXPANDED RESULTS
# ======================================================================

DISPLAY_COLUMNS = [
    "config_id",
    "T",
    "gamma_min",
    "ssr",
    "answer_retention",
    "reachable_questions",
    "edges_examined",
    "active_prefixes",
    "decision_hops",
    "avg_requested_budget",
    "avg_uncertainty",
    "fully_tied_decisions",
    "ar_floor_feasible",
]


def show_13b_results(
    dataset_name,
    df
):

    print(
        "\n"
        + "=" * 116
    )

    print(
        f"{dataset_name.upper()} "
        "AFP BOUNDARY-EXPANSION RESULTS"
    )

    print(
        "=" * 116
    )


    ordered = df.sort_values(
        by=[
            "answer_retention",
            "ssr",
        ],

        ascending=[
            False,
            False,
        ]
    )


    print(
        ordered[
            DISPLAY_COLUMNS
        ].to_string(
            index=False,

            float_format=lambda x:
                f"{x:.6f}"
        )
    )


show_13b_results(
    "webqsp",
    webqsp_13b_df
)


show_13b_results(
    "cwq",
    cwq_13b_df
)


# ======================================================================
# 11. COMPARE OLD vs NEW SELECTED AFP
# ======================================================================

def print_selection_comparison(
    dataset_name,
    old_tuning_df,
    new_selected
):

    old = (
        original_afp_row(
            old_tuning_df
        )
    )


    print(
        "\n"
        + "=" * 112
    )

    print(
        f"{dataset_name.upper()} OLD vs EXPANDED AFP"
    )

    print(
        "=" * 112
    )


    print(
        "\nOLD"
    )

    print(
        "  T:",
        float(
            old[
                "T"
            ]
        )
    )

    print(
        "  gamma_min:",
        float(
            old[
                "gamma_min"
            ]
        )
    )

    print(
        "  AR:",
        f"{float(old['answer_retention']):.6f}"
    )

    print(
        "  SSR:",
        f"{float(old['ssr']):.6f}"
    )


    print(
        "\nEXPANDED SELECTED"
    )

    print(
        "  T:",
        new_selected[
            "T"
        ]
    )

    print(
        "  gamma_min:",
        new_selected[
            "gamma_min"
        ]
    )

    print(
        "  AR:",
        f"{new_selected['answer_retention']:.6f}"
    )

    print(
        "  SSR:",
        f"{new_selected['ssr']:.6f}"
    )

    print(
        "  avg budget:",
        f"{new_selected['avg_requested_budget']:.6f}"
    )

    print(
        "  avg uncertainty:",
        f"{new_selected['avg_uncertainty']:.6f}"
    )

    print(
        "  selection mode:",
        new_selected[
            "selection_mode"
        ]
    )


print_selection_comparison(
    "webqsp",
    webqsp_tuning_df,
    webqsp_13b_selected
)


print_selection_comparison(
    "cwq",
    cwq_tuning_df,
    cwq_13b_selected
)


# ======================================================================
# 12. EXPOSE CANDIDATE SELECTED PARAMETERS
# ======================================================================
#
# DO NOT overwrite AFP_VALIDATION_SELECTED yet.
#
# We first inspect the result scientifically.
# ======================================================================

AFP_BOUNDARY_EXPANSION_SELECTED = {
    "webqsp": {
        "T":
            float(
                webqsp_13b_selected[
                    "T"
                ]
            ),

        "gamma_min":
            float(
                webqsp_13b_selected[
                    "gamma_min"
                ]
            ),

        "answer_retention":
            float(
                webqsp_13b_selected[
                    "answer_retention"
                ]
            ),

        "ssr":
            float(
                webqsp_13b_selected[
                    "ssr"
                ]
            ),
    },

    "cwq": {
        "T":
            float(
                cwq_13b_selected[
                    "T"
                ]
            ),

        "gamma_min":
            float(
                cwq_13b_selected[
                    "gamma_min"
                ]
            ),

        "answer_retention":
            float(
                cwq_13b_selected[
                    "answer_retention"
                ]
            ),

        "ssr":
            float(
                cwq_13b_selected[
                    "ssr"
                ]
            ),
    },
}


# ======================================================================
# 13. SAVE ARTIFACTS
# ======================================================================

WEBQSP_13B_CSV = (
    BOUNDARY_DIR
    / "webqsp_afp_boundary_expansion.csv"
)


CWQ_13B_CSV = (
    BOUNDARY_DIR
    / "cwq_afp_boundary_expansion.csv"
)


webqsp_13b_df.to_csv(
    WEBQSP_13B_CSV,
    index=False
)


cwq_13b_df.to_csv(
    CWQ_13B_CSV,
    index=False
)


CELL13B_MANIFEST = {
    "cell":
        "RQ2_CELL13B_AFP_SELECTOR_BOUNDARY_EXPANSION",

    "version":
        "rq2_cell13b_selector_boundary_expansion_v1",

    "spec_sha256":
        CELL13B_SPEC_SHA,

    "trigger":
        "Cell13 selected both lower grid boundaries on both datasets",

    "one_time_expansion_only":
        True,

    "expanded_grid": {
        "T":
            EXPANDED_T_GRID,

        "gamma_min":
            EXPANDED_GAMMA_MIN_GRID,
    },

    "selection_rule":
        "maximize SSR subject to AR >= 0.99; "
        "fallback maximize AR then SSR",

    "selected_candidate": {
        "webqsp":
            AFP_BOUNDARY_EXPANSION_SELECTED[
                "webqsp"
            ],

        "cwq":
            AFP_BOUNDARY_EXPANSION_SELECTED[
                "cwq"
            ],
    },

    "original_anchor_fidelity":
        True,

    "gold_used_by_scorer":
        False,

    "gold_used_by_selector":
        False,

    "gold_used_for":
        "post_traversal_validation_AR_only",

    "test_examples_accessed":
        False,

    "final_AFP_frozen":
        False,
}


CELL13B_MANIFEST_PATH = (
    BOUNDARY_DIR
    / "cell13b_selector_boundary_expansion_manifest.json"
)


with open(
    CELL13B_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        CELL13B_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False
    )


CELL13B_COMPLETE = True


# ======================================================================
# 14. FINAL REPORT
# ======================================================================

print(
    "\n"
    + "=" * 120
)

print(
    "=== RQ2 CELL 13B: "
    "ONE-TIME AFP SELECTOR BOUNDARY EXPANSION COMPLETE ==="
)

print(
    "=" * 120
)


print(
    "\nCandidate selected AFP parameters:"
)

print(
    "  WebQSP:",
    AFP_BOUNDARY_EXPANSION_SELECTED[
        "webqsp"
    ]
)

print(
    "  CWQ:   ",
    AFP_BOUNDARY_EXPANSION_SELECTED[
        "cwq"
    ]
)


print(
    "\nScientific status:"
)

print(
    "  Original Cell-13 anchor reproduced: YES"
)

print(
    "  Validation-only expansion:          YES"
)

print(
    "  Further automatic grid expansion:   NO"
)

print(
    "  AFP parameters overwritten:         NO"
)

print(
    "  TEST examples accessed:             NO"
)

print(
    "  Final AFP frozen:                   NO"
)


print(
    "\nOutputs:"
)

print(
    " ",
    WEBQSP_13B_CSV
)

print(
    " ",
    CWQ_13B_CSV
)

print(
    " ",
    CELL13B_MANIFEST_PATH
)


print(
    "\nNEXT:"
)

print(
    "Review the expanded selector result before "
    "re-running the controlled comparison or proceeding to ablation/freeze."
)

Cell 13B prerequisites: PASSED

ONE-TIME AFP SELECTOR BOUNDARY EXPANSION
Selection rule: maximize SSR subject to AR >= 0.99
Fallback: maximize AR, then SSR

T grid: [0.05, 0.1, 0.2, 0.3, 0.5]
gamma_min grid: [0.1, 0.3, 0.5]
Configurations: 15

IMPORTANT: no further automatic grid expansion after this cell.

Cell 13B specification SHA256: 5e882cc11191c1d477465d938da3a65648a9e60409a1b49cf287c0a84c6d5dd2


webqsp Cell13B:   0%|          | 0/246 [00:00<?, ?it/s]


WEBQSP Cell13B completed in 0.04 min


cwq Cell13B:   0%|          | 0/3519 [00:00<?, ?it/s]


CWQ Cell13B completed in 2.37 min
WebQSP original AFP anchor fidelity: PASSED
CWQ original AFP anchor fidelity: PASSED

WEBQSP AFP BOUNDARY-EXPANSION RESULTS
     config_id        T  gamma_min      ssr  answer_retention  reachable_questions  edges_examined  active_prefixes  decision_hops  avg_requested_budget  avg_uncertainty  fully_tied_decisions  ar_floor_feasible
afp_T0.05_g0.1 0.050000   0.100000 0.010272          1.000000                  205          338018             2293            147              9.891156         0.894528                    97               True
afp_T0.05_g0.3 0.050000   0.300000 0.009349          1.000000                  205          338333             2300            147              9.938776         0.894528                    97               True
afp_T0.05_g0.5 0.050000   0.500000 0.007973          1.000000                  205          338803             2312            147             10.020408         0.894528                    97               Tr

In [30]:
# ======================================================================
# RQ2 CELL 14B
# CONTROLLED VALIDATION COMPARISON WITH EXPANDED AFP SELECTOR
# ======================================================================
#
# RUN AS A NEW CELL.
# DO NOT REPLACE CELL 13, CELL 13B, OR CELL 14.
# DO NOT RESTART THE KERNEL.
#
# PURPOSE
# -------
# Re-run the controlled validation comparison after the single
# predeclared AFP selector boundary expansion.
#
# UPDATED AFP:
#
#   WebQSP:
#       T = 0.05
#       gamma_min = 0.10
#
#   CWQ:
#       T = 0.10
#       gamma_min = 0.10
#
# UNCHANGED:
#   RoG
#   Fixed Top-B
#   Fixed Threshold
#   Random-B
#   scorer
#   Feature-v2
#   plans
#   graphs
#   traversal
#   seeds
#   final-hop protection
#
# Adaptive-Budget Random inherits the NEW AFP adaptive retained counts.
#
# VALIDATION ONLY.
# NO TEST examples accessed.
# NO further hyperparameter search.
# ======================================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd


# ======================================================================
# 1. HARD PREREQUISITES
# ======================================================================

required = [
    "CELL13B_COMPLETE",
    "AFP_BOUNDARY_EXPANSION_SELECTED",

    "FIXED_TOP_B_VALIDATION_SELECTED",
    "FIXED_THRESHOLD_VALIDATION_SELECTED",

    "webqsp_13b_df",
    "cwq_13b_df",

    "webqsp_val_plan_rows",
    "cwq_val_plan_rows",

    "webqsp_val_runtime",
    "cwq_val_runtime",

    "webqsp_rog_reference",
    "cwq_rog_reference",

    # Cell 14 controlled-comparison functions
    "run_controlled_dataset",
    "summarize_method_runs",
    "assert_rog_control",
    "display_results",
    "print_causal_diagnostics",

    # Selected scorer artifact identity
    "webqsp_ckpt_sha",
    "cwq_ckpt_sha",
]

missing = [
    name
    for name in required
    if name not in globals()
]

assert not missing, (
    "Missing required Cell 13B / Cell 14 objects:\n  "
    + "\n  ".join(missing)
)

assert CELL13B_COMPLETE is True

print("Cell 14B prerequisites: PASSED")


# ======================================================================
# 2. OUTPUT DIRECTORY
# ======================================================================

ROOT = Path(
    "/kaggle/working/step3_rq2_dev_v1"
)

COMPARE14B_DIR = (
    ROOT
    / "12_validation_comparison"
    / "expanded_afp"
)

COMPARE14B_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ======================================================================
# 3. PROMOTE THE ONE-TIME EXPANSION RESULT FOR CONTROLLED EVALUATION
# ======================================================================
#
# This does NOT freeze final AFP.
#
# These are now the development-selected selector candidates because:
#
#   1. the expansion was triggered only because the original optimum
#      landed at both lower grid boundaries;
#   2. expansion was declared one-time;
#   3. the same AR >= 0.99 selection rule was retained;
#   4. the original grid anchor reproduced exactly.
# ======================================================================

AFP_14B_SELECTED = {
    "webqsp": {
        "T":
            float(
                AFP_BOUNDARY_EXPANSION_SELECTED[
                    "webqsp"
                ][
                    "T"
                ]
            ),

        "gamma_min":
            float(
                AFP_BOUNDARY_EXPANSION_SELECTED[
                    "webqsp"
                ][
                    "gamma_min"
                ]
            ),
    },

    "cwq": {
        "T":
            float(
                AFP_BOUNDARY_EXPANSION_SELECTED[
                    "cwq"
                ][
                    "T"
                ]
            ),

        "gamma_min":
            float(
                AFP_BOUNDARY_EXPANSION_SELECTED[
                    "cwq"
                ][
                    "gamma_min"
                ]
            ),
    },
}


assert AFP_14B_SELECTED[
    "webqsp"
] == {
    "T": 0.05,
    "gamma_min": 0.1,
}


assert AFP_14B_SELECTED[
    "cwq"
] == {
    "T": 0.1,
    "gamma_min": 0.1,
}


print(
    "\nExpanded AFP selected for Cell 14B:"
)

print(
    "  WebQSP:",
    AFP_14B_SELECTED[
        "webqsp"
    ]
)

print(
    "  CWQ:   ",
    AFP_14B_SELECTED[
        "cwq"
    ]
)


print(
    "\nUnchanged baselines:"
)

print(
    "  WebQSP Fixed Top-B:",
    FIXED_TOP_B_VALIDATION_SELECTED[
        "webqsp"
    ]
)

print(
    "  WebQSP threshold:",
    FIXED_THRESHOLD_VALIDATION_SELECTED[
        "webqsp"
    ]
)

print(
    "  CWQ Fixed Top-B:",
    FIXED_TOP_B_VALIDATION_SELECTED[
        "cwq"
    ]
)

print(
    "  CWQ threshold:",
    FIXED_THRESHOLD_VALIDATION_SELECTED[
        "cwq"
    ]
)


# ======================================================================
# 4. RANDOM SEEDS REMAIN FROZEN
# ======================================================================

RANDOM_SEEDS_14B = [
    42,
    43,
    44,
]


# ======================================================================
# 5. BUILD UPDATED METHOD SPECS
# ======================================================================

def build_method_specs_14b(
    dataset_name
):

    afp_params = (
        AFP_14B_SELECTED[
            dataset_name
        ]
    )


    selected_B = int(
        FIXED_TOP_B_VALIDATION_SELECTED[
            dataset_name
        ]
    )


    selected_tau = float(
        FIXED_THRESHOLD_VALIDATION_SELECTED[
            dataset_name
        ]
    )


    methods = [
        # --------------------------------------------------------------
        # Unpruned control
        # --------------------------------------------------------------
        {
            "method":
                "RoG",

            "family":
                "rog",

            "seed":
                None,
        },

        # --------------------------------------------------------------
        # Frozen fixed-budget baseline
        # --------------------------------------------------------------
        {
            "method":
                "Fixed-Top-B",

            "family":
                "fixed_top_b",

            "B":
                selected_B,

            "seed":
                None,
        },

        # --------------------------------------------------------------
        # Frozen threshold baseline
        # --------------------------------------------------------------
        {
            "method":
                "Fixed-Threshold",

            "family":
                "fixed_threshold",

            "tau":
                selected_tau,

            "seed":
                None,
        },

        # --------------------------------------------------------------
        # Expanded-validation-selected AFP
        # --------------------------------------------------------------
        {
            "method":
                "AFP",

            "family":
                "afp",

            "T":
                float(
                    afp_params[
                        "T"
                    ]
                ),

            "gamma_min":
                float(
                    afp_params[
                        "gamma_min"
                    ]
                ),

            "seed":
                None,
        },
    ]


    # ------------------------------------------------------------------
    # Random-B:
    # unchanged selected Fixed Top-B budget.
    # ------------------------------------------------------------------

    for seed in RANDOM_SEEDS_14B:

        methods.append(
            {
                "method":
                    "Random-B",

                "family":
                    "random_b",

                "B":
                    selected_B,

                "seed":
                    int(
                        seed
                    ),
            }
        )


    # ------------------------------------------------------------------
    # Adaptive-Budget Random:
    #
    # uses NEW AFP T/gamma_min.
    #
    # At each encountered dynamic group it receives AFP's actual
    # tie-expanded retained count, but randomly chooses WHICH candidates
    # survive.
    # ------------------------------------------------------------------

    for seed in RANDOM_SEEDS_14B:

        methods.append(
            {
                "method":
                    "Adaptive-Budget-Random",

                "family":
                    "adaptive_budget_random",

                "T":
                    float(
                        afp_params[
                            "T"
                        ]
                    ),

                "gamma_min":
                    float(
                        afp_params[
                            "gamma_min"
                        ]
                    ),

                "seed":
                    int(
                        seed
                    ),
            }
        )


    return methods


WEBQSP_METHODS_14B = (
    build_method_specs_14b(
        "webqsp"
    )
)


CWQ_METHODS_14B = (
    build_method_specs_14b(
        "cwq"
    )
)


# ======================================================================
# 6. RUN CONTROLLED COMPARISON
# ======================================================================

print(
    "\n"
    + "=" * 118
)

print(
    "RUNNING CELL 14B CONTROLLED VALIDATION COMPARISON"
)

print(
    "=" * 118
)


webqsp_comparison_14b_df, webqsp_question_14b_df = (
    run_controlled_dataset(
        dataset_name=
            "webqsp",

        planning_rows=
            webqsp_val_plan_rows,

        question_rows=
            webqsp_val_runtime,

        rog_reference=
            webqsp_rog_reference,

        method_specs=
            WEBQSP_METHODS_14B
    )
)


cwq_comparison_14b_df, cwq_question_14b_df = (
    run_controlled_dataset(
        dataset_name=
            "cwq",

        planning_rows=
            cwq_val_plan_rows,

        question_rows=
            cwq_val_runtime,

        rog_reference=
            cwq_rog_reference,

        method_specs=
            CWQ_METHODS_14B
    )
)


# ======================================================================
# 7. RoG CONTROL FIDELITY
# ======================================================================

assert_rog_control(
    "WebQSP",
    webqsp_comparison_14b_df,
    webqsp_rog_reference
)


assert_rog_control(
    "CWQ",
    cwq_comparison_14b_df,
    cwq_rog_reference
)


print(
    "\nCell 14B RoG control fidelity: PASSED"
)


# ======================================================================
# 8. FIXED BASELINES MUST MATCH ORIGINAL CELL 14 EXACTLY
# ======================================================================

def deterministic_baseline_match(
    dataset_name,
    old_df,
    new_df
):

    for method in [
        "RoG",
        "Fixed-Top-B",
        "Fixed-Threshold",
    ]:

        old_rows = old_df[
            old_df[
                "method"
            ]
            ==
            method
        ]


        new_rows = new_df[
            new_df[
                "method"
            ]
            ==
            method
        ]


        assert len(
            old_rows
        ) == 1

        assert len(
            new_rows
        ) == 1


        old = old_rows.iloc[
            0
        ]

        new = new_rows.iloc[
            0
        ]


        for key in [
            "edges_examined",
            "reachable_questions",
            "active_prefixes",
            "candidate_branches",
        ]:

            assert int(
                old[
                    key
                ]
            ) == int(
                new[
                    key
                ]
            ), (
                f"{dataset_name} {method} mismatch "
                f"for {key}"
            )


    print(
        f"{dataset_name} unchanged deterministic "
        "baseline fidelity: PASSED"
    )


deterministic_baseline_match(
    "WebQSP",
    webqsp_comparison_df,
    webqsp_comparison_14b_df
)


deterministic_baseline_match(
    "CWQ",
    cwq_comparison_df,
    cwq_comparison_14b_df
)


# ======================================================================
# 9. RANDOM-B MUST ALSO MATCH ORIGINAL CELL 14
# ======================================================================
#
# Random-B parameters and RNG are unchanged.
# ======================================================================

def random_b_match(
    dataset_name,
    old_df,
    new_df
):

    for seed in RANDOM_SEEDS_14B:

        old_rows = old_df[
            (
                old_df[
                    "method"
                ]
                ==
                "Random-B"
            )
            &
            (
                old_df[
                    "seed"
                ]
                ==
                seed
            )
        ]


        new_rows = new_df[
            (
                new_df[
                    "method"
                ]
                ==
                "Random-B"
            )
            &
            (
                new_df[
                    "seed"
                ]
                ==
                seed
            )
        ]


        assert len(
            old_rows
        ) == 1

        assert len(
            new_rows
        ) == 1


        old = old_rows.iloc[
            0
        ]

        new = new_rows.iloc[
            0
        ]


        for key in [
            "edges_examined",
            "reachable_questions",
            "active_prefixes",
            "candidate_branches",
        ]:

            assert int(
                old[
                    key
                ]
            ) == int(
                new[
                    key
                ]
            ), (
                f"{dataset_name} Random-B seed {seed} "
                f"mismatch for {key}"
            )


    print(
        f"{dataset_name} Random-B reproducibility: PASSED"
    )


random_b_match(
    "WebQSP",
    webqsp_comparison_df,
    webqsp_comparison_14b_df
)


random_b_match(
    "CWQ",
    cwq_comparison_df,
    cwq_comparison_14b_df
)


# ======================================================================
# 10. NEW AFP MUST EXACTLY MATCH CELL 13B SELECTED POINT
# ======================================================================

def selected_13b_row(
    dataset_name,
    df
):

    params = (
        AFP_14B_SELECTED[
            dataset_name
        ]
    )


    rows = df[
        np.isclose(
            df[
                "T"
            ],
            params[
                "T"
            ]
        )
        &
        np.isclose(
            df[
                "gamma_min"
            ],
            params[
                "gamma_min"
            ]
        )
    ]


    assert len(
        rows
    ) == 1


    return rows.iloc[
        0
    ]


def afp_13b_fidelity(
    dataset_name,
    comparison_df,
    boundary_df
):

    afp_rows = comparison_df[
        comparison_df[
            "method"
        ]
        ==
        "AFP"
    ]


    assert len(
        afp_rows
    ) == 1


    comparison = afp_rows.iloc[
        0
    ]


    boundary = (
        selected_13b_row(
            dataset_name,
            boundary_df
        )
    )


    assert int(
        comparison[
            "edges_examined"
        ]
    ) == int(
        boundary[
            "edges_examined"
        ]
    )


    assert int(
        comparison[
            "reachable_questions"
        ]
    ) == int(
        boundary[
            "reachable_questions"
        ]
    )


    assert np.isclose(
        float(
            comparison[
                "ssr"
            ]
        ),
        float(
            boundary[
                "ssr"
            ]
        ),
        atol=1e-12
    )


    assert np.isclose(
        float(
            comparison[
                "answer_retention"
            ]
        ),
        float(
            boundary[
                "answer_retention"
            ]
        ),
        atol=1e-12
    )


    print(
        f"{dataset_name} AFP Cell-13B fidelity: PASSED"
    )


afp_13b_fidelity(
    "webqsp",
    webqsp_comparison_14b_df,
    webqsp_13b_df
)


afp_13b_fidelity(
    "cwq",
    cwq_comparison_14b_df,
    cwq_13b_df
)


# ======================================================================
# 11. SUMMARY TABLES
# ======================================================================

webqsp_summary_14b_df = (
    summarize_method_runs(
        webqsp_comparison_14b_df
    )
)


cwq_summary_14b_df = (
    summarize_method_runs(
        cwq_comparison_14b_df
    )
)


display_results(
    "webqsp",
    webqsp_comparison_14b_df,
    webqsp_summary_14b_df
)


display_results(
    "cwq",
    cwq_comparison_14b_df,
    cwq_summary_14b_df
)


# ======================================================================
# 12. UPDATED CAUSAL DIAGNOSTICS
# ======================================================================

print_causal_diagnostics(
    "webqsp",
    webqsp_summary_14b_df
)


print_causal_diagnostics(
    "cwq",
    cwq_summary_14b_df
)


# ======================================================================
# 13. EXTRA PARETO / DOMINANCE DIAGNOSTIC
# ======================================================================

def single_summary_row(
    summary_df,
    method
):

    rows = summary_df[
        summary_df[
            "method"
        ]
        ==
        method
    ]


    assert len(
        rows
    ) == 1


    return rows.iloc[
        0
    ]


def dominance_report(
    dataset_name,
    summary_df
):

    afp = (
        single_summary_row(
            summary_df,
            "AFP"
        )
    )


    fixed_b = (
        single_summary_row(
            summary_df,
            "Fixed-Top-B"
        )
    )


    fixed_threshold = (
        single_summary_row(
            summary_df,
            "Fixed-Threshold"
        )
    )


    adaptive_random = (
        single_summary_row(
            summary_df,
            "Adaptive-Budget-Random"
        )
    )


    print(
        "\n"
        + "=" * 116
    )

    print(
        f"{dataset_name.upper()} DEVELOPMENT DOMINANCE DIAGNOSTIC"
    )

    print(
        "=" * 116
    )


    print(
        "\nAFP:"
    )

    print(
        f"  AR  = {afp['ar_mean']:.6f}"
    )

    print(
        f"  SSR = {afp['ssr_mean']:.6f}"
    )


    print(
        "\nFixed Top-B:"
    )

    print(
        f"  AR  = {fixed_b['ar_mean']:.6f}"
    )

    print(
        f"  SSR = {fixed_b['ssr_mean']:.6f}"
    )


    print(
        "\nFixed Threshold:"
    )

    print(
        f"  AR  = {fixed_threshold['ar_mean']:.6f}"
    )

    print(
        f"  SSR = {fixed_threshold['ssr_mean']:.6f}"
    )


    print(
        "\nAdaptive-Budget Random:"
    )

    print(
        f"  AR  = {adaptive_random['ar_mean']:.6f}"
    )

    print(
        f"  SSR = {adaptive_random['ssr_mean']:.6f}"
    )


    # --------------------------------------------------------------
    # Simple deterministic dominance statement for Fixed Top-B vs AFP
    # --------------------------------------------------------------

    fixed_dominates_afp = (
        float(
            fixed_b[
                "ar_mean"
            ]
        )
        >=
        float(
            afp[
                "ar_mean"
            ]
        )
        and
        float(
            fixed_b[
                "ssr_mean"
            ]
        )
        >=
        float(
            afp[
                "ssr_mean"
            ]
        )
        and
        (
            float(
                fixed_b[
                    "ar_mean"
                ]
            )
            >
            float(
                afp[
                    "ar_mean"
                ]
            )
            or
            float(
                fixed_b[
                    "ssr_mean"
                ]
            )
            >
            float(
                afp[
                    "ssr_mean"
                ]
            )
        )
    )


    print(
        "\nFixed Top-B Pareto-dominates AFP:",
        bool(
            fixed_dominates_afp
        )
    )


    print(
        "\nAFP minus Adaptive-Random:"
    )

    print(
        "  AR delta:",
        f"{float(afp['ar_mean'] - adaptive_random['ar_mean']):.6f}"
    )

    print(
        "  SSR delta:",
        f"{float(afp['ssr_mean'] - adaptive_random['ssr_mean']):.6f}"
    )


dominance_report(
    "webqsp",
    webqsp_summary_14b_df
)


dominance_report(
    "cwq",
    cwq_summary_14b_df
)


# ======================================================================
# 14. EXPORT ARTIFACTS
# ======================================================================

WEBQSP_14B_RUN_CSV = (
    COMPARE14B_DIR
    / "webqsp_expanded_afp_controlled_runs.csv"
)


CWQ_14B_RUN_CSV = (
    COMPARE14B_DIR
    / "cwq_expanded_afp_controlled_runs.csv"
)


WEBQSP_14B_SUMMARY_CSV = (
    COMPARE14B_DIR
    / "webqsp_expanded_afp_controlled_summary.csv"
)


CWQ_14B_SUMMARY_CSV = (
    COMPARE14B_DIR
    / "cwq_expanded_afp_controlled_summary.csv"
)


WEBQSP_14B_Q_CSV = (
    COMPARE14B_DIR
    / "webqsp_expanded_afp_per_question.csv"
)


CWQ_14B_Q_CSV = (
    COMPARE14B_DIR
    / "cwq_expanded_afp_per_question.csv"
)


webqsp_comparison_14b_df.to_csv(
    WEBQSP_14B_RUN_CSV,
    index=False
)


cwq_comparison_14b_df.to_csv(
    CWQ_14B_RUN_CSV,
    index=False
)


webqsp_summary_14b_df.to_csv(
    WEBQSP_14B_SUMMARY_CSV,
    index=False
)


cwq_summary_14b_df.to_csv(
    CWQ_14B_SUMMARY_CSV,
    index=False
)


webqsp_question_14b_df.to_csv(
    WEBQSP_14B_Q_CSV,
    index=False
)


cwq_question_14b_df.to_csv(
    CWQ_14B_Q_CSV,
    index=False
)


# ======================================================================
# 15. CELL 14B MANIFEST
# ======================================================================

CELL14B_MANIFEST = {
    "cell":
        "RQ2_CELL14B_EXPANDED_AFP_CONTROLLED_COMPARISON",

    "version":
        "rq2_cell14b_expanded_afp_controlled_comparison_v1",

    "selector_source":
        "one_time_Cell13B_boundary_expansion",

    "afp_selected": {
        "webqsp":
            AFP_14B_SELECTED[
                "webqsp"
            ],

        "cwq":
            AFP_14B_SELECTED[
                "cwq"
            ],
    },

    "fixed_top_b": {
        "webqsp":
            int(
                FIXED_TOP_B_VALIDATION_SELECTED[
                    "webqsp"
                ]
            ),

        "cwq":
            int(
                FIXED_TOP_B_VALIDATION_SELECTED[
                    "cwq"
                ]
            ),
    },

    "fixed_threshold": {
        "webqsp":
            float(
                FIXED_THRESHOLD_VALIDATION_SELECTED[
                    "webqsp"
                ]
            ),

        "cwq":
            float(
                FIXED_THRESHOLD_VALIDATION_SELECTED[
                    "cwq"
                ]
            ),
    },

    "random_seeds":
        RANDOM_SEEDS_14B,

    "adaptive_budget_random":
        {
            "budget_source":
                "new_AFP_actual_retained_count_after_tie_expansion",

            "branch_selection":
                "uniform_without_replacement",
        },

    "no_further_hyperparameter_search":
        True,

    "feature_version":
        "afp_features_v2_masked_entity_semantics",

    "selected_scorer_checkpoint_sha256": {
        "webqsp":
            webqsp_ckpt_sha,

        "cwq":
            cwq_ckpt_sha,
    },

    "control_gates": {
        "rog_fidelity":
            True,

        "unchanged_fixed_baselines_match_Cell14":
            True,

        "random_b_matches_Cell14":
            True,

        "afp_matches_Cell13B":
            True,
    },

    "gold_used_by_scorer":
        False,

    "gold_used_by_selector":
        False,

    "test_examples_accessed":
        False,

    "final_AFP_frozen":
        False,

    "next":
        "Cell15_ablations_and_final_development_decision",
}


CELL14B_MANIFEST_PATH = (
    COMPARE14B_DIR
    / "cell14b_expanded_afp_controlled_comparison_manifest.json"
)


with open(
    CELL14B_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        CELL14B_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False
    )


CELL14B_COMPLETE = True


# ======================================================================
# 16. FINAL REPORT
# ======================================================================

print(
    "\n"
    + "=" * 120
)

print(
    "=== RQ2 CELL 14B: "
    "EXPANDED-AFP CONTROLLED VALIDATION COMPARISON COMPLETE ==="
)

print(
    "=" * 120
)


print(
    "\nAFP development-selected candidates:"
)

print(
    "  WebQSP:",
    AFP_14B_SELECTED[
        "webqsp"
    ]
)

print(
    "  CWQ:   ",
    AFP_14B_SELECTED[
        "cwq"
    ]
)


print(
    "\nControl status:"
)

print(
    "  RoG fidelity:                    PASSED"
)

print(
    "  Fixed baselines unchanged:       PASSED"
)

print(
    "  Random-B reproducibility:         PASSED"
)

print(
    "  AFP matches Cell 13B selection:   PASSED"
)

print(
    "  Further hyperparameter search:    NO"
)

print(
    "  TEST examples accessed:           NO"
)

print(
    "  Final AFP frozen:                 NO"
)


print(
    "\nNext:"
)

print(
    "Cell 15 — causal ablations + final development decision/freeze."
)


print(
    "\nManifest:"
)

print(
    CELL14B_MANIFEST_PATH
)

Cell 14B prerequisites: PASSED

Expanded AFP selected for Cell 14B:
  WebQSP: {'T': 0.05, 'gamma_min': 0.1}
  CWQ:    {'T': 0.1, 'gamma_min': 0.1}

Unchanged baselines:
  WebQSP Fixed Top-B: 1
  WebQSP threshold: 0.3
  CWQ Fixed Top-B: 8
  CWQ threshold: 0.1

RUNNING CELL 14B CONTROLLED VALIDATION COMPARISON


webqsp controlled comparison:   0%|          | 0/246 [00:00<?, ?it/s]


WEBQSP controlled comparison completed.
Elapsed: 0.04 min


cwq controlled comparison:   0%|          | 0/3519 [00:00<?, ?it/s]


CWQ controlled comparison completed.
Elapsed: 2.21 min
WebQSP RoG control fidelity: PASSED
CWQ RoG control fidelity: PASSED

Cell 14B RoG control fidelity: PASSED
WebQSP unchanged deterministic baseline fidelity: PASSED
CWQ unchanged deterministic baseline fidelity: PASSED
WebQSP Random-B reproducibility: PASSED
CWQ Random-B reproducibility: PASSED
webqsp AFP Cell-13B fidelity: PASSED
cwq AFP Cell-13B fidelity: PASSED

WEBQSP CONTROLLED VALIDATION COMPARISON — RUN LEVEL
                method      seed  edges_examined      ssr  reachable_questions  answer_retention  active_prefixes  candidate_branches  decision_hops  avg_requested_budget  fully_tied_decisions  peak_frontier
                   RoG       NaN          341526 0.000000                  205          1.000000             2440                7983              0                   NaN                     0            533
           Fixed-Top-B       NaN          335061 0.018930                  205          1.000000            

##  Matched retraining feature-group ablations, selector/component ablations and final development freeze

In [32]:
# ======================================================================
# MATCHED FEATURE-FAMILY RETRAINING ABLATIONS
# ======================================================================
# PURPOSE
# -------
# Proper feature-family causal ablation by RETRAINING the scorer after
# removing one feature family.
#
# VARIANTS
# --------
# 1. Full Feature-v2 retrain control
# 2. -Semantic
# 3. -Path
# 4. -Structural
# 5. -Progress
#
# METHODOLOGICAL CONTROLS
# -----------------------
# - exact frozen train Feature-v2 matrices
# - same feasible-decision training population
# - same selected hidden dimension
# - same branch BCE
# - same optimizer
# - same LR / weight decay / epochs
# - same seed = 42
# - no class weighting
# - no early stopping
# - no ablation-specific hyperparameter tuning
# - same already-selected AFP selector parameters
#
# IMPORTANT SOFTWARE FIX
# ----------------------
# Persisted AFPScorer and AFPFeatureStandardizer were originally defined
# with global AFP_INPUT_DIM = 27.
#
# Feature-family ablations have dimensions 19 / 20 / 23.
#
# We DO NOT rewrite either algorithm.
#
# Instead, the EXACT persisted class source is instantiated in an
# isolated namespace with AFP_INPUT_DIM equal to the ablation dimension.
#
# Full 27-D behavior is software-gated against the frozen standardizer.
#
# VALIDATION ONLY.
# NO TEST access.
# NO final AFP freeze in this cell.
# ======================================================================


# ======================================================================
# 0. IMPORTS
# ======================================================================

import ast
import hashlib
import inspect
import json
import math
import re
import time
import typing
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm.auto import tqdm


# ======================================================================
# 1. HARD PREREQUISITES
# ======================================================================

required = [
    "CELL14B_COMPLETE",
    "CELL13B_COMPLETE",

    "AFP_14B_SELECTED",

    "AFP_RUNTIME_FEATURE_EXTRACTOR",
    "AFP_RUNTIME_SEMANTIC_ENCODER",
    "AFP_RUNTIME_FEATURE_VERSION",
    "AFP_RUNTIME_FEATURE_DIM",

    "webqsp_val_plan_rows",
    "cwq_val_plan_rows",

    "webqsp_val_runtime",
    "cwq_val_runtime",

    "webqsp_rog_reference",
    "cwq_rog_reference",

    "webqsp_comparison_14b_df",
    "cwq_comparison_14b_df",

    "build_exact_rog_adjacency",
    "as_entity_list",
    "normalize_relation_plans",

    "get_cached_relation_expansion",
    "frontier_cache_key",
    "select_policy_indices",
    "final_prefixes_reach_answer",

    "webqsp_ckpt_obj",
    "cwq_ckpt_obj",

    "webqsp_ckpt_sha",
    "cwq_ckpt_sha",
]

missing = [
    name
    for name in required
    if name not in globals()
]

assert not missing, (
    "Missing required prior-cell objects:\n  "
    + "\n  ".join(missing)
)

assert CELL14B_COMPLETE is True
assert CELL13B_COMPLETE is True
assert int(AFP_RUNTIME_FEATURE_DIM) == 27

print("Cell 15A prerequisites: PASSED")
print("Feature version:", AFP_RUNTIME_FEATURE_VERSION)
print("Feature dimension:", AFP_RUNTIME_FEATURE_DIM)


# ======================================================================
# 2. PATHS
# ======================================================================

ROOT = Path(
    "/kaggle/working/step3_rq2_dev_v1"
)

FEATURE_ROOT = (
    ROOT
    / "03_features"
)

ABLATION_DIR = (
    ROOT
    / "13_feature_ablations"
)

ABLATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

NOTEBOOK_PATH = Path(
    "/kaggle/input/notebooks/"
    "mdsadmansamikhan/rog-ap/"
    "__notebook__.ipynb"
)

assert FEATURE_ROOT.exists()
assert NOTEBOOK_PATH.exists()

print("Feature root:", FEATURE_ROOT)
print("Ablation output:", ABLATION_DIR)


# ======================================================================
# 3. VERIFIED FROZEN TRAIN REFERENCES
# ======================================================================

EXPECTED_TRAIN = {
    "webqsp": {
        "branches": 18437,
        "groups": 1457,
        "positive": 7301,
        "negative": 11136,
    },

    "cwq": {
        "branches": 218544,
        "groups": 15937,
        "positive": 52746,
        "negative": 165798,
    },
}


# ======================================================================
# 4. SHA256 HELPER
# ======================================================================

def sha256_file(path, chunk_size=1024 * 1024):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            block = f.read(chunk_size)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


# ======================================================================
# 5. DISCOVER EXACT FROZEN TRAIN NPZs
# ======================================================================

def inspect_npz_candidate(path, expected):

    try:
        data = np.load(
            path,
            allow_pickle=False
        )

    except Exception:
        return None

    try:

        files = set(data.files)

        if not {
            "X",
            "y",
            "group_ptr",
        }.issubset(files):
            return None

        X = data["X"]
        y = data["y"]
        group_ptr = data["group_ptr"]

        if tuple(X.shape) != (
            expected["branches"],
            27
        ):
            return None

        if tuple(y.shape) != (
            expected["branches"],
        ):
            return None

        if len(group_ptr) != (
            expected["groups"] + 1
        ):
            return None

        y_int = np.asarray(
            y,
            dtype=np.int64
        )

        positive = int(
            np.sum(y_int == 1)
        )

        negative = int(
            np.sum(y_int == 0)
        )

        if positive != expected["positive"]:
            return None

        if negative != expected["negative"]:
            return None

        return {
            "path": path,
            "shape": tuple(X.shape),
            "groups": int(len(group_ptr) - 1),
            "positive": positive,
            "negative": negative,
            "sha256": sha256_file(path),
        }

    finally:
        data.close()


def discover_training_npz(dataset_name):

    expected = EXPECTED_TRAIN[
        dataset_name
    ]

    candidates = []

    for path in FEATURE_ROOT.rglob(
        "*.npz"
    ):

        result = inspect_npz_candidate(
            path,
            expected
        )

        if result is not None:
            candidates.append(result)

    assert len(candidates) >= 1, (
        f"{dataset_name}: no train NPZ matched "
        "the frozen training reference."
    )

    print(
        f"\n{dataset_name.upper()} "
        "matching train NPZ candidate(s):"
    )

    for row in candidates:

        print(" ", row["path"])
        print("    SHA:", row["sha256"])

    if len(candidates) == 1:
        return candidates[0]["path"]

    hashes = {
        row["sha256"]
        for row in candidates
    }

    if len(hashes) == 1:

        chosen = sorted(
            [
                row["path"]
                for row in candidates
            ],
            key=lambda p: str(p)
        )[0]

        print(
            f"{dataset_name}: byte-identical "
            f"duplicate copies; using {chosen}"
        )

        return chosen

    # Scientific-array identity gate.
    reference_path = candidates[0]["path"]

    with np.load(
        reference_path,
        allow_pickle=False
    ) as ref:

        ref_X = np.asarray(ref["X"])
        ref_y = np.asarray(ref["y"])
        ref_ptr = np.asarray(ref["group_ptr"])

    for row in candidates[1:]:

        with np.load(
            row["path"],
            allow_pickle=False
        ) as other:

            assert np.array_equal(
                ref_X,
                other["X"]
            )

            assert np.array_equal(
                ref_y,
                other["y"]
            )

            assert np.array_equal(
                ref_ptr,
                other["group_ptr"]
            )

    chosen = sorted(
        [
            row["path"]
            for row in candidates
        ],
        key=lambda p: str(p)
    )[0]

    print(
        f"{dataset_name}: scientifically identical "
        f"copies; using {chosen}"
    )

    return chosen


WEBQSP_TRAIN_NPZ = (
    discover_training_npz(
        "webqsp"
    )
)

CWQ_TRAIN_NPZ = (
    discover_training_npz(
        "cwq"
    )
)

print(
    "\nFrozen train feature files identified: PASSED"
)


# ======================================================================
# 6. LOAD FROZEN TRAIN MATRICES
# ======================================================================

def load_train_matrix(
    path,
    dataset_name
):

    expected = EXPECTED_TRAIN[
        dataset_name
    ]

    with np.load(
        path,
        allow_pickle=False
    ) as data:

        X = np.asarray(
            data["X"],
            dtype=np.float32
        )

        y = np.asarray(
            data["y"],
            dtype=np.float32
        )

        group_ptr = np.asarray(
            data["group_ptr"],
            dtype=np.int64
        )

    assert X.shape == (
        expected["branches"],
        27
    )

    assert y.shape == (
        expected["branches"],
    )

    assert len(group_ptr) == (
        expected["groups"] + 1
    )

    assert int(
        np.sum(y == 1)
    ) == expected["positive"]

    assert int(
        np.sum(y == 0)
    ) == expected["negative"]

    assert np.all(
        np.isfinite(X)
    )

    return {
        "X": X,
        "y": y,
        "group_ptr": group_ptr,
    }


webqsp_train_ablation = (
    load_train_matrix(
        WEBQSP_TRAIN_NPZ,
        "webqsp"
    )
)

cwq_train_ablation = (
    load_train_matrix(
        CWQ_TRAIN_NPZ,
        "cwq"
    )
)


print(
    "\nTraining matrices:"
)

print(
    "  WebQSP:",
    webqsp_train_ablation["X"].shape,
    "| groups:",
    len(
        webqsp_train_ablation["group_ptr"]
    ) - 1
)

print(
    "  CWQ:   ",
    cwq_train_ablation["X"].shape,
    "| groups:",
    len(
        cwq_train_ablation["group_ptr"]
    ) - 1
)

print(
    "Frozen training-cache gate: PASSED"
)


# ======================================================================
# 7. FEATURE-FAMILY DEFINITIONS
# ======================================================================

FEATURE_GROUPS = {
    "semantic": list(range(0, 8)),
    "path": list(range(8, 15)),
    "structural": list(range(15, 23)),
    "progress": list(range(23, 27)),
}

all_feature_columns = (
    FEATURE_GROUPS["semantic"]
    + FEATURE_GROUPS["path"]
    + FEATURE_GROUPS["structural"]
    + FEATURE_GROUPS["progress"]
)

assert sorted(
    all_feature_columns
) == list(range(27))

assert len(
    set(all_feature_columns)
) == 27


ABLATION_VARIANTS = {
    "full_v2_retrain":
        list(range(27)),

    "minus_semantic":
        [
            i
            for i in range(27)
            if i not in FEATURE_GROUPS["semantic"]
        ],

    "minus_path":
        [
            i
            for i in range(27)
            if i not in FEATURE_GROUPS["path"]
        ],

    "minus_structural":
        [
            i
            for i in range(27)
            if i not in FEATURE_GROUPS["structural"]
        ],

    "minus_progress":
        [
            i
            for i in range(27)
            if i not in FEATURE_GROUPS["progress"]
        ],
}


EXPECTED_DIMS = {
    "full_v2_retrain": 27,
    "minus_semantic": 19,
    "minus_path": 20,
    "minus_structural": 19,
    "minus_progress": 23,
}


for name, columns in (
    ABLATION_VARIANTS.items()
):

    assert len(columns) == EXPECTED_DIMS[name]


print(
    "\nFeature ablation variants:"
)

for name, columns in (
    ABLATION_VARIANTS.items()
):

    print(
        f"  {name:<20} "
        f"{len(columns):>2} features"
    )


# ======================================================================
# 8. LOAD PERSISTED NOTEBOOK
# ======================================================================

with open(
    NOTEBOOK_PATH,
    "r",
    encoding="utf-8"
) as f:

    notebook_15a = json.load(f)


# ======================================================================
# 9. RECOVER EXACT PERSISTED CLASS SOURCES
# ======================================================================

def recover_latest_class_source(
    notebook,
    class_name
):

    matches = []

    for cell_idx, cell in enumerate(
        notebook["cells"]
    ):

        if cell.get(
            "cell_type"
        ) != "code":
            continue

        source = "".join(
            cell.get(
                "source",
                []
            )
        )

        if (
            f"class {class_name}"
            not in source
        ):
            continue

        try:
            tree = ast.parse(source)

        except Exception:
            continue

        for node in tree.body:

            if (
                isinstance(node, ast.ClassDef)
                and
                node.name == class_name
            ):

                class_source = (
                    ast.get_source_segment(
                        source,
                        node
                    )
                )

                if class_source is not None:

                    matches.append(
                        {
                            "cell_idx":
                                int(cell_idx),

                            "class_source":
                                class_source,

                            "cell_source":
                                source,
                        }
                    )

    assert len(matches) >= 1, (
        f"Could not recover {class_name} "
        "from persisted notebook."
    )

    # Latest persisted definition.
    return matches[-1]


scorer_recovery = (
    recover_latest_class_source(
        notebook_15a,
        "AFPScorer"
    )
)

standardizer_recovery = (
    recover_latest_class_source(
        notebook_15a,
        "AFPFeatureStandardizer"
    )
)


SCORER_SOURCE_15A = (
    scorer_recovery[
        "class_source"
    ]
)

STANDARDIZER_SOURCE_15A = (
    standardizer_recovery[
        "class_source"
    ]
)


print(
    "\nRecovered persisted classes:"
)

print(
    "  AFPScorer: cell",
    scorer_recovery[
        "cell_idx"
    ]
)

print(
    "  AFPFeatureStandardizer: cell",
    standardizer_recovery[
        "cell_idx"
    ]
)


# ======================================================================
# 10. RECOVER LITERAL AFP_* DEPENDENCIES FROM CLASS CELLS
# ======================================================================

def recover_literal_afp_constants(
    source
):

    recovered = {}

    try:
        tree = ast.parse(source)

    except Exception:
        return recovered

    for node in tree.body:

        if isinstance(
            node,
            (
                ast.Assign,
                ast.AnnAssign,
            )
        ):

            targets = []

            if isinstance(
                node,
                ast.Assign
            ):
                targets = node.targets
                value_node = node.value

            else:
                targets = [node.target]
                value_node = node.value

            if value_node is None:
                continue

            for target in targets:

                if not isinstance(
                    target,
                    ast.Name
                ):
                    continue

                name = target.id

                if not name.startswith(
                    "AFP_"
                ):
                    continue

                try:
                    value = ast.literal_eval(
                        value_node
                    )

                except Exception:
                    continue

                recovered[name] = value

    return recovered


PERSISTED_AFP_CONSTANTS = {}

PERSISTED_AFP_CONSTANTS.update(
    recover_literal_afp_constants(
        scorer_recovery[
            "cell_source"
        ]
    )
)

PERSISTED_AFP_CONSTANTS.update(
    recover_literal_afp_constants(
        standardizer_recovery[
            "cell_source"
        ]
    )
)


print(
    "\nRecovered persisted class constants:"
)

for key in sorted(
    PERSISTED_AFP_CONSTANTS
):

    print(
        f"  {key} = "
        f"{PERSISTED_AFP_CONSTANTS[key]!r}"
    )


# ======================================================================
# 11. DIMENSION-AWARE EXACT CLASS FACTORIES
# ======================================================================
#
# The original classes were created with AFP_INPUT_DIM = 27.
#
# For ablation dimensions we recreate the EXACT source in an isolated
# globals namespace and override ONLY AFP_INPUT_DIM.
# ======================================================================

_DIMENSIONAL_STANDARDIZER_CLASSES = {}
_DIMENSIONAL_SCORER_CLASSES = {}


def build_exact_class_namespace(
    input_dim
):

    input_dim = int(input_dim)

    # Copy current globals to preserve harmless imported dependencies,
    # then inject persisted literals and the dimension-specific invariant.
    ns = dict(globals())

    ns.update(
        {
            "np": np,
            "torch": torch,
            "nn": nn,
            "F": F,
            "Path": Path,
            "typing": typing,
        }
    )

    ns.update(
        PERSISTED_AFP_CONSTANTS
    )

    # Critical dimension specialization.
    ns[
        "AFP_INPUT_DIM"
    ] = input_dim

    return ns


def get_exact_standardizer_class_for_dim(
    input_dim
):

    input_dim = int(input_dim)

    if input_dim in (
        _DIMENSIONAL_STANDARDIZER_CLASSES
    ):

        return (
            _DIMENSIONAL_STANDARDIZER_CLASSES[
                input_dim
            ]
        )

    ns = build_exact_class_namespace(
        input_dim
    )

    exec(
        STANDARDIZER_SOURCE_15A,
        ns
    )

    cls = ns[
        "AFPFeatureStandardizer"
    ]

    _DIMENSIONAL_STANDARDIZER_CLASSES[
        input_dim
    ] = cls

    return cls


def get_exact_scorer_class_for_dim(
    input_dim
):

    input_dim = int(input_dim)

    if input_dim in (
        _DIMENSIONAL_SCORER_CLASSES
    ):

        return (
            _DIMENSIONAL_SCORER_CLASSES[
                input_dim
            ]
        )

    ns = build_exact_class_namespace(
        input_dim
    )

    exec(
        SCORER_SOURCE_15A,
        ns
    )

    cls = ns[
        "AFPScorer"
    ]

    _DIMENSIONAL_SCORER_CLASSES[
        input_dim
    ] = cls

    return cls


ABLATION_INPUT_DIMS = sorted(
    set(
        EXPECTED_DIMS.values()
    )
)


assert ABLATION_INPUT_DIMS == [
    19,
    20,
    23,
    27,
]


print(
    "\nPreparing exact dimension-aware classes:"
)


for dim in ABLATION_INPUT_DIMS:

    std_cls = (
        get_exact_standardizer_class_for_dim(
            dim
        )
    )

    scorer_cls = (
        get_exact_scorer_class_for_dim(
            dim
        )
    )

    print(
        f"  {dim:>2}-D -> "
        f"{std_cls.__name__}, "
        f"{scorer_cls.__name__}"
    )


print(
    "Dimension-aware exact classes: READY"
)


# ======================================================================
# 12. DIMENSION-AWARE SOFTWARE SANITY GATE
# ======================================================================

for dim in ABLATION_INPUT_DIMS:

    rng = np.random.default_rng(
        15000 + dim
    )

    X_sanity = rng.normal(
        size=(
            8,
            dim
        )
    ).astype(
        np.float32
    )

    StdClass = (
        get_exact_standardizer_class_for_dim(
            dim
        )
    )

    std = StdClass()

    std.fit(
        X_sanity
    )

    Z_sanity = np.asarray(
        std.transform(
            X_sanity
        ),
        dtype=np.float32
    )

    assert Z_sanity.shape == (
        8,
        dim
    )

    assert np.all(
        np.isfinite(
            Z_sanity
        )
    )

    ScorerClass = (
        get_exact_scorer_class_for_dim(
            dim
        )
    )

    # The actual hidden dimension is dataset-specific later.
    scorer = ScorerClass(
        input_dim=dim,
        hidden_dim=32,
        dropout=0.0
    )

    with torch.inference_mode():

        out = scorer(
            torch.from_numpy(
                Z_sanity
            )
        )

    assert out.reshape(
        -1
    ).shape == (
        8,
    )


print(
    "Dimension-aware scorer/standardizer "
    "sanity gate: PASSED"
)


# ======================================================================
# 13. STANDARDIZER HELPERS
# ======================================================================

def fit_exact_standardizer(
    X
):

    X = np.asarray(
        X,
        dtype=np.float32
    )

    assert X.ndim == 2

    input_dim = int(
        X.shape[1]
    )

    StandardizerClass = (
        get_exact_standardizer_class_for_dim(
            input_dim
        )
    )

    standardizer = (
        StandardizerClass()
    )

    assert hasattr(
        standardizer,
        "fit"
    )

    assert hasattr(
        standardizer,
        "transform"
    )

    standardizer.fit(
        X
    )

    Z = np.asarray(
        standardizer.transform(
            X
        ),
        dtype=np.float32
    )

    assert Z.shape == X.shape

    assert np.all(
        np.isfinite(Z)
    )

    return (
        standardizer,
        Z
    )


def transform_exact_standardizer(
    standardizer,
    X
):

    X = np.asarray(
        X,
        dtype=np.float32
    )

    Z = np.asarray(
        standardizer.transform(
            X
        ),
        dtype=np.float32
    )

    assert Z.shape == X.shape

    assert np.all(
        np.isfinite(Z)
    )

    return Z


# ======================================================================
# 14. STANDARDIZER STATE EXTRACTION
# ======================================================================

def extract_standardizer_arrays(
    standardizer,
    expected_dim
):

    candidate_dicts = []

    if hasattr(
        standardizer,
        "state_dict"
    ):

        try:

            state = (
                standardizer.state_dict()
            )

            if isinstance(
                state,
                dict
            ):
                candidate_dicts.append(
                    state
                )

        except Exception:
            pass

    if hasattr(
        standardizer,
        "__dict__"
    ):

        candidate_dicts.append(
            vars(
                standardizer
            )
        )

    mean = None
    std = None

    for mapping in candidate_dicts:

        for key, value in (
            mapping.items()
        ):

            key_lower = str(
                key
            ).lower()

            try:

                if torch.is_tensor(
                    value
                ):

                    arr = (
                        value
                        .detach()
                        .cpu()
                        .numpy()
                    )

                else:

                    arr = np.asarray(
                        value
                    )

            except Exception:
                continue

            if arr.shape != (
                expected_dim,
            ):
                continue

            if "mean" in key_lower:

                mean = np.asarray(
                    arr,
                    dtype=np.float32
                )

            if (
                "std" in key_lower
                or
                "scale" in key_lower
            ):

                std = np.asarray(
                    arr,
                    dtype=np.float32
                )

    assert mean is not None, (
        "Could not recover fitted standardizer mean."
    )

    assert std is not None, (
        "Could not recover fitted standardizer std."
    )

    assert mean.shape == (
        expected_dim,
    )

    assert std.shape == (
        expected_dim,
    )

    return {
        "mean": mean,
        "std": std,
    }


def checkpoint_standardizer_arrays(
    checkpoint,
    expected_dim
):

    assert (
        "standardizer_state"
        in checkpoint
    )

    state = checkpoint[
        "standardizer_state"
    ]

    assert isinstance(
        state,
        dict
    )

    mean = None
    std = None

    for key, value in (
        state.items()
    ):

        key_lower = str(
            key
        ).lower()

        if torch.is_tensor(
            value
        ):

            arr = (
                value
                .detach()
                .cpu()
                .numpy()
            )

        else:

            arr = np.asarray(
                value
            )

        if arr.shape != (
            expected_dim,
        ):
            continue

        if "mean" in key_lower:

            mean = np.asarray(
                arr,
                dtype=np.float32
            )

        if (
            "std" in key_lower
            or
            "scale" in key_lower
        ):

            std = np.asarray(
                arr,
                dtype=np.float32
            )

    assert mean is not None
    assert std is not None

    return {
        "mean": mean,
        "std": std,
    }


# ======================================================================
# 15. 27-D STANDARDIZER SOFTWARE FIDELITY
# ======================================================================

for (
    dataset_name,
    train_data,
    checkpoint
) in [
    (
        "WebQSP",
        webqsp_train_ablation,
        webqsp_ckpt_obj
    ),
    (
        "CWQ",
        cwq_train_ablation,
        cwq_ckpt_obj
    ),
]:

    standardizer, _ = (
        fit_exact_standardizer(
            train_data["X"]
        )
    )

    fitted = (
        extract_standardizer_arrays(
            standardizer,
            27
        )
    )

    frozen = (
        checkpoint_standardizer_arrays(
            checkpoint,
            27
        )
    )

    mean_diff = float(
        np.max(
            np.abs(
                fitted["mean"].astype(
                    np.float64
                )
                -
                frozen["mean"].astype(
                    np.float64
                )
            )
        )
    )

    std_diff = float(
        np.max(
            np.abs(
                fitted["std"].astype(
                    np.float64
                )
                -
                frozen["std"].astype(
                    np.float64
                )
            )
        )
    )

    print(
        f"\n{dataset_name} full-v2 "
        "standardizer fidelity"
    )

    print(
        "  max mean diff:",
        mean_diff
    )

    print(
        "  max std diff: ",
        std_diff
    )

    assert mean_diff <= 1e-7
    assert std_diff <= 1e-7


print(
    "\nTrain-only standardizer "
    "reproduction: PASSED"
)


# ======================================================================
# 16. RECOVER SELECTED-CHECKPOINT TRAINING METADATA
# ======================================================================

def checkpoint_training_metadata(
    checkpoint,
    dataset_name
):

    required_keys = [
        "epochs",
        "learning_rate",
        "weight_decay",
        "loss_name",
        "seed",
        "hidden_dim",
    ]

    missing_keys = [
        key
        for key in required_keys
        if key not in checkpoint
    ]

    assert not missing_keys, (
        f"{dataset_name}: checkpoint missing "
        f"{missing_keys}"
    )

    return {
        "epochs":
            int(
                checkpoint["epochs"]
            ),

        "learning_rate":
            float(
                checkpoint[
                    "learning_rate"
                ]
            ),

        "weight_decay":
            float(
                checkpoint[
                    "weight_decay"
                ]
            ),

        "loss_name":
            str(
                checkpoint[
                    "loss_name"
                ]
            ),

        "seed":
            int(
                checkpoint["seed"]
            ),

        "hidden_dim":
            int(
                checkpoint[
                    "hidden_dim"
                ]
            ),
    }


WEBQSP_TRAIN_META = (
    checkpoint_training_metadata(
        webqsp_ckpt_obj,
        "WebQSP"
    )
)

CWQ_TRAIN_META = (
    checkpoint_training_metadata(
        cwq_ckpt_obj,
        "CWQ"
    )
)


print(
    "\nRecovered selected-checkpoint "
    "training metadata:"
)

print(
    "  WebQSP:",
    WEBQSP_TRAIN_META
)

print(
    "  CWQ:   ",
    CWQ_TRAIN_META
)


for (
    dataset_name,
    meta,
    expected_H
) in [
    (
        "WebQSP",
        WEBQSP_TRAIN_META,
        32
    ),
    (
        "CWQ",
        CWQ_TRAIN_META,
        64
    ),
]:

    assert meta[
        "epochs"
    ] == 80

    assert np.isclose(
        meta[
            "learning_rate"
        ],
        1e-3
    )

    assert np.isclose(
        meta[
            "weight_decay"
        ],
        1e-4
    )

    assert meta[
        "loss_name"
    ] == "branch_bce"

    assert meta[
        "seed"
    ] == 42

    assert meta[
        "hidden_dim"
    ] == expected_H


print(
    "Frozen training-metadata gate: PASSED"
)


# ======================================================================
# 17. RECOVER ORIGINAL OPTIMIZER FROM NOTEBOOK
# ======================================================================

optimizer_hits = []


for cell_idx, cell in enumerate(
    notebook_15a["cells"]
):

    if cell.get(
        "cell_type"
    ) != "code":
        continue

    source = "".join(
        cell.get(
            "source",
            []
        )
    )

    if "torch.optim." not in source:
        continue

    training_context = (
        "branch_bce" in source
        or
        "AFPScorer" in source
        or
        "loss_name" in source
    )

    if not training_context:
        continue

    names = re.findall(
        r"torch\.optim\.(AdamW|Adam)\s*\(",
        source
    )

    for name in names:

        optimizer_hits.append(
            {
                "cell_idx":
                    int(cell_idx),

                "optimizer":
                    name,
            }
        )


print(
    "\nPersisted AFP-training optimizer evidence:"
)

for row in optimizer_hits:

    print(
        "  cell",
        row["cell_idx"],
        "->",
        row["optimizer"]
    )


optimizer_names = {
    row["optimizer"]
    for row in optimizer_hits
}


assert len(
    optimizer_names
) == 1, (
    "Could not uniquely recover original AFP optimizer. "
    "Do NOT guess."
)


AFP_ABLATION_OPTIMIZER_NAME = (
    next(
        iter(optimizer_names)
    )
)


if AFP_ABLATION_OPTIMIZER_NAME == "AdamW":

    AFP_ABLATION_OPTIMIZER_CLASS = (
        torch.optim.AdamW
    )

elif AFP_ABLATION_OPTIMIZER_NAME == "Adam":

    AFP_ABLATION_OPTIMIZER_CLASS = (
        torch.optim.Adam
    )

else:

    raise AssertionError(
        "Unsupported recovered optimizer."
    )


print(
    "Recovered optimizer:",
    AFP_ABLATION_OPTIMIZER_NAME
)


# ======================================================================
# 18. SCHEDULER GATE
# ======================================================================

optimizer_cells = {
    row["cell_idx"]
    for row in optimizer_hits
}

scheduler_evidence = []

for cell_idx in sorted(
    optimizer_cells
):

    source = "".join(
        notebook_15a[
            "cells"
        ][
            cell_idx
        ].get(
            "source",
            []
        )
    )

    if (
        "lr_scheduler" in source
        or
        "scheduler.step" in source
    ):

        scheduler_evidence.append(
            cell_idx
        )


assert not scheduler_evidence, (
    "Scheduler detected in recovered AFP "
    "training cell. Stop rather than omit it."
)

print(
    "Learning-rate scheduler in AFP training: NONE"
)

print(
    "Original optimizer recovery: PASSED"
)


# ======================================================================
# 19. TRAINING DEVICE
# ======================================================================

ABLATION_DEVICE = torch.device(
    "cpu"
)

print(
    "Matched ablation training device:",
    ABLATION_DEVICE
)


# ======================================================================
# 20. DETERMINISTIC SEED HELPER
# ======================================================================

def set_training_seed(seed):

    seed = int(seed)

    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            seed
        )


# ======================================================================
# 21. MATCHED RETRAINING FUNCTION
# ======================================================================

def train_ablation_model(
    dataset_name,
    X_full,
    y,
    columns,
    meta,
    variant_name
):

    columns = list(columns)

    X = np.asarray(
        X_full[
            :,
            columns
        ],
        dtype=np.float32
    )

    y_np = np.asarray(
        y,
        dtype=np.float32
    )

    input_dim = int(
        X.shape[1]
    )

    assert input_dim == len(
        columns
    )

    # --------------------------------------------------------------
    # Fresh TRAIN-only standardizer.
    # --------------------------------------------------------------

    standardizer, Z = (
        fit_exact_standardizer(
            X
        )
    )

    # --------------------------------------------------------------
    # Same initialization seed for every matched variant.
    # --------------------------------------------------------------

    set_training_seed(
        meta["seed"]
    )

    ScorerClass = (
        get_exact_scorer_class_for_dim(
            input_dim
        )
    )

    model = ScorerClass(
        input_dim=input_dim,
        hidden_dim=int(
            meta["hidden_dim"]
        ),
        dropout=0.0
    ).to(
        ABLATION_DEVICE
    )

    model.train()

    optimizer = (
        AFP_ABLATION_OPTIMIZER_CLASS(
            model.parameters(),
            lr=float(
                meta["learning_rate"]
            ),
            weight_decay=float(
                meta["weight_decay"]
            )
        )
    )

    criterion = (
        nn.BCEWithLogitsLoss()
    )

    X_tensor = (
        torch.from_numpy(Z)
        .to(ABLATION_DEVICE)
    )

    y_tensor = (
        torch.from_numpy(y_np)
        .to(ABLATION_DEVICE)
    )

    losses = []

    for epoch in range(
        int(meta["epochs"])
    ):

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = (
            model(
                X_tensor
            )
            .reshape(-1)
        )

        assert logits.shape == (
            len(y_np),
        )

        loss = criterion(
            logits,
            y_tensor
        )

        loss.backward()

        optimizer.step()

        losses.append(
            float(
                loss
                .detach()
                .cpu()
                .item()
            )
        )

    model.eval()

    standardizer_state = (
        extract_standardizer_arrays(
            standardizer,
            input_dim
        )
    )

    print(
        f"{dataset_name:<7} "
        f"{variant_name:<20} "
        f"dim={input_dim:<2} "
        f"loss0={losses[0]:.6f} "
        f"loss80={losses[-1]:.6f}"
    )

    return {
        "model":
            model,

        "standardizer":
            standardizer,

        "standardizer_state":
            standardizer_state,

        "columns":
            columns,

        "input_dim":
            input_dim,

        "loss_history":
            losses,
    }


# ======================================================================
# 22. TRAIN ALL MATCHED VARIANTS
# ======================================================================

print(
    "\n"
    + "=" * 116
)

print(
    "TRAINING MATCHED FEATURE ABLATIONS"
)

print(
    "=" * 116
)


ABLATION_MODELS = {
    "webqsp": {},
    "cwq": {},
}


training_start = time.time()


for (
    dataset_name,
    train_data,
    meta
) in [
    (
        "webqsp",
        webqsp_train_ablation,
        WEBQSP_TRAIN_META
    ),
    (
        "cwq",
        cwq_train_ablation,
        CWQ_TRAIN_META
    ),
]:

    print(
        f"\n{dataset_name.upper()}"
    )

    for variant_name, columns in (
        ABLATION_VARIANTS.items()
    ):

        result = (
            train_ablation_model(
                dataset_name=
                    dataset_name,

                X_full=
                    train_data["X"],

                y=
                    train_data["y"],

                columns=
                    columns,

                meta=
                    meta,

                variant_name=
                    variant_name
            )
        )

        ABLATION_MODELS[
            dataset_name
        ][
            variant_name
        ] = result


print(
    "\nMatched ablation training elapsed:",
    f"{(time.time()-training_start)/60:.2f} min"
)


# ======================================================================
# 23. SAVE ABLATION CHECKPOINTS
# ======================================================================

ABLATION_CHECKPOINT_PATHS = {
    "webqsp": {},
    "cwq": {},
}


for dataset_name in [
    "webqsp",
    "cwq",
]:

    dataset_dir = (
        ABLATION_DIR
        / dataset_name
    )

    dataset_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    meta = (
        WEBQSP_TRAIN_META
        if dataset_name == "webqsp"
        else
        CWQ_TRAIN_META
    )

    for variant_name, bundle in (
        ABLATION_MODELS[
            dataset_name
        ].items()
    ):

        path = (
            dataset_dir
            / f"{variant_name}.pt"
        )

        torch.save(
            {
                "dataset":
                    dataset_name,

                "variant":
                    variant_name,

                "feature_version":
                    AFP_RUNTIME_FEATURE_VERSION,

                "feature_columns":
                    bundle["columns"],

                "input_dim":
                    bundle["input_dim"],

                "hidden_dim":
                    meta["hidden_dim"],

                "seed":
                    meta["seed"],

                "epochs":
                    meta["epochs"],

                "learning_rate":
                    meta["learning_rate"],

                "weight_decay":
                    meta["weight_decay"],

                "loss_name":
                    "branch_bce",

                "optimizer":
                    AFP_ABLATION_OPTIMIZER_NAME,

                "model_state_dict":
                    bundle[
                        "model"
                    ].state_dict(),

                "standardizer_state":
                    bundle[
                        "standardizer_state"
                    ],

                "final_training_loss":
                    bundle[
                        "loss_history"
                    ][-1],

                "ablation_specific_tuning":
                    False,

                "test_examples_accessed":
                    False,
            },
            path
        )

        ABLATION_CHECKPOINT_PATHS[
            dataset_name
        ][
            variant_name
        ] = path


print(
    "\nAblation checkpoints saved: PASSED"
)


# ======================================================================
# 24. RUNTIME FEATURE EXTRACTOR SIGNATURE
# ======================================================================

RUNTIME_EXTRACTOR_SIGNATURE = (
    inspect.signature(
        AFP_RUNTIME_FEATURE_EXTRACTOR
    )
)


print(
    "\nRuntime Feature-v2 extractor signature:"
)

print(
    " ",
    RUNTIME_EXTRACTOR_SIGNATURE
)


# ======================================================================
# 25. SAFE ADAPTER TO EXACT RUNTIME FEATURE EXTRACTOR
# ======================================================================
#
# We do not guess the exact persisted function signature.
#
# Only arguments explicitly accepted by the recovered callable are
# supplied. Unknown REQUIRED arguments trigger a hard failure.
# ======================================================================

def call_runtime_feature_extractor(
    dataset_name,
    question_id,
    question,
    plan,
    hop,
    candidate_rows
):

    available = {
        "dataset_name":
            dataset_name,

        "dataset":
            dataset_name,

        "question_id":
            question_id,

        "qid":
            question_id,

        "question":
            question,

        "plan":
            list(plan),

        "hop":
            int(hop),

        "candidate_rows":
            candidate_rows,

        "semantic_encoder":
            AFP_RUNTIME_SEMANTIC_ENCODER,

        "entity_name_map":
            None,
    }

    sig = inspect.signature(
        AFP_RUNTIME_FEATURE_EXTRACTOR
    )

    kwargs = {}

    has_var_keyword = False

    for name, parameter in (
        sig.parameters.items()
    ):

        if parameter.kind == (
            inspect.Parameter.VAR_KEYWORD
        ):

            has_var_keyword = True
            continue

        if parameter.kind == (
            inspect.Parameter.VAR_POSITIONAL
        ):
            continue

        if name in available:

            kwargs[name] = (
                available[name]
            )

            continue

        if parameter.default is (
            inspect.Parameter.empty
        ):

            raise AssertionError(
                "Unknown required runtime feature "
                f"extractor argument: {name}. "
                "Do not guess the mapping."
            )

    # For **kwargs-style wrapper, supplying our normal names is safe.
    if has_var_keyword:

        for key, value in available.items():

            if key not in kwargs:

                # Avoid supplying aliases simultaneously.
                if key in {
                    "dataset",
                    "qid",
                }:
                    continue

                kwargs[key] = value

    X = (
        AFP_RUNTIME_FEATURE_EXTRACTOR(
            **kwargs
        )
    )

    return np.asarray(
        X,
        dtype=np.float32
    )


# ======================================================================
# 26. FULL FEATURE CACHE FOR DYNAMIC VALIDATION GROUPS
# ======================================================================

def get_full_runtime_group_features(
    dataset_name,
    question_id,
    question,
    plan_index,
    plan,
    hop,
    active_prefixes,
    candidate_rows,
    feature_cache
):

    key = frontier_cache_key(
        plan_index=
            plan_index,

        hop=
            hop,

        active_prefixes=
            active_prefixes
    )

    if key in feature_cache:

        cached = feature_cache[
            key
        ]

        assert cached.shape == (
            len(candidate_rows),
            27
        )

        return cached

    X = (
        call_runtime_feature_extractor(
            dataset_name=
                dataset_name,

            question_id=
                question_id,

            question=
                question,

            plan=
                plan,

            hop=
                hop,

            candidate_rows=
                candidate_rows
        )
    )

    assert X.shape == (
        len(candidate_rows),
        27
    ), (
        "Runtime Feature-v2 shape mismatch: "
        f"{X.shape}"
    )

    assert np.all(
        np.isfinite(X)
    )

    feature_cache[
        key
    ] = X

    return X


# ======================================================================
# 27. ABLATION SCORING
# ======================================================================

def score_ablation_group(
    bundle,
    X_full
):

    columns = bundle[
        "columns"
    ]

    X_subset = np.asarray(
        X_full[
            :,
            columns
        ],
        dtype=np.float32
    )

    assert X_subset.shape[1] == (
        bundle["input_dim"]
    )

    Z = (
        transform_exact_standardizer(
            bundle[
                "standardizer"
            ],
            X_subset
        )
    )

    with torch.inference_mode():

        tensor = (
            torch.from_numpy(Z)
            .to(ABLATION_DEVICE)
        )

        logits = (
            bundle["model"](
                tensor
            )
            .reshape(-1)
            .detach()
            .cpu()
            .numpy()
            .astype(np.float32)
        )

    assert logits.shape == (
        X_subset.shape[0],
    )

    assert np.all(
        np.isfinite(logits)
    )

    return logits


# ======================================================================
# 28. ABLATION TRAVERSAL
# ======================================================================

def traverse_ablation_variant(
    dataset_name,
    bundle,
    selector_params,
    question_id,
    question,
    adjacency,
    topic_entities,
    plan_index,
    plan,
    expansion_cache,
    feature_cache,
    logit_cache
):

    active = [
        (
            str(entity),
        )
        for entity in topic_entities
    ]

    L = len(plan)

    active_hop_rows = 0
    active_prefixes = 0

    edges_examined = 0
    candidate_branches = 0

    decision_hops = 0

    retained_after_decision = 0

    requested_budget_total = 0
    requested_budget_observations = 0

    uncertainty_total = 0.0
    uncertainty_observations = 0

    fully_tied_decisions = 0

    peak_frontier = len(active)

    for hop, target_relation in enumerate(
        plan
    ):

        if not active:
            break

        active_hop_rows += 1
        active_prefixes += len(active)

        expansion = (
            get_cached_relation_expansion(
                adjacency=
                    adjacency,

                plan_index=
                    plan_index,

                hop=
                    hop,

                target_relation=
                    target_relation,

                active_prefixes=
                    active,

                expansion_cache=
                    expansion_cache
            )
        )

        candidates = expansion[
            "candidates"
        ]

        candidate_rows = expansion[
            "candidate_rows"
        ]

        edges_examined += int(
            expansion[
                "edges_cost"
            ]
        )

        candidate_branches += len(
            candidates
        )

        if not candidates:

            active = []
            break

        # --------------------------------------------------------------
        # FINAL-HOP PROTECTION
        # --------------------------------------------------------------

        if hop == L - 1:

            active = candidates

            peak_frontier = max(
                peak_frontier,
                len(active)
            )

            continue

        # --------------------------------------------------------------
        # SINGLETON BYPASS
        # --------------------------------------------------------------

        if len(candidates) <= 1:

            active = candidates

            peak_frontier = max(
                peak_frontier,
                len(active)
            )

            continue

        decision_hops += 1

        frontier_key = (
            frontier_cache_key(
                plan_index=
                    plan_index,

                hop=
                    hop,

                active_prefixes=
                    active
            )
        )

        variant_key = (
            id(
                bundle["model"]
            ),
            frontier_key,
        )

        if variant_key in logit_cache:

            logits = (
                logit_cache[
                    variant_key
                ]
            )

        else:

            X_full = (
                get_full_runtime_group_features(
                    dataset_name=
                        dataset_name,

                    question_id=
                        question_id,

                    question=
                        question,

                    plan_index=
                        plan_index,

                    plan=
                        plan,

                    hop=
                        hop,

                    active_prefixes=
                        active,

                    candidate_rows=
                        candidate_rows,

                    feature_cache=
                        feature_cache
                )
            )

            logits = (
                score_ablation_group(
                    bundle=
                        bundle,

                    X_full=
                        X_full
                )
            )

            logit_cache[
                variant_key
            ] = logits

        selector_config = {
            "family":
                "afp",

            "T":
                float(
                    selector_params["T"]
                ),

            "gamma_min":
                float(
                    selector_params[
                        "gamma_min"
                    ]
                ),
        }

        selection = (
            select_policy_indices(
                selector_config,
                logits
            )
        )

        selected_indices = (
            selection[
                "selected_indices"
            ]
        )

        active = [
            candidates[index]
            for index in selected_indices
        ]

        retained_after_decision += len(
            active
        )

        if (
            selection[
                "requested_budget"
            ]
            is not None
        ):

            requested_budget_total += int(
                selection[
                    "requested_budget"
                ]
            )

            requested_budget_observations += 1

        if (
            selection[
                "uncertainty"
            ]
            is not None
        ):

            uncertainty_total += float(
                selection[
                    "uncertainty"
                ]
            )

            uncertainty_observations += 1

        if selection[
            "fully_tied"
        ]:

            fully_tied_decisions += 1

        peak_frontier = max(
            peak_frontier,
            len(active)
        )

    return {
        "final_prefixes":
            active,

        "active_hop_rows":
            int(active_hop_rows),

        "active_prefixes":
            int(active_prefixes),

        "edges_examined":
            int(edges_examined),

        "candidate_branches":
            int(candidate_branches),

        "decision_hops":
            int(decision_hops),

        "retained_after_decision":
            int(
                retained_after_decision
            ),

        "requested_budget_total":
            int(
                requested_budget_total
            ),

        "requested_budget_observations":
            int(
                requested_budget_observations
            ),

        "uncertainty_total":
            float(
                uncertainty_total
            ),

        "uncertainty_observations":
            int(
                uncertainty_observations
            ),

        "fully_tied_decisions":
            int(
                fully_tied_decisions
            ),

        "peak_frontier":
            int(peak_frontier),
    }


# ======================================================================
# 29. RUN ONE DATASET'S FEATURE ABLATIONS
# ======================================================================

def run_feature_ablation_dataset(
    dataset_name,
    planning_rows,
    question_rows,
    rog_reference,
    selector_params,
    variant_bundles
):

    assert len(
        planning_rows
    ) == len(
        question_rows
    )

    stats = {
        variant: {
            "active_hop_rows": 0,
            "active_prefixes": 0,
            "edges_examined": 0,
            "candidate_branches": 0,
            "reachable_plans": 0,
            "reachable_questions": 0,
            "decision_hops": 0,
            "retained_after_decision": 0,
            "requested_budget_total": 0,
            "requested_budget_observations": 0,
            "uncertainty_total": 0.0,
            "uncertainty_observations": 0,
            "fully_tied_decisions": 0,
            "peak_frontier": 0,
        }
        for variant in variant_bundles
    }

    start_time = time.time()

    for source_index in tqdm(
        range(len(planning_rows)),
        desc=(
            f"{dataset_name} feature ablations"
        )
    ):

        plan_rec = (
            planning_rows[
                source_index
            ]
        )

        question_rec = (
            question_rows[
                source_index
            ]
        )

        assert str(
            plan_rec["id"]
        ) == str(
            question_rec["id"]
        )

        question_id = str(
            plan_rec["id"]
        )

        question = str(
            question_rec[
                "question"
            ]
        )

        topic_entities = (
            as_entity_list(
                plan_rec[
                    "q_entity"
                ]
            )
        )

        gold_answers = (
            as_entity_list(
                plan_rec[
                    "a_entity"
                ]
            )
        )

        plans = (
            normalize_relation_plans(
                plan_rec[
                    "predicted_paths"
                ]
            )
        )

        adjacency = (
            build_exact_rog_adjacency(
                plan_rec["graph"]
            )
        )

        # Computational caches shared only within this question.
        expansion_cache = {}
        feature_cache = {}
        logit_cache = {}

        question_reachable = {
            variant: False
            for variant in variant_bundles
        }

        for plan_index, plan in enumerate(
            plans
        ):

            if len(plan) == 0:
                continue

            for variant_name, bundle in (
                variant_bundles.items()
            ):

                result = (
                    traverse_ablation_variant(
                        dataset_name=
                            dataset_name,

                        bundle=
                            bundle,

                        selector_params=
                            selector_params,

                        question_id=
                            question_id,

                        question=
                            question,

                        adjacency=
                            adjacency,

                        topic_entities=
                            topic_entities,

                        plan_index=
                            plan_index,

                        plan=
                            plan,

                        expansion_cache=
                            expansion_cache,

                        feature_cache=
                            feature_cache,

                        logit_cache=
                            logit_cache
                    )
                )

                s = stats[
                    variant_name
                ]

                for key in [
                    "active_hop_rows",
                    "active_prefixes",
                    "edges_examined",
                    "candidate_branches",
                    "decision_hops",
                    "retained_after_decision",
                    "requested_budget_total",
                    "requested_budget_observations",
                    "fully_tied_decisions",
                ]:

                    s[key] += result[key]

                s[
                    "uncertainty_total"
                ] += result[
                    "uncertainty_total"
                ]

                s[
                    "uncertainty_observations"
                ] += result[
                    "uncertainty_observations"
                ]

                s[
                    "peak_frontier"
                ] = max(
                    s["peak_frontier"],
                    result[
                        "peak_frontier"
                    ]
                )

                # ------------------------------------------------------
                # Gold used ONLY after traversal.
                # ------------------------------------------------------

                reachable = (
                    final_prefixes_reach_answer(
                        final_prefixes=
                            result[
                                "final_prefixes"
                            ],

                        gold_answers=
                            gold_answers
                    )
                )

                if reachable:

                    s[
                        "reachable_plans"
                    ] += 1

                    question_reachable[
                        variant_name
                    ] = True

        for variant_name, reachable in (
            question_reachable.items()
        ):

            if reachable:

                stats[
                    variant_name
                ][
                    "reachable_questions"
                ] += 1

    elapsed = (
        time.time() - start_time
    )

    print(
        f"\n{dataset_name.upper()} feature "
        f"ablation evaluation completed "
        f"in {elapsed/60:.2f} min"
    )

    rog_edges = float(
        rog_reference[
            "edges_examined"
        ]
    )

    rog_reachable = int(
        rog_reference[
            "reachable_questions"
        ]
    )

    rows = []

    for variant_name, s in (
        stats.items()
    ):

        edges = int(
            s["edges_examined"]
        )

        reachable_q = int(
            s[
                "reachable_questions"
            ]
        )

        avg_budget = (
            s[
                "requested_budget_total"
            ]
            /
            s[
                "requested_budget_observations"
            ]
            if
            s[
                "requested_budget_observations"
            ] > 0
            else
            np.nan
        )

        avg_uncertainty = (
            s[
                "uncertainty_total"
            ]
            /
            s[
                "uncertainty_observations"
            ]
            if
            s[
                "uncertainty_observations"
            ] > 0
            else
            np.nan
        )

        rows.append(
            {
                "dataset":
                    dataset_name,

                "variant":
                    variant_name,

                "input_dim":
                    variant_bundles[
                        variant_name
                    ][
                        "input_dim"
                    ],

                "edges_examined":
                    edges,

                "ssr":
                    float(
                        1.0
                        -
                        edges
                        /
                        rog_edges
                    ),

                "reachable_questions":
                    reachable_q,

                "rog_reachable_questions":
                    rog_reachable,

                "answer_retention":
                    float(
                        reachable_q
                        /
                        rog_reachable
                    ),

                "active_hop_rows":
                    int(
                        s[
                            "active_hop_rows"
                        ]
                    ),

                "active_prefixes":
                    int(
                        s[
                            "active_prefixes"
                        ]
                    ),

                "candidate_branches":
                    int(
                        s[
                            "candidate_branches"
                        ]
                    ),

                "reachable_plans":
                    int(
                        s[
                            "reachable_plans"
                        ]
                    ),

                "decision_hops":
                    int(
                        s[
                            "decision_hops"
                        ]
                    ),

                "avg_requested_budget":
                    float(avg_budget),

                "avg_uncertainty":
                    float(avg_uncertainty),

                "fully_tied_decisions":
                    int(
                        s[
                            "fully_tied_decisions"
                        ]
                    ),

                "peak_frontier":
                    int(
                        s[
                            "peak_frontier"
                        ]
                    ),
            }
        )

    return pd.DataFrame(
        rows
    )


# ======================================================================
# 30. RUN VALIDATION ABLATIONS
# ======================================================================

print(
    "\n"
    + "=" * 116
)

print(
    "RUNNING VALIDATION FEATURE-FAMILY ABLATIONS"
)

print(
    "=" * 116
)


webqsp_feature_ablation_df = (
    run_feature_ablation_dataset(
        dataset_name=
            "webqsp",

        planning_rows=
            webqsp_val_plan_rows,

        question_rows=
            webqsp_val_runtime,

        rog_reference=
            webqsp_rog_reference,

        selector_params=
            AFP_14B_SELECTED[
                "webqsp"
            ],

        variant_bundles=
            ABLATION_MODELS[
                "webqsp"
            ]
    )
)


cwq_feature_ablation_df = (
    run_feature_ablation_dataset(
        dataset_name=
            "cwq",

        planning_rows=
            cwq_val_plan_rows,

        question_rows=
            cwq_val_runtime,

        rog_reference=
            cwq_rog_reference,

        selector_params=
            AFP_14B_SELECTED[
                "cwq"
            ],

        variant_bundles=
            ABLATION_MODELS[
                "cwq"
            ]
    )
)


# ======================================================================
# 31. ORIGINAL CELL-14B AFP REFERENCE
# ======================================================================

def original_afp_reference_row(
    comparison_df
):

    rows = comparison_df[
        comparison_df[
            "method"
        ] == "AFP"
    ]

    assert len(rows) == 1

    return rows.iloc[0]


webqsp_original_afp = (
    original_afp_reference_row(
        webqsp_comparison_14b_df
    )
)

cwq_original_afp = (
    original_afp_reference_row(
        cwq_comparison_14b_df
    )
)


# ======================================================================
# 32. DELTAS RELATIVE TO MATCHED FULL RETRAIN CONTROL
# ======================================================================

def add_ablation_deltas(df):

    df = df.copy()

    full_rows = df[
        df["variant"]
        ==
        "full_v2_retrain"
    ]

    assert len(full_rows) == 1

    full = full_rows.iloc[0]

    df[
        "delta_ar_vs_full_retrain"
    ] = (
        df["answer_retention"]
        -
        float(
            full[
                "answer_retention"
            ]
        )
    )

    df[
        "delta_ssr_vs_full_retrain"
    ] = (
        df["ssr"]
        -
        float(
            full["ssr"]
        )
    )

    df[
        "delta_edges_vs_full_retrain"
    ] = (
        df["edges_examined"]
        -
        int(
            full[
                "edges_examined"
            ]
        )
    )

    return df


webqsp_feature_ablation_df = (
    add_ablation_deltas(
        webqsp_feature_ablation_df
    )
)

cwq_feature_ablation_df = (
    add_ablation_deltas(
        cwq_feature_ablation_df
    )
)


# ======================================================================
# 33. FULL-RETRAIN REPRODUCIBILITY CONTROL
# ======================================================================
#
# Original selected model was trained previously on CUDA.
# This matched ablation protocol uses CPU.
#
# Therefore we REPORT rather than assume exact equality.
#
# The causal feature comparison itself is against full_v2_retrain,
# which shares exactly the same retraining protocol as every ablation.
# ======================================================================

def print_full_retrain_control(
    dataset_name,
    ablation_df,
    original_afp
):

    full = (
        ablation_df[
            ablation_df[
                "variant"
            ]
            ==
            "full_v2_retrain"
        ]
        .iloc[0]
    )

    print(
        "\n"
        + "=" * 112
    )

    print(
        f"{dataset_name.upper()} "
        "FULL-RETRAIN REPRODUCIBILITY CONTROL"
    )

    print(
        "=" * 112
    )

    print(
        "\nOriginal selected AFP (Cell 14B)"
    )

    print(
        "  AR:",
        f"{float(original_afp['answer_retention']):.6f}"
    )

    print(
        "  SSR:",
        f"{float(original_afp['ssr']):.6f}"
    )

    print(
        "  edges:",
        int(
            original_afp[
                "edges_examined"
            ]
        )
    )

    print(
        "\nFull Feature-v2 matched retrain"
    )

    print(
        "  AR:",
        f"{float(full['answer_retention']):.6f}"
    )

    print(
        "  SSR:",
        f"{float(full['ssr']):.6f}"
    )

    print(
        "  edges:",
        int(
            full[
                "edges_examined"
            ]
        )
    )

    print(
        "\nDifference (retrain - original)"
    )

    print(
        "  AR delta:",
        f"{float(full['answer_retention'] - original_afp['answer_retention']):.6f}"
    )

    print(
        "  SSR delta:",
        f"{float(full['ssr'] - original_afp['ssr']):.6f}"
    )

    print(
        "  edge delta:",
        int(
            full[
                "edges_examined"
            ]
            -
            original_afp[
                "edges_examined"
            ]
        )
    )


print_full_retrain_control(
    "webqsp",
    webqsp_feature_ablation_df,
    webqsp_original_afp
)

print_full_retrain_control(
    "cwq",
    cwq_feature_ablation_df,
    cwq_original_afp
)


# ======================================================================
# 34. DISPLAY FEATURE ABLATION TABLES
# ======================================================================

ABLATION_DISPLAY_COLUMNS = [
    "variant",
    "input_dim",
    "answer_retention",
    "ssr",
    "edges_examined",
    "active_prefixes",
    "candidate_branches",
    "decision_hops",
    "avg_requested_budget",
    "avg_uncertainty",
    "fully_tied_decisions",
    "delta_ar_vs_full_retrain",
    "delta_ssr_vs_full_retrain",
    "delta_edges_vs_full_retrain",
]


def display_ablation_table(
    dataset_name,
    df
):

    print(
        "\n"
        + "=" * 124
    )

    print(
        f"{dataset_name.upper()} "
        "FEATURE-FAMILY RETRAINING ABLATION"
    )

    print(
        "=" * 124
    )

    ordered_names = [
        "full_v2_retrain",
        "minus_semantic",
        "minus_path",
        "minus_structural",
        "minus_progress",
    ]

    ordered = (
        df.set_index(
            "variant"
        )
        .loc[
            ordered_names
        ]
        .reset_index()
    )

    print(
        ordered[
            ABLATION_DISPLAY_COLUMNS
        ].to_string(
            index=False,
            float_format=lambda x:
                f"{x:.6f}"
        )
    )


display_ablation_table(
    "webqsp",
    webqsp_feature_ablation_df
)

display_ablation_table(
    "cwq",
    cwq_feature_ablation_df
)


# ======================================================================
# 35. CAUSAL FEATURE-FAMILY SUMMARY
# ======================================================================
#
# Interpret AR and SSR jointly.
#
# Greater SSR is NOT automatically beneficial if AR falls.
# ======================================================================

def causal_feature_summary(df):

    rows = []

    for _, row in df.iterrows():

        if (
            row["variant"]
            ==
            "full_v2_retrain"
        ):
            continue

        rows.append(
            {
                "variant":
                    row["variant"],

                "delta_AR":
                    float(
                        row[
                            "delta_ar_vs_full_retrain"
                        ]
                    ),

                "delta_SSR":
                    float(
                        row[
                            "delta_ssr_vs_full_retrain"
                        ]
                    ),

                "removal_harmed_AR":
                    bool(
                        row[
                            "delta_ar_vs_full_retrain"
                        ] < 0
                    ),

                "removal_improved_AR":
                    bool(
                        row[
                            "delta_ar_vs_full_retrain"
                        ] > 0
                    ),
            }
        )

    return pd.DataFrame(rows)


webqsp_feature_causal_df = (
    causal_feature_summary(
        webqsp_feature_ablation_df
    )
)

cwq_feature_causal_df = (
    causal_feature_summary(
        cwq_feature_ablation_df
    )
)


print(
    "\nWEBQSP causal feature deltas"
)

print(
    webqsp_feature_causal_df.to_string(
        index=False,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


print(
    "\nCWQ causal feature deltas"
)

print(
    cwq_feature_causal_df.to_string(
        index=False,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


# ======================================================================
# 36. EXPORT RESULT CSVs
# ======================================================================

WEBQSP_ABLATION_CSV = (
    ABLATION_DIR
    / "webqsp_feature_family_retraining_ablation.csv"
)

CWQ_ABLATION_CSV = (
    ABLATION_DIR
    / "cwq_feature_family_retraining_ablation.csv"
)

WEBQSP_CAUSAL_CSV = (
    ABLATION_DIR
    / "webqsp_feature_family_causal_deltas.csv"
)

CWQ_CAUSAL_CSV = (
    ABLATION_DIR
    / "cwq_feature_family_causal_deltas.csv"
)


webqsp_feature_ablation_df.to_csv(
    WEBQSP_ABLATION_CSV,
    index=False
)

cwq_feature_ablation_df.to_csv(
    CWQ_ABLATION_CSV,
    index=False
)

webqsp_feature_causal_df.to_csv(
    WEBQSP_CAUSAL_CSV,
    index=False
)

cwq_feature_causal_df.to_csv(
    CWQ_CAUSAL_CSV,
    index=False
)


# ======================================================================
# 37. CHECKPOINT MANIFEST HELPER
# ======================================================================

def model_checkpoint_manifest(
    dataset_name
):

    output = {}

    for variant_name, path in (
        ABLATION_CHECKPOINT_PATHS[
            dataset_name
        ].items()
    ):

        output[
            variant_name
        ] = {
            "path":
                str(path),

            "sha256":
                sha256_file(path),
        }

    return output


# ======================================================================
# 38. CELL 15A MANIFEST
# ======================================================================

CELL15A_MANIFEST = {
    "cell":
        "RQ2_CELL15A_FEATURE_FAMILY_RETRAINING_ABLATIONS",

    "version":
        "rq2_cell15a_feature_ablation_v2_dimension_aware_exact_classes",

    "methodology":
        "remove_feature_family_then_retrain",

    "dimension_handling":
        {
            "approach":
                "exact_persisted_class_source_reinstantiated_with_variant_AFP_INPUT_DIM",

            "scorer_algorithm_changed":
                False,

            "standardizer_algorithm_changed":
                False,
        },

    "variants": {
        variant: {
            "feature_columns":
                columns,

            "input_dim":
                len(columns),
        }
        for variant, columns in (
            ABLATION_VARIANTS.items()
        )
    },

    "feature_groups": {
        key: value
        for key, value in (
            FEATURE_GROUPS.items()
        )
    },

    "training_protocol": {
        "optimizer":
            AFP_ABLATION_OPTIMIZER_NAME,

        "loss":
            "branch_bce",

        "epochs":
            80,

        "learning_rate":
            1e-3,

        "weight_decay":
            1e-4,

        "seed":
            42,

        "dropout":
            0.0,

        "class_weighting":
            False,

        "early_stopping":
            False,

        "device":
            str(
                ABLATION_DEVICE
            ),

        "ablation_specific_tuning":
            False,
    },

    "hidden_dimensions": {
        "webqsp":
            32,

        "cwq":
            64,
    },

    "selector_parameters_fixed_from_Cell13B": {
        "webqsp":
            AFP_14B_SELECTED[
                "webqsp"
            ],

        "cwq":
            AFP_14B_SELECTED[
                "cwq"
            ],
    },

    "train_feature_npz": {
        "webqsp": {
            "path":
                str(
                    WEBQSP_TRAIN_NPZ
                ),

            "sha256":
                sha256_file(
                    WEBQSP_TRAIN_NPZ
                ),
        },

        "cwq": {
            "path":
                str(
                    CWQ_TRAIN_NPZ
                ),

            "sha256":
                sha256_file(
                    CWQ_TRAIN_NPZ
                ),
        },
    },

    "ablation_checkpoints": {
        "webqsp":
            model_checkpoint_manifest(
                "webqsp"
            ),

        "cwq":
            model_checkpoint_manifest(
                "cwq"
            ),
    },

    "original_selected_checkpoint_sha256": {
        "webqsp":
            webqsp_ckpt_sha,

        "cwq":
            cwq_ckpt_sha,
    },

    "full_27d_standardizer_fidelity":
        True,

    "gold_used_by_scorer":
        False,

    "gold_used_by_selector":
        False,

    "gold_used_for":
        "post_traversal_validation_reachability_only",

    "test_examples_accessed":
        False,

    "final_AFP_frozen":
        False,

    "next":
        "Cell15B_selector_component_ablations_and_final_development_freeze",
}


CELL15A_MANIFEST_PATH = (
    ABLATION_DIR
    / "cell15a_feature_family_retraining_ablation_manifest.json"
)


with open(
    CELL15A_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        CELL15A_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False
    )


CELL15A_COMPLETE = True


# ======================================================================
# 39. FINAL REPORT
# ======================================================================

print(
    "\n"
    + "=" * 124
)

print(
    "=== RQ2 CELL 15A: "
    "FEATURE-FAMILY RETRAINING ABLATIONS COMPLETE ==="
)

print(
    "=" * 124
)


print(
    "\nAblation methodology:"
)

print(
    "  Full Feature-v2 retraining control"
)

print(
    "  -Semantic   -> retrained"
)

print(
    "  -Path       -> retrained"
)

print(
    "  -Structural -> retrained"
)

print(
    "  -Progress   -> retrained"
)


print(
    "\nDimension handling:"
)

print(
    "  Exact persisted AFPScorer source:          YES"
)

print(
    "  Exact persisted standardizer source:       YES"
)

print(
    "  Variant-specific AFP_INPUT_DIM:             YES"
)

print(
    "  Scorer algorithm modified:                  NO"
)

print(
    "  Standardizer algorithm modified:            NO"
)


print(
    "\nTraining controls:"
)

print(
    "  Optimizer:",
    AFP_ABLATION_OPTIMIZER_NAME
)

print(
    "  Loss: branch_bce"
)

print(
    "  Epochs: 80"
)

print(
    "  LR: 1e-3"
)

print(
    "  Weight decay: 1e-4"
)

print(
    "  Seed: 42"
)

print(
    "  Ablation-specific tuning: NO"
)


print(
    "\nSelector remains fixed:"
)

print(
    "  WebQSP:",
    AFP_14B_SELECTED[
        "webqsp"
    ]
)

print(
    "  CWQ:   ",
    AFP_14B_SELECTED[
        "cwq"
    ]
)


print(
    "\nScientific status:"
)

print(
    "  Proper feature-removal retraining: COMPLETE"
)

print(
    "  Gold in scorer/selector:           NO"
)

print(
    "  TEST examples accessed:            NO"
)

print(
    "  Final AFP frozen:                  NO"
)


print(
    "\nOutputs:"
)

print(
    " ",
    WEBQSP_ABLATION_CSV
)

print(
    " ",
    CWQ_ABLATION_CSV
)

print(
    " ",
    WEBQSP_CAUSAL_CSV
)

print(
    " ",
    CWQ_CAUSAL_CSV
)

print(
    " ",
    CELL15A_MANIFEST_PATH
)


print(
    "\nNEXT STEP:"
)

print(
    "Cell 15B — selector/component ablations "
    "+ final development decision/freeze."
)

Cell 15A prerequisites: PASSED
Feature version: afp_features_v2_masked_entity_semantics
Feature dimension: 27
Feature root: /kaggle/working/step3_rq2_dev_v1/03_features
Ablation output: /kaggle/working/step3_rq2_dev_v1/13_feature_ablations

WEBQSP matching train NPZ candidate(s):
  /kaggle/working/step3_rq2_dev_v1/03_features/webqsp/webqsp_train_afp_features_v2.npz
    SHA: 55db1698520e6c6be3f330a11c4c38334adc02923de650a6a76ce0cc757800b0

CWQ matching train NPZ candidate(s):
  /kaggle/working/step3_rq2_dev_v1/03_features/cwq/cwq_train_afp_features_v2.npz
    SHA: 1c5a273ca6aa46ce2ad153a0921d824ce3e62eff817c935f468c0d93d8602508

Frozen train feature files identified: PASSED

Training matrices:
  WebQSP: (18437, 27) | groups: 1457
  CWQ:    (218544, 27) | groups: 15937
Frozen training-cache gate: PASSED

Feature ablation variants:
  full_v2_retrain      27 features
  minus_semantic       19 features
  minus_path           20 features
  minus_structural     19 features
  minus_progress   

webqsp feature ablations:   0%|          | 0/246 [00:00<?, ?it/s]


WEBQSP feature ablation evaluation completed in 0.04 min


cwq feature ablations:   0%|          | 0/3519 [00:00<?, ?it/s]


CWQ feature ablation evaluation completed in 2.50 min

WEBQSP FULL-RETRAIN REPRODUCIBILITY CONTROL

Original selected AFP (Cell 14B)
  AR: 1.000000
  SSR: 0.010272
  edges: 338018

Full Feature-v2 matched retrain
  AR: 1.000000
  SSR: 0.010304
  edges: 338007

Difference (retrain - original)
  AR delta: 0.000000
  SSR delta: 0.000032
  edge delta: -11

CWQ FULL-RETRAIN REPRODUCIBILITY CONTROL

Original selected AFP (Cell 14B)
  AR: 0.990103
  SSR: 0.062031
  edges: 4931160

Full Feature-v2 matched retrain
  AR: 0.990103
  SSR: 0.062182
  edges: 4930362

Difference (retrain - original)
  AR delta: 0.000000
  SSR delta: 0.000152
  edge delta: -798

WEBQSP FEATURE-FAMILY RETRAINING ABLATION
         variant  input_dim  answer_retention      ssr  edges_examined  active_prefixes  candidate_branches  decision_hops  avg_requested_budget  avg_uncertainty  fully_tied_decisions  delta_ar_vs_full_retrain  delta_ssr_vs_full_retrain  delta_edges_vs_full_retrain
 full_v2_retrain         27         

In [33]:
# ======================================================================
# RQ2 CELL 15B
# SELECTOR / COMPONENT ABLATIONS + FINAL AFP DEVELOPMENT FREEZE
# ======================================================================
#
# RUN AS A NEW CELL.
#
# DO NOT replace Cell 15A.
# DO NOT restart the kernel.
# DO NOT rerun Cells 13 / 13B / 14 / 14B / 15A.
#
# ======================================================================
# PURPOSE
# ======================================================================
#
# Complete the remaining RQ2 development-stage causal analysis and
# freeze the final AFP configuration BEFORE any TEST evaluation.
#
# SELECTOR / COMPONENT ABLATIONS
# --------------------------------
#
# A. Full AFP
#    selected scorer + selected T + selected gamma_min
#
# B. No uncertainty adaptation
#    same scorer
#    same temperature T
#    same gamma_min
#    BUT:
#
#       gamma_h = gamma_min
#
#    instead of:
#
#       gamma_h = gamma_min + u_h (1 - gamma_min)
#
#    Fully tied groups STILL retain all.
#    Tie-expansion behavior unchanged.
#
# C. No temperature scaling
#    same scorer
#    same gamma_min
#    same uncertainty-adaptive rule
#    BUT:
#
#       T = 1.0
#
#    This isolates the contribution of validation-selected temperature
#    calibration.
#
#
# EXISTING CAUSAL CONTROLS FROM CELL 14B
# --------------------------------------
#
# D. Adaptive-Budget Random
#    same AFP retained-count mechanism, random branch identity
#    -> ranking-value control
#
# E. Fixed Top-B
#    fixed-budget controlled baseline
#    -> adaptive-budget comparison
#
#
# IMPORTANT
# ---------
# - NO further hyperparameter search
# - NO feature redesign
# - NO retraining for selector ablations
# - NO TEST access
# - Ablations are diagnostic only
# - Final method remains:
#
#       full Feature-v2
#       original selected scorer checkpoint
#       selected expanded AFP selector parameters
#
# After this cell:
#
#       AFP_DEVELOPMENT_FROZEN = True
#
# and NO development changes are allowed before TEST.
# ======================================================================


# ======================================================================
# 0. IMPORTS
# ======================================================================

import hashlib
import json
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd

from tqdm.auto import tqdm


# ======================================================================
# 1. HARD PREREQUISITES
# ======================================================================

required = [
    # --------------------------------------------------------------
    # Development stage
    # --------------------------------------------------------------
    "CELL15A_COMPLETE",
    "CELL14B_COMPLETE",
    "CELL13B_COMPLETE",

    # --------------------------------------------------------------
    # Final selected AFP parameters
    # --------------------------------------------------------------
    "AFP_14B_SELECTED",

    # --------------------------------------------------------------
    # Selected original scorer runtime
    # --------------------------------------------------------------
    "AFP_RUNTIME_SCORE_GROUP",
    "AFP_RUNTIME_FEATURE_VERSION",

    # --------------------------------------------------------------
    # Frozen validation data
    # --------------------------------------------------------------
    "webqsp_val_plan_rows",
    "cwq_val_plan_rows",

    "webqsp_val_runtime",
    "cwq_val_runtime",

    "webqsp_rog_reference",
    "cwq_rog_reference",

    # --------------------------------------------------------------
    # Exact traversal utilities
    # --------------------------------------------------------------
    "build_exact_rog_adjacency",
    "as_entity_list",
    "normalize_relation_plans",

    "get_cached_relation_expansion",
    "get_group_logits",
    "frontier_cache_key",

    "final_prefixes_reach_answer",

    # --------------------------------------------------------------
    # Cell 14B controlled evidence
    # --------------------------------------------------------------
    "webqsp_comparison_14b_df",
    "cwq_comparison_14b_df",

    "webqsp_summary_14b_df",
    "cwq_summary_14b_df",

    # --------------------------------------------------------------
    # Cell 15A causal feature evidence
    # --------------------------------------------------------------
    "webqsp_feature_ablation_df",
    "cwq_feature_ablation_df",

    # --------------------------------------------------------------
    # Original selected checkpoints
    # --------------------------------------------------------------
    "webqsp_ckpt_obj",
    "cwq_ckpt_obj",

    "webqsp_ckpt_sha",
    "cwq_ckpt_sha",
]

missing = [
    name
    for name in required
    if name not in globals()
]

assert not missing, (
    "Missing required prior-cell objects:\n  "
    + "\n  ".join(missing)
)

assert CELL15A_COMPLETE is True
assert CELL14B_COMPLETE is True
assert CELL13B_COMPLETE is True


print(
    "Cell 15B prerequisites: PASSED"
)

print(
    "Feature version:",
    AFP_RUNTIME_FEATURE_VERSION
)


# ======================================================================
# 2. OUTPUT DIRECTORY
# ======================================================================

ROOT = Path(
    "/kaggle/working/step3_rq2_dev_v1"
)

CELL15B_DIR = (
    ROOT
    / "14_selector_ablations_and_final_freeze"
)

CELL15B_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print(
    "Cell 15B output:",
    CELL15B_DIR
)


# ======================================================================
# 3. FINAL DEVELOPMENT-SELECTED AFP PARAMETERS
# ======================================================================

FINAL_SELECTOR_PARAMS = {
    "webqsp": {
        "T":
            float(
                AFP_14B_SELECTED[
                    "webqsp"
                ][
                    "T"
                ]
            ),

        "gamma_min":
            float(
                AFP_14B_SELECTED[
                    "webqsp"
                ][
                    "gamma_min"
                ]
            ),
    },

    "cwq": {
        "T":
            float(
                AFP_14B_SELECTED[
                    "cwq"
                ][
                    "T"
                ]
            ),

        "gamma_min":
            float(
                AFP_14B_SELECTED[
                    "cwq"
                ][
                    "gamma_min"
                ]
            ),
    },
}


assert FINAL_SELECTOR_PARAMS[
    "webqsp"
] == {
    "T": 0.05,
    "gamma_min": 0.1,
}


assert FINAL_SELECTOR_PARAMS[
    "cwq"
] == {
    "T": 0.1,
    "gamma_min": 0.1,
}


print(
    "\nFinal development-selected selector:"
)

print(
    "  WebQSP:",
    FINAL_SELECTOR_PARAMS[
        "webqsp"
    ]
)

print(
    "  CWQ:   ",
    FINAL_SELECTOR_PARAMS[
        "cwq"
    ]
)


# ======================================================================
# 4. PREDECLARE SELECTOR ABLATIONS
# ======================================================================
#
# These are NOT candidate configurations for selection.
#
# They are fixed causal diagnostics.
# ======================================================================

SELECTOR_ABLATIONS = [
    "full_afp",
    "no_uncertainty_adaptation",
    "no_temperature_scaling",
]


print(
    "\nSelector/component ablations:"
)

for name in SELECTOR_ABLATIONS:

    print(
        " ",
        name
    )


print(
    "\nNo ablation-specific tuning: YES"
)


# ======================================================================
# 5. NUMERICAL SELECTOR HELPERS
# ======================================================================

TIE_ATOL_15B = 1e-8


def stable_softmax_15b(
    logits,
    temperature
):

    logits = np.asarray(
        logits,
        dtype=np.float64
    )

    temperature = float(
        temperature
    )

    assert temperature > 0.0


    values = (
        logits
        /
        temperature
    )


    values = (
        values
        -
        np.max(
            values
        )
    )


    exp_values = np.exp(
        values
    )


    denominator = float(
        np.sum(
            exp_values
        )
    )


    assert denominator > 0
    assert np.isfinite(
        denominator
    )


    return (
        exp_values
        /
        denominator
    )


def normalized_entropy_15b(
    probabilities
):

    p = np.asarray(
        probabilities,
        dtype=np.float64
    )


    n = len(
        p
    )


    if n <= 1:

        return 0.0


    safe = np.clip(
        p,
        1e-12,
        1.0
    )


    entropy = -float(
        np.sum(
            safe
            *
            np.log(
                safe
            )
        )
    )


    denominator = math.log(
        n
    )


    if denominator <= 0:

        return 0.0


    return float(
        np.clip(
            entropy
            /
            denominator,
            0.0,
            1.0
        )
    )


def all_logits_tied_15b(
    logits
):

    logits = np.asarray(
        logits,
        dtype=np.float64
    )


    if len(
        logits
    ) <= 1:

        return True


    return bool(
        (
            np.max(
                logits
            )
            -
            np.min(
                logits
            )
        )
        <=
        TIE_ATOL_15B
    )


def stable_descending_order_15b(
    logits
):

    return np.argsort(
        -np.asarray(
            logits,
            dtype=np.float64
        ),
        kind="stable"
    )


def tie_expanded_top_k_15b(
    logits,
    requested_k
):

    logits = np.asarray(
        logits,
        dtype=np.float64
    )


    n = len(
        logits
    )


    requested_k = int(
        requested_k
    )


    assert requested_k >= 1


    if requested_k >= n:

        return list(
            range(
                n
            )
        )


    order = (
        stable_descending_order_15b(
            logits
        )
    )


    cutoff_index = order[
        requested_k
        - 1
    ]


    cutoff_score = logits[
        cutoff_index
    ]


    selected = []


    for index, score in enumerate(
        logits
    ):

        if (
            score
            >
            cutoff_score
            or
            np.isclose(
                score,
                cutoff_score,
                atol=TIE_ATOL_15B,
                rtol=0.0
            )
        ):

            selected.append(
                index
            )


    return selected


# ======================================================================
# 6. SELECTOR COMPONENT ABLATION
# ======================================================================

def select_component_variant(
    logits,
    selected_T,
    gamma_min,
    variant
):

    logits = np.asarray(
        logits,
        dtype=np.float64
    )


    n = len(
        logits
    )


    assert n > 1


    selected_T = float(
        selected_T
    )

    gamma_min = float(
        gamma_min
    )


    assert variant in (
        SELECTOR_ABLATIONS
    )


    # --------------------------------------------------------------
    # Preserve the full-method abstention behavior under complete tie.
    #
    # This is NOT removed because we want to isolate uncertainty
    # adaptation and temperature calibration separately.
    # --------------------------------------------------------------

    if all_logits_tied_15b(
        logits
    ):

        return {
            "selected_indices":
                list(
                    range(
                        n
                    )
                ),

            "requested_budget":
                n,

            "retained_count":
                n,

            "uncertainty":
                1.0,

            "gamma":
                1.0,

            "temperature_used":
                (
                    1.0
                    if
                    variant
                    ==
                    "no_temperature_scaling"
                    else
                    selected_T
                ),

            "fully_tied":
                True,
        }


    # --------------------------------------------------------------
    # Temperature component
    # --------------------------------------------------------------

    if (
        variant
        ==
        "no_temperature_scaling"
    ):

        temperature = 1.0

    else:

        temperature = (
            selected_T
        )


    probabilities = (
        stable_softmax_15b(
            logits=
                logits,

            temperature=
                temperature
        )
    )


    uncertainty = (
        normalized_entropy_15b(
            probabilities
        )
    )


    # --------------------------------------------------------------
    # Uncertainty-adaptation component
    # --------------------------------------------------------------

    if (
        variant
        ==
        "no_uncertainty_adaptation"
    ):

        gamma = (
            gamma_min
        )

    else:

        gamma = (
            gamma_min
            +
            uncertainty
            *
            (
                1.0
                -
                gamma_min
            )
        )


    gamma = float(
        np.clip(
            gamma,
            0.0,
            1.0
        )
    )


    order = (
        stable_descending_order_15b(
            logits
        )
    )


    sorted_probabilities = (
        probabilities[
            order
        ]
    )


    cumulative = np.cumsum(
        sorted_probabilities
    )


    requested_budget = int(
        np.searchsorted(
            cumulative,
            gamma,
            side="left"
        )
        + 1
    )


    requested_budget = min(
        max(
            requested_budget,
            1
        ),
        n
    )


    selected_indices = (
        tie_expanded_top_k_15b(
            logits=
                logits,

            requested_k=
                requested_budget
        )
    )


    return {
        "selected_indices":
            selected_indices,

        "requested_budget":
            requested_budget,

        "retained_count":
            len(
                selected_indices
            ),

        "uncertainty":
            float(
                uncertainty
            ),

        "gamma":
            float(
                gamma
            ),

        "temperature_used":
            float(
                temperature
            ),

        "fully_tied":
            False,
    }


# ======================================================================
# 7. COMPONENT TRAVERSAL
# ======================================================================

def traverse_selector_ablation(
    dataset_name,
    variant,
    selector_params,
    question_id,
    question,
    adjacency,
    topic_entities,
    plan_index,
    plan,
    expansion_cache,
    score_cache
):

    active = [
        (
            str(
                entity
            ),
        )
        for entity in topic_entities
    ]


    L = len(
        plan
    )


    active_hop_rows = 0
    active_prefixes = 0

    edges_examined = 0
    candidate_branches = 0

    decision_hops = 0

    retained_after_decision = 0

    requested_budget_total = 0
    requested_budget_observations = 0

    uncertainty_total = 0.0
    uncertainty_observations = 0

    gamma_total = 0.0
    gamma_observations = 0

    fully_tied_decisions = 0

    peak_frontier = len(
        active
    )


    for hop, target_relation in enumerate(
        plan
    ):

        if not active:

            break


        active_hop_rows += 1

        active_prefixes += len(
            active
        )


        expansion = (
            get_cached_relation_expansion(
                adjacency=
                    adjacency,

                plan_index=
                    plan_index,

                hop=
                    hop,

                target_relation=
                    target_relation,

                active_prefixes=
                    active,

                expansion_cache=
                    expansion_cache
            )
        )


        candidates = expansion[
            "candidates"
        ]


        candidate_rows = expansion[
            "candidate_rows"
        ]


        edges_examined += int(
            expansion[
                "edges_cost"
            ]
        )


        candidate_branches += len(
            candidates
        )


        if not candidates:

            active = []

            break


        # --------------------------------------------------------------
        # FINAL-HOP PROTECTION
        # --------------------------------------------------------------

        if hop == (
            L
            -
            1
        ):

            active = candidates


            peak_frontier = max(
                peak_frontier,
                len(
                    active
                )
            )


            continue


        # --------------------------------------------------------------
        # SINGLETON BYPASS
        # --------------------------------------------------------------

        if len(
            candidates
        ) <= 1:

            active = candidates


            peak_frontier = max(
                peak_frontier,
                len(
                    active
                )
            )


            continue


        decision_hops += 1


        logits = (
            get_group_logits(
                dataset_name=
                    dataset_name,

                question_id=
                    question_id,

                question=
                    question,

                plan_index=
                    plan_index,

                plan=
                    plan,

                hop=
                    hop,

                active_prefixes=
                    active,

                candidate_rows=
                    candidate_rows,

                score_cache=
                    score_cache
            )
        )


        selection = (
            select_component_variant(
                logits=
                    logits,

                selected_T=
                    selector_params[
                        "T"
                    ],

                gamma_min=
                    selector_params[
                        "gamma_min"
                    ],

                variant=
                    variant
            )
        )


        selected_indices = (
            selection[
                "selected_indices"
            ]
        )


        active = [
            candidates[
                index
            ]
            for index
            in selected_indices
        ]


        retained_after_decision += len(
            active
        )


        requested_budget_total += int(
            selection[
                "requested_budget"
            ]
        )


        requested_budget_observations += 1


        uncertainty_total += float(
            selection[
                "uncertainty"
            ]
        )


        uncertainty_observations += 1


        gamma_total += float(
            selection[
                "gamma"
            ]
        )


        gamma_observations += 1


        if selection[
            "fully_tied"
        ]:

            fully_tied_decisions += 1


        peak_frontier = max(
            peak_frontier,
            len(
                active
            )
        )


    return {
        "final_prefixes":
            active,

        "active_hop_rows":
            int(
                active_hop_rows
            ),

        "active_prefixes":
            int(
                active_prefixes
            ),

        "edges_examined":
            int(
                edges_examined
            ),

        "candidate_branches":
            int(
                candidate_branches
            ),

        "decision_hops":
            int(
                decision_hops
            ),

        "retained_after_decision":
            int(
                retained_after_decision
            ),

        "requested_budget_total":
            int(
                requested_budget_total
            ),

        "requested_budget_observations":
            int(
                requested_budget_observations
            ),

        "uncertainty_total":
            float(
                uncertainty_total
            ),

        "uncertainty_observations":
            int(
                uncertainty_observations
            ),

        "gamma_total":
            float(
                gamma_total
            ),

        "gamma_observations":
            int(
                gamma_observations
            ),

        "fully_tied_decisions":
            int(
                fully_tied_decisions
            ),

        "peak_frontier":
            int(
                peak_frontier
            ),
    }


# ======================================================================
# 8. RUN ONE DATASET
# ======================================================================

def run_selector_ablation_dataset(
    dataset_name,
    planning_rows,
    question_rows,
    rog_reference,
    selector_params
):

    assert len(
        planning_rows
    ) == len(
        question_rows
    )


    stats = {
        variant: {
            "active_hop_rows": 0,
            "active_prefixes": 0,
            "edges_examined": 0,
            "candidate_branches": 0,
            "reachable_plans": 0,
            "reachable_questions": 0,
            "decision_hops": 0,
            "retained_after_decision": 0,
            "requested_budget_total": 0,
            "requested_budget_observations": 0,
            "uncertainty_total": 0.0,
            "uncertainty_observations": 0,
            "gamma_total": 0.0,
            "gamma_observations": 0,
            "fully_tied_decisions": 0,
            "peak_frontier": 0,
        }

        for variant
        in SELECTOR_ABLATIONS
    }


    start_time = time.time()


    for source_index in tqdm(
        range(
            len(
                planning_rows
            )
        ),
        desc=(
            f"{dataset_name} selector ablations"
        )
    ):

        plan_rec = (
            planning_rows[
                source_index
            ]
        )


        question_rec = (
            question_rows[
                source_index
            ]
        )


        assert str(
            plan_rec[
                "id"
            ]
        ) == str(
            question_rec[
                "id"
            ]
        )


        question_id = str(
            plan_rec[
                "id"
            ]
        )


        question = str(
            question_rec[
                "question"
            ]
        )


        topic_entities = (
            as_entity_list(
                plan_rec[
                    "q_entity"
                ]
            )
        )


        gold_answers = (
            as_entity_list(
                plan_rec[
                    "a_entity"
                ]
            )
        )


        plans = (
            normalize_relation_plans(
                plan_rec[
                    "predicted_paths"
                ]
            )
        )


        adjacency = (
            build_exact_rog_adjacency(
                plan_rec[
                    "graph"
                ]
            )
        )


        expansion_cache = {}

        score_cache = {}


        question_reachable = {
            variant:
                False

            for variant
            in SELECTOR_ABLATIONS
        }


        for plan_index, plan in enumerate(
            plans
        ):

            if len(
                plan
            ) == 0:

                continue


            for variant in (
                SELECTOR_ABLATIONS
            ):

                result = (
                    traverse_selector_ablation(
                        dataset_name=
                            dataset_name,

                        variant=
                            variant,

                        selector_params=
                            selector_params,

                        question_id=
                            question_id,

                        question=
                            question,

                        adjacency=
                            adjacency,

                        topic_entities=
                            topic_entities,

                        plan_index=
                            plan_index,

                        plan=
                            plan,

                        expansion_cache=
                            expansion_cache,

                        score_cache=
                            score_cache
                    )
                )


                s = stats[
                    variant
                ]


                for key in [
                    "active_hop_rows",
                    "active_prefixes",
                    "edges_examined",
                    "candidate_branches",
                    "decision_hops",
                    "retained_after_decision",
                    "requested_budget_total",
                    "requested_budget_observations",
                    "fully_tied_decisions",
                ]:

                    s[
                        key
                    ] += result[
                        key
                    ]


                s[
                    "uncertainty_total"
                ] += result[
                    "uncertainty_total"
                ]


                s[
                    "uncertainty_observations"
                ] += result[
                    "uncertainty_observations"
                ]


                s[
                    "gamma_total"
                ] += result[
                    "gamma_total"
                ]


                s[
                    "gamma_observations"
                ] += result[
                    "gamma_observations"
                ]


                s[
                    "peak_frontier"
                ] = max(
                    s[
                        "peak_frontier"
                    ],
                    result[
                        "peak_frontier"
                    ]
                )


                # ------------------------------------------------------
                # Gold is used ONLY after traversal.
                # ------------------------------------------------------

                reachable = (
                    final_prefixes_reach_answer(
                        final_prefixes=
                            result[
                                "final_prefixes"
                            ],

                        gold_answers=
                            gold_answers
                    )
                )


                if reachable:

                    s[
                        "reachable_plans"
                    ] += 1


                    question_reachable[
                        variant
                    ] = True


        for variant, reachable in (
            question_reachable.items()
        ):

            if reachable:

                stats[
                    variant
                ][
                    "reachable_questions"
                ] += 1


    elapsed = (
        time.time()
        -
        start_time
    )


    print(
        f"\n{dataset_name.upper()} selector "
        f"ablations completed in "
        f"{elapsed/60:.2f} min"
    )


    rog_edges = float(
        rog_reference[
            "edges_examined"
        ]
    )


    rog_reachable = int(
        rog_reference[
            "reachable_questions"
        ]
    )


    rows = []


    for variant, s in (
        stats.items()
    ):

        edges = int(
            s[
                "edges_examined"
            ]
        )


        reachable_q = int(
            s[
                "reachable_questions"
            ]
        )


        avg_budget = (
            s[
                "requested_budget_total"
            ]
            /
            s[
                "requested_budget_observations"
            ]

            if
            s[
                "requested_budget_observations"
            ]
            > 0

            else
            np.nan
        )


        avg_uncertainty = (
            s[
                "uncertainty_total"
            ]
            /
            s[
                "uncertainty_observations"
            ]

            if
            s[
                "uncertainty_observations"
            ]
            > 0

            else
            np.nan
        )


        avg_gamma = (
            s[
                "gamma_total"
            ]
            /
            s[
                "gamma_observations"
            ]

            if
            s[
                "gamma_observations"
            ]
            > 0

            else
            np.nan
        )


        rows.append(
            {
                "dataset":
                    dataset_name,

                "variant":
                    variant,

                "selected_T":
                    float(
                        selector_params[
                            "T"
                        ]
                    ),

                "selected_gamma_min":
                    float(
                        selector_params[
                            "gamma_min"
                        ]
                    ),

                "edges_examined":
                    edges,

                "ssr":
                    float(
                        1.0
                        -
                        edges
                        /
                        rog_edges
                    ),

                "reachable_questions":
                    reachable_q,

                "rog_reachable_questions":
                    rog_reachable,

                "answer_retention":
                    float(
                        reachable_q
                        /
                        rog_reachable
                    ),

                "active_hop_rows":
                    int(
                        s[
                            "active_hop_rows"
                        ]
                    ),

                "active_prefixes":
                    int(
                        s[
                            "active_prefixes"
                        ]
                    ),

                "candidate_branches":
                    int(
                        s[
                            "candidate_branches"
                        ]
                    ),

                "reachable_plans":
                    int(
                        s[
                            "reachable_plans"
                        ]
                    ),

                "decision_hops":
                    int(
                        s[
                            "decision_hops"
                        ]
                    ),

                "avg_requested_budget":
                    float(
                        avg_budget
                    ),

                "avg_uncertainty":
                    float(
                        avg_uncertainty
                    ),

                "avg_gamma":
                    float(
                        avg_gamma
                    ),

                "fully_tied_decisions":
                    int(
                        s[
                            "fully_tied_decisions"
                        ]
                    ),

                "peak_frontier":
                    int(
                        s[
                            "peak_frontier"
                        ]
                    ),
            }
        )


    return pd.DataFrame(
        rows
    )


# ======================================================================
# 9. RUN WEBQSP + CWQ
# ======================================================================

print(
    "\n"
    + "=" * 118
)

print(
    "RUNNING SELECTOR / COMPONENT ABLATIONS"
)

print(
    "=" * 118
)


webqsp_selector_ablation_df = (
    run_selector_ablation_dataset(
        dataset_name=
            "webqsp",

        planning_rows=
            webqsp_val_plan_rows,

        question_rows=
            webqsp_val_runtime,

        rog_reference=
            webqsp_rog_reference,

        selector_params=
            FINAL_SELECTOR_PARAMS[
                "webqsp"
            ]
    )
)


cwq_selector_ablation_df = (
    run_selector_ablation_dataset(
        dataset_name=
            "cwq",

        planning_rows=
            cwq_val_plan_rows,

        question_rows=
            cwq_val_runtime,

        rog_reference=
            cwq_rog_reference,

        selector_params=
            FINAL_SELECTOR_PARAMS[
                "cwq"
            ]
    )
)


# ======================================================================
# 10. FULL AFP SOFTWARE FIDELITY TO CELL 14B
# ======================================================================

def get_single_row(
    df,
    column,
    value
):

    rows = df[
        df[
            column
        ]
        ==
        value
    ]


    assert len(
        rows
    ) == 1


    return rows.iloc[
        0
    ]


def full_afp_fidelity_gate(
    dataset_name,
    selector_df,
    comparison_df
):

    component = (
        get_single_row(
            selector_df,
            "variant",
            "full_afp"
        )
    )


    reference = (
        get_single_row(
            comparison_df,
            "method",
            "AFP"
        )
    )


    for key in [
        "edges_examined",
        "reachable_questions",
        "active_prefixes",
        "candidate_branches",
        "decision_hops",
    ]:

        assert int(
            component[
                key
            ]
        ) == int(
            reference[
                key
            ]
        ), (
            f"{dataset_name}: full AFP mismatch "
            f"for {key}: "
            f"{component[key]} != "
            f"{reference[key]}"
        )


    assert np.isclose(
        float(
            component[
                "ssr"
            ]
        ),
        float(
            reference[
                "ssr"
            ]
        ),
        atol=1e-12
    )


    assert np.isclose(
        float(
            component[
                "answer_retention"
            ]
        ),
        float(
            reference[
                "answer_retention"
            ]
        ),
        atol=1e-12
    )


    print(
        f"{dataset_name} full AFP "
        "Cell-14B fidelity: PASSED"
    )


full_afp_fidelity_gate(
    "WebQSP",
    webqsp_selector_ablation_df,
    webqsp_comparison_14b_df
)


full_afp_fidelity_gate(
    "CWQ",
    cwq_selector_ablation_df,
    cwq_comparison_14b_df
)


# ======================================================================
# 11. ADD DELTAS RELATIVE TO FULL AFP
# ======================================================================

def add_selector_deltas(
    df
):

    df = df.copy()


    full = (
        get_single_row(
            df,
            "variant",
            "full_afp"
        )
    )


    df[
        "delta_ar_vs_full"
    ] = (
        df[
            "answer_retention"
        ]
        -
        float(
            full[
                "answer_retention"
            ]
        )
    )


    df[
        "delta_ssr_vs_full"
    ] = (
        df[
            "ssr"
        ]
        -
        float(
            full[
                "ssr"
            ]
        )
    )


    df[
        "delta_edges_vs_full"
    ] = (
        df[
            "edges_examined"
        ]
        -
        int(
            full[
                "edges_examined"
            ]
        )
    )


    return df


webqsp_selector_ablation_df = (
    add_selector_deltas(
        webqsp_selector_ablation_df
    )
)


cwq_selector_ablation_df = (
    add_selector_deltas(
        cwq_selector_ablation_df
    )
)


# ======================================================================
# 12. DISPLAY SELECTOR ABLATIONS
# ======================================================================

SELECTOR_DISPLAY_COLUMNS = [
    "variant",
    "answer_retention",
    "ssr",
    "edges_examined",
    "active_prefixes",
    "candidate_branches",
    "decision_hops",
    "avg_requested_budget",
    "avg_uncertainty",
    "avg_gamma",
    "fully_tied_decisions",
    "delta_ar_vs_full",
    "delta_ssr_vs_full",
    "delta_edges_vs_full",
]


def display_selector_ablation(
    dataset_name,
    df
):

    print(
        "\n"
        + "=" * 124
    )

    print(
        f"{dataset_name.upper()} "
        "SELECTOR / COMPONENT ABLATIONS"
    )

    print(
        "=" * 124
    )


    ordered = (
        df.set_index(
            "variant"
        )
        .loc[
            SELECTOR_ABLATIONS
        ]
        .reset_index()
    )


    print(
        ordered[
            SELECTOR_DISPLAY_COLUMNS
        ].to_string(
            index=False,

            float_format=lambda x:
                f"{x:.6f}"
        )
    )


display_selector_ablation(
    "webqsp",
    webqsp_selector_ablation_df
)


display_selector_ablation(
    "cwq",
    cwq_selector_ablation_df
)


# ======================================================================
# 13. EXTRACT EXISTING CELL 14B CAUSAL CONTROLS
# ======================================================================

def summary_method_row(
    summary_df,
    method
):

    rows = summary_df[
        summary_df[
            "method"
        ]
        ==
        method
    ]


    assert len(
        rows
    ) == 1


    return rows.iloc[
        0
    ]


def build_component_evidence_table(
    dataset_name,
    selector_df,
    summary_14b_df
):

    rows = []


    # --------------------------------------------------------------
    # New selector ablations
    # --------------------------------------------------------------

    for _, row in (
        selector_df.iterrows()
    ):

        rows.append(
            {
                "dataset":
                    dataset_name,

                "component_test":
                    row[
                        "variant"
                    ],

                "type":
                    "selector_ablation",

                "AR":
                    float(
                        row[
                            "answer_retention"
                        ]
                    ),

                "SSR":
                    float(
                        row[
                            "ssr"
                        ]
                    ),

                "n_runs":
                    1,
            }
        )


    # --------------------------------------------------------------
    # Existing matched/random causal controls
    # --------------------------------------------------------------

    for method, label in [
        (
            "Adaptive-Budget-Random",
            "ranking_removed_matched_adaptive_budget"
        ),
        (
            "Fixed-Top-B",
            "fixed_budget_control"
        ),
        (
            "Random-B",
            "random_fixed_budget_control"
        ),
    ]:

        row = summary_method_row(
            summary_14b_df,
            method
        )


        rows.append(
            {
                "dataset":
                    dataset_name,

                "component_test":
                    label,

                "type":
                    "existing_control",

                "AR":
                    float(
                        row[
                            "ar_mean"
                        ]
                    ),

                "SSR":
                    float(
                        row[
                            "ssr_mean"
                        ]
                    ),

                "n_runs":
                    int(
                        row[
                            "n_runs"
                        ]
                    ),
            }
        )


    return pd.DataFrame(
        rows
    )


webqsp_component_evidence_df = (
    build_component_evidence_table(
        dataset_name=
            "webqsp",

        selector_df=
            webqsp_selector_ablation_df,

        summary_14b_df=
            webqsp_summary_14b_df
    )
)


cwq_component_evidence_df = (
    build_component_evidence_table(
        dataset_name=
            "cwq",

        selector_df=
            cwq_selector_ablation_df,

        summary_14b_df=
            cwq_summary_14b_df
    )
)


print(
    "\n"
    + "=" * 124
)

print(
    "WEBQSP COMPLETE COMPONENT EVIDENCE"
)

print(
    "=" * 124
)


print(
    webqsp_component_evidence_df.to_string(
        index=False,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


print(
    "\n"
    + "=" * 124
)

print(
    "CWQ COMPLETE COMPONENT EVIDENCE"
)

print(
    "=" * 124
)


print(
    cwq_component_evidence_df.to_string(
        index=False,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


# ======================================================================
# 14. VALIDATION DOMINANCE STATUS
# ======================================================================
#
# IMPORTANT:
#
# We explicitly preserve negative / inconvenient development evidence.
#
# If Fixed Top-B Pareto-dominates AFP, the final freeze manifest records
# that fact rather than hiding it.
# ======================================================================

def compute_fixed_top_b_dominance(
    summary_df
):

    afp = summary_method_row(
        summary_df,
        "AFP"
    )


    fixed = summary_method_row(
        summary_df,
        "Fixed-Top-B"
    )


    afp_ar = float(
        afp[
            "ar_mean"
        ]
    )


    afp_ssr = float(
        afp[
            "ssr_mean"
        ]
    )


    fixed_ar = float(
        fixed[
            "ar_mean"
        ]
    )


    fixed_ssr = float(
        fixed[
            "ssr_mean"
        ]
    )


    dominates = (
        fixed_ar
        >=
        afp_ar
        and
        fixed_ssr
        >=
        afp_ssr
        and
        (
            fixed_ar
            >
            afp_ar
            or
            fixed_ssr
            >
            afp_ssr
        )
    )


    return {
        "fixed_top_b_pareto_dominates_afp":
            bool(
                dominates
            ),

        "afp_AR":
            afp_ar,

        "afp_SSR":
            afp_ssr,

        "fixed_top_b_AR":
            fixed_ar,

        "fixed_top_b_SSR":
            fixed_ssr,
    }


WEBQSP_DOMINANCE = (
    compute_fixed_top_b_dominance(
        webqsp_summary_14b_df
    )
)


CWQ_DOMINANCE = (
    compute_fixed_top_b_dominance(
        cwq_summary_14b_df
    )
)


print(
    "\nDevelopment Pareto status:"
)

print(
    "  WebQSP:",
    WEBQSP_DOMINANCE
)

print(
    "  CWQ:   ",
    CWQ_DOMINANCE
)


# ======================================================================
# 15. FEATURE ABLATION FREEZE DECISION
# ======================================================================
#
# Ablations are diagnostic.
#
# We DO NOT use their outcome to redesign the feature set after seeing
# validation results.
#
# Final AFP retains the original full 27-D Feature-v2 representation.
# ======================================================================

FINAL_FEATURE_DECISION = {
    "feature_version":
        AFP_RUNTIME_FEATURE_VERSION,

    "input_dim":
        27,

    "decision":
        "retain_full_feature_v2",

    "reason":
        (
            "Feature-family ablations are diagnostic only; "
            "no feature-removal variant is adopted post hoc. "
            "Full Feature-v2 remains the pre-existing method "
            "representation."
        ),
}


print(
    "\nFinal feature decision:"
)

print(
    "  retain full 27-D Feature-v2"
)


# ======================================================================
# 16. ORIGINAL SELECTED SCORER METADATA
# ======================================================================

def selected_checkpoint_metadata(
    checkpoint,
    checkpoint_sha
):

    required = [
        "dataset",
        "hidden_dim",
        "loss_name",
        "seed",
        "epochs",
        "learning_rate",
        "weight_decay",
        "feature_spec_sha256",
        "scorer_spec_sha256",
        "training_spec_sha256",
    ]


    missing_keys = [
        key
        for key in required
        if key not in checkpoint
    ]


    assert not missing_keys, (
        "Selected checkpoint missing metadata: "
        f"{missing_keys}"
    )


    return {
        "checkpoint_sha256":
            str(
                checkpoint_sha
            ),

        "dataset":
            str(
                checkpoint[
                    "dataset"
                ]
            ),

        "hidden_dim":
            int(
                checkpoint[
                    "hidden_dim"
                ]
            ),

        "loss_name":
            str(
                checkpoint[
                    "loss_name"
                ]
            ),

        "seed":
            int(
                checkpoint[
                    "seed"
                ]
            ),

        "epochs":
            int(
                checkpoint[
                    "epochs"
                ]
            ),

        "learning_rate":
            float(
                checkpoint[
                    "learning_rate"
                ]
            ),

        "weight_decay":
            float(
                checkpoint[
                    "weight_decay"
                ]
            ),

        "feature_spec_sha256":
            str(
                checkpoint[
                    "feature_spec_sha256"
                ]
            ),

        "scorer_spec_sha256":
            str(
                checkpoint[
                    "scorer_spec_sha256"
                ]
            ),

        "training_spec_sha256":
            str(
                checkpoint[
                    "training_spec_sha256"
                ]
            ),
    }


WEBQSP_SCORER_FREEZE = (
    selected_checkpoint_metadata(
        webqsp_ckpt_obj,
        webqsp_ckpt_sha
    )
)


CWQ_SCORER_FREEZE = (
    selected_checkpoint_metadata(
        cwq_ckpt_obj,
        cwq_ckpt_sha
    )
)


assert WEBQSP_SCORER_FREEZE[
    "hidden_dim"
] == 32


assert CWQ_SCORER_FREEZE[
    "hidden_dim"
] == 64


assert WEBQSP_SCORER_FREEZE[
    "seed"
] == 42


assert CWQ_SCORER_FREEZE[
    "seed"
] == 42


assert WEBQSP_SCORER_FREEZE[
    "loss_name"
] == "branch_bce"


assert CWQ_SCORER_FREEZE[
    "loss_name"
] == "branch_bce"


print(
    "\nSelected original scorer metadata: PASSED"
)


# ======================================================================
# 17. JSON-SAFE CONVERSION
# ======================================================================

def json_safe_15b(
    value
):

    if isinstance(
        value,
        dict
    ):

        return {
            str(
                key
            ):
                json_safe_15b(
                    item
                )

            for key, item
            in value.items()
        }


    if isinstance(
        value,
        (
            list,
            tuple,
        )
    ):

        return [
            json_safe_15b(
                item
            )
            for item in value
        ]


    if isinstance(
        value,
        np.integer
    ):

        return int(
            value
        )


    if isinstance(
        value,
        np.floating
    ):

        value = float(
            value
        )

        if math.isnan(
            value
        ):

            return None

        return value


    if isinstance(
        value,
        np.bool_
    ):

        return bool(
            value
        )


    if isinstance(
        value,
        float
    ):

        if math.isnan(
            value
        ):

            return None


    return value


def df_records_json_safe(
    df
):

    return [
        json_safe_15b(
            row
        )

        for row
        in df.to_dict(
            orient="records"
        )
    ]


# ======================================================================
# 18. FINAL DEVELOPMENT FREEZE PAYLOAD
# ======================================================================
#
# THIS is the immutable configuration to carry to TEST.
#
# Note carefully:
#
# - Final scorer is the ORIGINAL selected checkpoint from Cell 9C/B4.
# - The full-v2 retrain in Cell 15A was ONLY an ablation-control model.
# - Ablation models are NOT promoted to the final method.
# ======================================================================

FINAL_AFP_CONFIG = {
    "framework":
        "AdaPruner-KGQA",

    "component":
        "Adaptive Frontier Pruning",

    "development_freeze_version":
        "afp_final_development_freeze_v1",

    # --------------------------------------------------------------
    # Architectural scope
    # --------------------------------------------------------------

    "scope": {
        "position":
            "inside_RoG_relation_constrained_retrieval",

        "relation_planner_changed":
            False,

        "reasoner_changed":
            False,

        "unrestricted_KG_search":
            False,

        "intermediate_hop_pruning_only":
            True,

        "final_hop_protection":
            True,

        "singleton_bypass":
            True,
    },

    # --------------------------------------------------------------
    # Representation
    # --------------------------------------------------------------

    "features": {
        "version":
            AFP_RUNTIME_FEATURE_VERSION,

        "dimension":
            27,

        "families": [
            "semantic",
            "path",
            "structural",
            "progress",
        ],

        "raw_MID_embedding":
            False,

        "external_entity_resolver":
            False,

        "future_candidate_neighborhood_features":
            False,

        "final_decision":
            "retain_full_feature_v2",
    },

    # --------------------------------------------------------------
    # Scorers
    # --------------------------------------------------------------

    "scorer": {
        "architecture":
            "AFPScorer_MLP",

        "activation":
            "ReLU",

        "dropout":
            0.0,

        "loss":
            "branch_bce",

        "deployment_seed":
            42,

        "webqsp":
            WEBQSP_SCORER_FREEZE,

        "cwq":
            CWQ_SCORER_FREEZE,

        "selected_checkpoint_source":
            "original_Cell9C_B4_selected_checkpoint",

        "Cell15A_full_retrain_promoted":
            False,
    },

    # --------------------------------------------------------------
    # Selector
    # --------------------------------------------------------------

    "selector": {
        "webqsp":
            FINAL_SELECTOR_PARAMS[
                "webqsp"
            ],

        "cwq":
            FINAL_SELECTOR_PARAMS[
                "cwq"
            ],

        "probability":
            "softmax(logits / T)",

        "uncertainty":
            "normalized_entropy",

        "gamma_rule":
            "gamma_min + uncertainty * (1 - gamma_min)",

        "budget_rule":
            (
                "smallest cumulative-probability top-B "
                "reaching gamma"
            ),

        "fully_tied_behavior":
            "retain_all",

        "cutoff_tie_behavior":
            "expand_cutoff_ties",

        "one_time_boundary_expansion_used":
            True,

        "further_hyperparameter_search_allowed":
            False,
    },

    # --------------------------------------------------------------
    # Training / leakage controls
    # --------------------------------------------------------------

    "leakage_controls": {
        "scorer_training_labels":
            "train_only",

        "selector_tuning":
            "validation_only",

        "test_used_for_training":
            False,

        "test_used_for_model_selection":
            False,

        "test_used_for_hyperparameter_tuning":
            False,

        "test_examples_accessed_during_development":
            False,
    },

    # --------------------------------------------------------------
    # Metrics
    # --------------------------------------------------------------

    "evaluation": {
        "primary_search_cost":
            "edges_examined",

        "SSR":
            "1 - E_method / E_RoG",

        "answer_retention":
            (
                "method_reachable_questions / "
                "RoG_reachable_questions"
            ),

        "AR_selection_floor":
            0.99,

        "active_hop_rows_distinct_from_active_prefixes":
            True,

        "candidate_branches_distinct_from_unique_entities":
            True,
    },

    # --------------------------------------------------------------
    # Development evidence
    # --------------------------------------------------------------

    "validation_selected_AFP": {
        "webqsp": {
            "AR":
                float(
                    get_single_row(
                        webqsp_comparison_14b_df,
                        "method",
                        "AFP"
                    )[
                        "answer_retention"
                    ]
                ),

            "SSR":
                float(
                    get_single_row(
                        webqsp_comparison_14b_df,
                        "method",
                        "AFP"
                    )[
                        "ssr"
                    ]
                ),
        },

        "cwq": {
            "AR":
                float(
                    get_single_row(
                        cwq_comparison_14b_df,
                        "method",
                        "AFP"
                    )[
                        "answer_retention"
                    ]
                ),

            "SSR":
                float(
                    get_single_row(
                        cwq_comparison_14b_df,
                        "method",
                        "AFP"
                    )[
                        "ssr"
                    ]
                ),
        },
    },

    "validation_fixed_top_b_dominance":
        {
            "webqsp":
                WEBQSP_DOMINANCE,

            "cwq":
                CWQ_DOMINANCE,

            "interpretation":
                (
                    "Validation evidence must not be reported "
                    "as showing AFP outperforming Fixed Top-B "
                    "when Fixed Top-B Pareto-dominates AFP."
                ),
        },

    "feature_ablation_results": {
        "webqsp":
            df_records_json_safe(
                webqsp_feature_ablation_df
            ),

        "cwq":
            df_records_json_safe(
                cwq_feature_ablation_df
            ),
    },

    "selector_ablation_results": {
        "webqsp":
            df_records_json_safe(
                webqsp_selector_ablation_df
            ),

        "cwq":
            df_records_json_safe(
                cwq_selector_ablation_df
            ),
    },

    "component_control_results": {
        "webqsp":
            df_records_json_safe(
                webqsp_component_evidence_df
            ),

        "cwq":
            df_records_json_safe(
                cwq_component_evidence_df
            ),
    },

    # --------------------------------------------------------------
    # Freeze state
    # --------------------------------------------------------------

    "development_status": {
        "feature_development_complete":
            True,

        "scorer_development_complete":
            True,

        "selector_development_complete":
            True,

        "controlled_validation_comparison_complete":
            True,

        "feature_ablations_complete":
            True,

        "selector_component_ablations_complete":
            True,

        "no_further_development_changes_before_test":
            True,

        "final_AFP_configuration_frozen":
            True,

        "test_evaluation_started":
            False,
    },
}


# ======================================================================
# 19. CANONICAL FREEZE HASH
# ======================================================================

FINAL_AFP_CONFIG_SAFE = (
    json_safe_15b(
        FINAL_AFP_CONFIG
    )
)


FREEZE_CANONICAL_JSON = (
    json.dumps(
        FINAL_AFP_CONFIG_SAFE,
        sort_keys=True,
        separators=(
            ",",
            ":"
        ),
        ensure_ascii=False
    )
)


FINAL_AFP_FREEZE_SHA256 = (
    hashlib.sha256(
        FREEZE_CANONICAL_JSON.encode(
            "utf-8"
        )
    ).hexdigest()
)


print(
    "\nFinal AFP development-freeze SHA256:"
)

print(
    " ",
    FINAL_AFP_FREEZE_SHA256
)


# ======================================================================
# 20. SAVE FINAL FREEZE ARTIFACT
# ======================================================================

FINAL_FREEZE_JSON_PATH = (
    CELL15B_DIR
    / "final_afp_development_freeze.json"
)


FINAL_FREEZE_SHA_PATH = (
    CELL15B_DIR
    / "final_afp_development_freeze.sha256"
)


with open(
    FINAL_FREEZE_JSON_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        FINAL_AFP_CONFIG_SAFE,
        f,
        indent=2,
        ensure_ascii=False
    )


with open(
    FINAL_FREEZE_SHA_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        FINAL_AFP_FREEZE_SHA256
        +
        "\n"
    )


# ======================================================================
# 21. SAVE ABLATION TABLES
# ======================================================================

WEBQSP_SELECTOR_ABLATION_CSV = (
    CELL15B_DIR
    / "webqsp_selector_component_ablation.csv"
)


CWQ_SELECTOR_ABLATION_CSV = (
    CELL15B_DIR
    / "cwq_selector_component_ablation.csv"
)


WEBQSP_COMPONENT_EVIDENCE_CSV = (
    CELL15B_DIR
    / "webqsp_complete_component_evidence.csv"
)


CWQ_COMPONENT_EVIDENCE_CSV = (
    CELL15B_DIR
    / "cwq_complete_component_evidence.csv"
)


webqsp_selector_ablation_df.to_csv(
    WEBQSP_SELECTOR_ABLATION_CSV,
    index=False
)


cwq_selector_ablation_df.to_csv(
    CWQ_SELECTOR_ABLATION_CSV,
    index=False
)


webqsp_component_evidence_df.to_csv(
    WEBQSP_COMPONENT_EVIDENCE_CSV,
    index=False
)


cwq_component_evidence_df.to_csv(
    CWQ_COMPONENT_EVIDENCE_CSV,
    index=False
)


# ======================================================================
# 22. FINAL FREEZE SOFTWARE GATES
# ======================================================================

assert FINAL_AFP_CONFIG_SAFE[
    "development_status"
][
    "final_AFP_configuration_frozen"
] is True


assert FINAL_AFP_CONFIG_SAFE[
    "development_status"
][
    "test_evaluation_started"
] is False


assert FINAL_AFP_CONFIG_SAFE[
    "selector"
][
    "further_hyperparameter_search_allowed"
] is False


assert FINAL_AFP_CONFIG_SAFE[
    "scorer"
][
    "Cell15A_full_retrain_promoted"
] is False


assert FINAL_AFP_CONFIG_SAFE[
    "features"
][
    "dimension"
] == 27


assert FINAL_AFP_CONFIG_SAFE[
    "selector"
][
    "webqsp"
] == {
    "T": 0.05,
    "gamma_min": 0.1,
}


assert FINAL_AFP_CONFIG_SAFE[
    "selector"
][
    "cwq"
] == {
    "T": 0.1,
    "gamma_min": 0.1,
}


assert (
    sha256_file(
        Path(
            "/kaggle/working/"
            "step3_rq2_dev_v1/"
            "07_final_scorer/"
            "webqsp_afp_scorer_selected.pt"
        )
    )
    ==
    webqsp_ckpt_sha
)


assert (
    sha256_file(
        Path(
            "/kaggle/working/"
            "step3_rq2_dev_v1/"
            "07_final_scorer/"
            "cwq_afp_scorer_selected.pt"
        )
    )
    ==
    cwq_ckpt_sha
)


print(
    "\nFinal checkpoint identity gate: PASSED"
)


# ======================================================================
# 23. DEVELOPMENT FREEZE FLAGS
# ======================================================================

CELL15B_COMPLETE = True

AFP_DEVELOPMENT_FROZEN = True

FINAL_AFP_CONFIG_FROZEN = (
    FINAL_AFP_CONFIG_SAFE
)


# ======================================================================
# 24. FINAL REPORT
# ======================================================================

print(
    "\n"
    + "=" * 126
)

print(
    "=== RQ2 CELL 15B: "
    "SELECTOR ABLATIONS + FINAL AFP DEVELOPMENT FREEZE COMPLETE ==="
)

print(
    "=" * 126
)


print(
    "\nFINAL AFP CONFIGURATION"
)


print(
    "\nFeature representation:"
)

print(
    "  Feature-v2"
)

print(
    "  Dimension: 27"
)

print(
    "  Semantic + Path + Structural + Progress"
)


print(
    "\nSelected scorer checkpoints:"
)

print(
    "  WebQSP:"
)

print(
    "   H =",
    WEBQSP_SCORER_FREEZE[
        "hidden_dim"
    ]
)

print(
    "   SHA =",
    webqsp_ckpt_sha
)


print(
    "  CWQ:"
)

print(
    "   H =",
    CWQ_SCORER_FREEZE[
        "hidden_dim"
    ]
)

print(
    "   SHA =",
    cwq_ckpt_sha
)


print(
    "\nFinal AFP selector:"
)

print(
    "  WebQSP:",
    FINAL_SELECTOR_PARAMS[
        "webqsp"
    ]
)

print(
    "  CWQ:   ",
    FINAL_SELECTOR_PARAMS[
        "cwq"
    ]
)


print(
    "\nPolicy:"
)

print(
    "  intermediate pruning only"
)

print(
    "  final-hop protection = TRUE"
)

print(
    "  singleton bypass = TRUE"
)

print(
    "  fully tied scores = retain all"
)

print(
    "  cutoff ties = expand ties"
)


print(
    "\nDevelopment evidence status:"
)

print(
    "  Controlled validation comparison: COMPLETE"
)

print(
    "  Feature-family ablations:          COMPLETE"
)

print(
    "  Selector/component ablations:      COMPLETE"
)

print(
    "  Further hyperparameter search:     PROHIBITED"
)


print(
    "\nValidation caveat explicitly frozen:"
)

print(
    "  WebQSP Fixed Top-B Pareto-dominates AFP:",
    WEBQSP_DOMINANCE[
        "fixed_top_b_pareto_dominates_afp"
    ]
)

print(
    "  CWQ Fixed Top-B Pareto-dominates AFP:",
    CWQ_DOMINANCE[
        "fixed_top_b_pareto_dominates_afp"
    ]
)


print(
    "\nLeakage status:"
)

print(
    "  Train labels for scorer: TRAIN only"
)

print(
    "  Selector tuning: VALIDATION only"
)

print(
    "  TEST examples accessed: NO"
)

print(
    "  TEST evaluation started: NO"
)


print(
    "\nFREEZE STATUS:"
)

print(
    "  AFP_DEVELOPMENT_FROZEN = TRUE"
)

print(
    "  No further development changes before TEST."
)


print(
    "\nFreeze SHA256:"
)

print(
    " ",
    FINAL_AFP_FREEZE_SHA256
)


print(
    "\nArtifacts:"
)

print(
    " ",
    WEBQSP_SELECTOR_ABLATION_CSV
)

print(
    " ",
    CWQ_SELECTOR_ABLATION_CSV
)

print(
    " ",
    WEBQSP_COMPONENT_EVIDENCE_CSV
)

print(
    " ",
    CWQ_COMPONENT_EVIDENCE_CSV
)

print(
    " ",
    FINAL_FREEZE_JSON_PATH
)

print(
    " ",
    FINAL_FREEZE_SHA_PATH
)


print(
    "\nNEXT STEP:"
)

print(
    "FROZEN TEST evaluation."
)

print(
    "No tuning, retraining, feature changes, "
    "or selector changes are allowed after this point."
)

Cell 15B prerequisites: PASSED
Feature version: afp_features_v2_masked_entity_semantics
Cell 15B output: /kaggle/working/step3_rq2_dev_v1/14_selector_ablations_and_final_freeze

Final development-selected selector:
  WebQSP: {'T': 0.05, 'gamma_min': 0.1}
  CWQ:    {'T': 0.1, 'gamma_min': 0.1}

Selector/component ablations:
  full_afp
  no_uncertainty_adaptation
  no_temperature_scaling

No ablation-specific tuning: YES

RUNNING SELECTOR / COMPONENT ABLATIONS


webqsp selector ablations:   0%|          | 0/246 [00:00<?, ?it/s]


WEBQSP selector ablations completed in 0.03 min


cwq selector ablations:   0%|          | 0/3519 [00:00<?, ?it/s]


CWQ selector ablations completed in 2.18 min
WebQSP full AFP Cell-14B fidelity: PASSED
CWQ full AFP Cell-14B fidelity: PASSED

WEBQSP SELECTOR / COMPONENT ABLATIONS
                  variant  answer_retention      ssr  edges_examined  active_prefixes  candidate_branches  decision_hops  avg_requested_budget  avg_uncertainty  avg_gamma  fully_tied_decisions  delta_ar_vs_full  delta_ssr_vs_full  delta_edges_vs_full
                 full_afp          1.000000 0.010272          338018             2293                6659            147              9.891156         0.894528   0.905075                    97          0.000000           0.000000                    0
no_uncertainty_adaptation          1.000000 0.017855          335428             2110                6012            147              7.306122         0.894528   0.693878                    97          0.000000           0.007584                -2590
   no_temperature_scaling          1.000000 0.000123          341484             